PROGRAM 1

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
=============================================================================
PENELITIAN: Klasifikasi Stunting pada Balita di Kalimantan Tengah
            Menggunakan Logika Fuzzy (Studi Kasus Data SKI 2023)
=============================================================================
TASK 1: INPUT DATA - Loading Dataset & Seleksi Fitur
TASK 2: PREPROCESSING - Pembersihan, Penyaringan, Transformasi
TASK 3: STUNTING LABELLING & Z-SCORE - Ground Truth WHO
=============================================================================
Lokasi: train/fuzzytrain.py
Struktur:
  PROJECTS2/
    train/
      fuzzytrain.py              ← FILE INI
      dataset/
        Individu_balita_2023_merged.csv
      trainfile/                 ← output hasil train disimpan di sini
      zs/
        boys02.xlsx   (boys 0-24 bulan)
        boys25.xlsx   (boys 24-60 bulan)
        girls02.xlsx  (girls 0-24 bulan)
        girls25.xlsx  (girls 24-60 bulan)
    mainprogram/
    test/
=============================================================================
"""

import pandas as pd
import numpy as np
import json
import os
from datetime import datetime

# ============================================================================
# PATH CONFIGURATION (Relative to this script's location)
# ============================================================================
BASE_DIR = os.path.dirname(os.path.abspath(__file__))

DATASET_PATH   = os.path.join(BASE_DIR, 'dataset', 'Individu_balita_2023_merged.csv')
WHO_ZS_DIR     = os.path.join(BASE_DIR, 'zs')
TRAINFILE_DIR  = os.path.join(BASE_DIR, 'trainfile')

OUTPUT_CLEAN_CSV   = os.path.join(TRAINFILE_DIR, 'dataset_clean.csv')
OUTPUT_LABELED_CSV = os.path.join(TRAINFILE_DIR, 'dataset_labeled.csv')
OUTPUT_REPORT      = os.path.join(TRAINFILE_DIR, 'preprocessing_report.json')

os.makedirs(TRAINFILE_DIR, exist_ok=True)

# WHO Z-Score table file paths
WHO_TABLES = {
    'boys_0_24':  os.path.join(WHO_ZS_DIR, 'boys02.xlsx'),
    'boys_24_60': os.path.join(WHO_ZS_DIR, 'boys25.xlsx'),
    'girls_0_24':  os.path.join(WHO_ZS_DIR, 'girls02.xlsx'),
    'girls_24_60': os.path.join(WHO_ZS_DIR, 'girls25.xlsx'),
}

# ============================================================================
# KONFIGURASI BATAS NORMAL MEDIS
# ============================================================================
VALID_RANGES = {
    'usia_bulan':    (0, 59),
    'jenis_kelamin': [1, 2],
    'tinggi_badan':  (35.0, 130.0),
    'bbl_gram':      (500, 6000),
    'pbl_cm':        (25.0, 60.0),
    'mpasi_bulan':   (0, 12),
}

NOISE_CODES = {
    'bbl': ['8888', '9999', '88.8'],
    'pbl': ['88', '88.8', '99', '99.9'],
}

MPASI_CODE_MAP = {
    'a': 0.25, 'b': 0.5,  'c': 1.0,  'd': 2.0,  'e': 3.0,
    'f': 4.0,  'g': 5.0,  'h': 6.0,  'i': 7.0,  'j': 8.0,
    'k': 9.0,  'l': 10.0, 'm': 11.0, 'n': 12.0,
}

MPASI_NOISE = {'o', 'p', ' ', ''}


# ============================================================================
# FUNGSI UTILITAS
# ============================================================================
def print_header(title, char='=', width=75):
    print(f"\n{char * width}")
    print(f"  {title}")
    print(f"{char * width}")

def print_step(step_num, description):
    print(f"\n  [{step_num}] {description}")
    print(f"  {'-' * 65}")

def safe_numeric(series, errors='coerce'):
    cleaned = series.astype(str).str.strip().replace({'': np.nan, ' ': np.nan})
    return pd.to_numeric(cleaned, errors=errors)


# ============================================================================
# TASK 1: INPUT DATA
# ============================================================================
def task1_input_data():
    print_header("TASK 1: INPUT DATA - Loading Dataset & Seleksi Fitur")
    
    print_step("1.1", "Memuat dataset SKI 2023 Kalimantan Tengah")
    print(f"    Path: {DATASET_PATH}")
    
    if not os.path.exists(DATASET_PATH):
        raise FileNotFoundError(
            f"\n    ✗ DATASET TIDAK DITEMUKAN!\n"
            f"    Pastikan file berada di: {DATASET_PATH}\n"
        )
    
    df_raw = pd.read_csv(DATASET_PATH, low_memory=False)
    print(f"    ✓ Dataset dimuat: {df_raw.shape[0]:,} baris × {df_raw.shape[1]} kolom")
    
    print_step("1.2", "Mapping kolom dataset ke variabel penelitian")
    
    COLUMN_MAP = {
        'Umur_BLN':             'usia_bulan',
        'Jenis Kelamin':        'jenis_kelamin',
        'Panjang Badan Anak':   'tinggi_badan_cm',
        'Berat Badan Lahir':    'bbl_raw',
        'Panjang Badan Lahir':  'pbl_raw',
        'I43_MP ASI':           'mpasi_raw',
    }
    
    df = df_raw[['ID_balita'] + list(COLUMN_MAP.keys())].copy()
    df.rename(columns=COLUMN_MAP, inplace=True)
    
    for var_lama, var_baru in COLUMN_MAP.items():
        print(f"    • {var_lama:30s} → {var_baru}")
    
    print(f"\n    ✓ Seleksi fitur selesai: {df.shape[1]} kolom dipilih dari {df_raw.shape[1]} kolom")
    
    print_step("1.3", "Statistik awal dataset (sebelum preprocessing)")
    print(f"    Total balita   : {len(df):,}")
    for col in df.columns:
        if col == 'ID_balita':
            continue
        non_null = df[col].notna().sum()
        blank = (df[col].astype(str).str.strip() == '').sum()
        print(f"    • {col:20s}: {non_null:,} terisi, {blank} kosong/blank")
    
    return df, df_raw.shape


# ============================================================================
# TASK 2: PREPROCESSING
# ============================================================================
def task2_preprocessing(df, raw_shape):
    print_header("TASK 2: PREPROCESSING")
    
    report = {
        'raw_total': len(df),
        'raw_shape': list(raw_shape),
        'steps': {},
        'timestamp': datetime.now().isoformat()
    }
    
    # ==== STEP 2d: TRANSFORMASI & REKODING ====
    print_header("STEP 2d: Transformasi & Rekoding", char='-')
    
    # 2d.1: Usia
    print_step("2d.1", "Transformasi Usia Balita → bulan (integer)")
    df['usia_bulan'] = pd.to_numeric(df['usia_bulan'], errors='coerce')
    df['usia_label'] = df['usia_bulan'].apply(
        lambda x: '<1 bulan' if pd.notna(x) and x < 1 else
                  (f'{int(x)} bulan' if pd.notna(x) else None))
    usia_bawah_1 = (df['usia_bulan'] < 1).sum()
    print(f"    ✓ Balita usia <1 bulan: {usia_bawah_1} → dilabeli '<1 bulan'")
    print(f"    Distribusi: min={df['usia_bulan'].min()}, max={df['usia_bulan'].max()}, mean={df['usia_bulan'].mean():.1f}")
    
    # 2d.2: Jenis Kelamin
    print_step("2d.2", "Validasi Jenis Kelamin → 1=Laki-laki, 2=Perempuan")
    df['jenis_kelamin'] = pd.to_numeric(df['jenis_kelamin'], errors='coerce')
    jk_invalid = ~df['jenis_kelamin'].isin([1, 2]) & df['jenis_kelamin'].notna()
    print(f"    ✓ Valid (1/2): {df['jenis_kelamin'].isin([1, 2]).sum():,} | Invalid: {jk_invalid.sum()}")
    
    # 2d.3: Tinggi Badan
    print_step("2d.3", "Transformasi Panjang/Tinggi Badan → cm (float)")
    df['tinggi_badan_cm'] = safe_numeric(df['tinggi_badan_cm'])
    print(f"    ✓ Valid: {df['tinggi_badan_cm'].notna().sum():,} | NaN: {df['tinggi_badan_cm'].isna().sum()}")
    
    # 2d.4: BBL
    print_step("2d.4", "Transformasi BBL → gram (float)")
    bbl_str = df['bbl_raw'].astype(str).str.strip()
    bbl_is_noise = bbl_str.isin(NOISE_CODES['bbl'])
    df['bbl_gram'] = safe_numeric(df['bbl_raw'])
    df.loc[bbl_is_noise, 'bbl_gram'] = np.nan
    print(f"    ✓ Noise (8888): {bbl_is_noise.sum()} | Valid: {df['bbl_gram'].notna().sum():,}")
    
    # 2d.5: PBL
    print_step("2d.5", "Transformasi PBL → cm (float)")
    pbl_str = df['pbl_raw'].astype(str).str.strip()
    pbl_is_noise = pbl_str.isin(NOISE_CODES['pbl'])
    df['pbl_cm'] = safe_numeric(df['pbl_raw'])
    df.loc[pbl_is_noise, 'pbl_cm'] = np.nan
    print(f"    ✓ Noise (88/88.8): {pbl_is_noise.sum()} | Valid: {df['pbl_cm'].notna().sum():,}")
    
    # 2d.6: MP-ASI
    print_step("2d.6", "Transformasi MP-ASI (kode huruf) → bulan (float)")
    mpasi_str = df['mpasi_raw'].astype(str).str.strip().str.lower()
    df['mpasi_bulan'] = mpasi_str.map(MPASI_CODE_MAP)
    mapped_ok = df['mpasi_bulan'].notna().sum()
    
    print(f"    Mapping kode huruf → bulan:")
    for code, bulan in sorted(MPASI_CODE_MAP.items()):
        count = (mpasi_str == code).sum()
        if count > 0:
            lbl = f"<1 bln" if bulan < 1 else f"{bulan:.0f} bln"
            print(f"      '{code}' → {lbl:>8s}: {count:>4d} balita")
    print(f"    ✓ Mapped: {mapped_ok:,} | Missing: {df['mpasi_bulan'].isna().sum():,}")
    
    df['mpasi_label'] = df['mpasi_bulan'].apply(
        lambda x: '<1 bulan' if pd.notna(x) and x < 1 else
                  (f'{int(x)} bulan' if pd.notna(x) and x == int(x) else
                   (f'{x} bulan' if pd.notna(x) else None)))
    
    report['steps']['2d_transformasi'] = {
        'usia_bawah_1_bulan': int(usia_bawah_1),
        'jk_invalid': int(jk_invalid.sum()),
        'bbl_noise_8888': int(bbl_is_noise.sum()),
        'pbl_noise_88': int(pbl_is_noise.sum()),
        'mpasi_mapped': int(mapped_ok),
        'mpasi_missing': int(df['mpasi_bulan'].isna().sum()),
    }
    
    # ==== STEP 2b: FILTER USIA ====
    print_header("STEP 2b: Penyaringan Wilayah & Usia", char='-')
    print(f"    ✓ Dataset khusus Kalimantan Tengah ({len(df):,} balita)")
    n_before_age = len(df)
    usia_valid = df['usia_bulan'].between(0, 59, inclusive='both')
    usia_diluar = ~usia_valid & df['usia_bulan'].notna()
    if usia_diluar.any():
        df = df[usia_valid | df['usia_bulan'].isna()].copy()
        print(f"    ✓ Dihapus: {n_before_age - len(df)} balita di luar 0-59 bulan")
    else:
        print(f"    ✓ Semua dalam rentang 0-59 bulan")
    
    report['steps']['2b_filter_usia'] = {
        'before': n_before_age, 'after': len(df),
    }
    
    # ==== STEP 2c: DATA REDUCTION ====
    print_header("STEP 2c: Data Reduction (Seleksi Fitur)", char='-')
    FINAL_COLUMNS = ['ID_balita', 'usia_bulan', 'usia_label', 'jenis_kelamin',
                     'tinggi_badan_cm', 'bbl_gram', 'pbl_cm', 'mpasi_bulan', 'mpasi_label']
    df = df[FINAL_COLUMNS].copy()
    print(f"    ✓ {len(FINAL_COLUMNS)} kolom dipertahankan dari {raw_shape[1]} kolom asli")
    
    # ==== STEP 2a: LISTWISE DELETION ====
    print_header("STEP 2a: Pemeriksaan Kelengkapan & Pembuangan Noise", char='-')
    print(f"    Metode: LISTWISE DELETION (Tabachnick & Fidell, 2014)")
    
    CORE_VARS = ['usia_bulan', 'jenis_kelamin', 'tinggi_badan_cm',
                 'bbl_gram', 'pbl_cm', 'mpasi_bulan']
    
    n_before = len(df)
    
    print_step("2a.1", "Missing values per variabel inti")
    missing_detail = {}
    for var in CORE_VARS:
        n_miss = df[var].isna().sum()
        pct = n_miss / len(df) * 100
        missing_detail[var] = n_miss
        s = "✓" if n_miss == 0 else "⚠"
        print(f"    {s} {var:20s}: {n_miss:>5d} missing ({pct:>5.1f}%)")
    
    print_step("2a.2", "Deteksi outlier medis")
    outlier_flags = pd.DataFrame(index=df.index)
    
    for col, key in [('usia_bulan','usia_bulan'), ('tinggi_badan_cm','tinggi_badan'),
                     ('bbl_gram','bbl_gram'), ('pbl_cm','pbl_cm'), ('mpasi_bulan','mpasi_bulan')]:
        lo, hi = VALID_RANGES[key]
        mask = ~df[col].between(lo, hi) & df[col].notna()
        outlier_flags[f'{col}_out'] = mask
        if mask.any():
            print(f"    ⚠ {col}: {mask.sum()} di luar [{lo}-{hi}]")
            print(f"      Nilai: {sorted(df.loc[mask, col].unique()[:5])}")
        else:
            print(f"    ✓ {col}: OK [{lo}-{hi}]")
    
    mask_jk = ~df['jenis_kelamin'].isin([1, 2]) & df['jenis_kelamin'].notna()
    outlier_flags['jk_out'] = mask_jk
    print(f"    {'⚠' if mask_jk.any() else '✓'} jenis_kelamin: {mask_jk.sum()} invalid")
    
    print_step("2a.3", "Eksekusi Listwise Deletion")
    any_missing = df[CORE_VARS].isna().any(axis=1)
    any_outlier = outlier_flags.any(axis=1)
    rows_to_drop = any_missing | any_outlier
    
    reasons = {}
    for var in CORE_VARS:
        vm = df[var].isna()
        oc = f'{var}_out' if f'{var}_out' in outlier_flags else 'jk_out'
        vo = outlier_flags.get(oc, pd.Series(False, index=df.index))
        vp = vm | vo
        reasons[var] = int(vp.sum())
        print(f"    • {var:20s}: {vp.sum():>5d} bermasalah (miss={vm.sum()}, outlier={vo.sum()})")
    
    df_clean = df[~rows_to_drop].copy().reset_index(drop=True)
    n_dropped = n_before - len(df_clean)
    
    print(f"\n    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    print(f"    LISTWISE DELETION RESULT:")
    print(f"    • Sebelum : {n_before:,} balita")
    print(f"    • Dihapus : {n_dropped:,} ({n_dropped/n_before*100:.1f}%)")
    print(f"    • Sesudah : {len(df_clean):,} balita")
    print(f"    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    
    report['steps']['2a_listwise'] = {
        'before': n_before, 'total_dropped': n_dropped, 'after': len(df_clean),
        'missing_per_var': {k: int(v) for k, v in missing_detail.items()},
        'reasons': reasons,
    }
    
    # Validasi
    print(f"\n    Validasi: zero missing di semua variabel inti:")
    for var in CORE_VARS:
        n_miss = df_clean[var].isna().sum()
        print(f"    {'✓' if n_miss==0 else '✗'} {var}: {n_miss} missing")
    
    report['final_clean'] = {'rows': len(df_clean), 'cols': df_clean.shape[1]}
    
    return df_clean, report


# ============================================================================
# TASK 3: STUNTING LABELLING & Z-SCORE
# ============================================================================
def task3_zscore_labelling(df):
    """
    Task 3: Menghitung Z-Score WHO (HAZ/LAZ) dan membuat label stunting.
    
    ═══════════════════════════════════════════════════════════════════
    PENTING: Dataset dianalisa PER BARIS (row-by-row).
    Setiap baris = 1 balita → dihitung Z-Score-nya secara INDIVIDUAL.
    ═══════════════════════════════════════════════════════════════════
    
    Alur ANALISA PER BARIS sesuai Proposal Bab 3.3.3:
    ┌─────────────────────────────────────────────────────────────┐
    │  Untuk SETIAP BARIS data balita (iterasi baris ke-1..N):   │
    │                                                             │
    │  1. Baca: jenis_kelamin, usia_bulan, tinggi_badan_cm       │
    │  2. Pilih tabel WHO sesuai jenis kelamin:                  │
    │     • JK=1 (Laki-laki) → boys02/boys25.xlsx                │
    │     • JK=2 (Perempuan) → girls02/girls25.xlsx              │
    │  3. Cari baris WHO berdasarkan usia (bulan) → ambil L,M,S  │
    │  4. Hitung Z-score: Z = [(x/M)^L - 1] / (L × S)          │
    │  5. Labelling:                                              │
    │     • HAZ < -2.00 SD → Stunting                            │
    │     • HAZ ≥ -2.00 SD → Normal                              │
    └─────────────────────────────────────────────────────────────┘
    
    Rumus Z-Score WHO (Persamaan II.2 Proposal):
    Z = [(x/M)^L - 1] / (L × S)
 - x = nilai pengukuran (tinggi/panjang badan cm)
 - M (Miu) = median populasi referensi
 - L (Lambda) = parameter power (transformasi normalitas)
 - S (Sigma) = koefisien variasi
    """
    
    print_header("TASK 3: STUNTING LABELLING & Z-SCORE (Ground Truth WHO)")
    
    print(f"""
    ╔═══════════════════════════════════════════════════════════════╗
    ║  METODE ANALISA: PER BARIS (ROW-BY-ROW ANALYSIS)            ║
    ║                                                              ║
    ║  Setiap baris dataset = 1 balita individual.                 ║
    ║  Analisa dilakukan SATU PER SATU untuk setiap balita:        ║
    ║                                                              ║
    ║  Baris ke-i → Baca (JK, Usia, TB) → Cari WHO(L,M,S)        ║
    ║            → Hitung Z = [(x/M)^L-1]/(L×S) → Label           ║
    ║                                                              ║
    ║  Total baris yang akan dianalisa: {len(df):>4d} balita              ║
    ╚═══════════════════════════════════════════════════════════════╝
    """)
    
    # ══════════════════════════════════════════════════════════════
    # 3.1: Load Tabel WHO
    # ══════════════════════════════════════════════════════════════
    print_step("3.1", "Memuat tabel referensi WHO (LMS parameters)")
    
    who_lookup = {}  # {(jenis_kelamin, usia_bulan): {'L': ..., 'M': ..., 'S': ...}}
    
    table_config = [
        ('boys_0_24',  1, 'Laki-laki 0-24 bln (LAZ)'),
        ('boys_24_60', 1, 'Laki-laki 24-60 bln (HAZ)'),
        ('girls_0_24',  2, 'Perempuan 0-24 bln (LAZ)'),
        ('girls_24_60', 2, 'Perempuan 24-60 bln (HAZ)'),
    ]
    
    for table_key, jk, label in table_config:
        fpath = WHO_TABLES[table_key]
        if not os.path.exists(fpath):
            raise FileNotFoundError(f"    ✗ WHO table tidak ditemukan: {fpath}")
        
        df_who = pd.read_excel(fpath)
        df_who.columns = df_who.columns.str.strip()
        
        n_loaded = 0
        for _, row in df_who.iterrows():
            month = int(row['Month'])
            L_val = float(row['L'])
            M_val = float(row['M'])
            S_val = float(row['S'])
            
            key = (jk, month)
            who_lookup[key] = {'L': L_val, 'M': M_val, 'S': S_val}
            n_loaded += 1
        
        print(f"    ✓ {label:35s} : {n_loaded:>3d} baris dimuat")
    
    total_entries = len(who_lookup)
    print(f"\n    ✓ Total lookup entries: {total_entries}")
    print(f"    ✓ Coverage: Laki-laki 0-60 bln + Perempuan 0-60 bln")
    
    # ══════════════════════════════════════════════════════════════
    # 3.2: Verifikasi Cakupan
    # ══════════════════════════════════════════════════════════════
    print_step("3.2", "Verifikasi cakupan usia dataset vs tabel WHO")
    
    usia_range = df['usia_bulan'].unique()
    jk_range = df['jenis_kelamin'].unique()
    
    missing_keys = []
    for jk in sorted(jk_range):
        for usia in sorted(usia_range):
            key = (int(jk), int(usia))
            if key not in who_lookup:
                missing_keys.append(key)
    
    if missing_keys:
        print(f"    ⚠ {len(missing_keys)} kombinasi TIDAK ditemukan di tabel WHO:")
        for k in missing_keys[:10]:
            print(f"      JK={k[0]}, Usia={k[1]} bulan")
    else:
        print(f"    ✓ Semua kombinasi (jenis_kelamin, usia) tercakup dalam tabel WHO")
    
    print(f"    Usia dataset: {int(min(usia_range))}-{int(max(usia_range))} bulan")
    print(f"    Jenis kelamin: {sorted([int(x) for x in jk_range])}")
    
    # ══════════════════════════════════════════════════════════════
    # 3.3: ANALISA PER BARIS - Hitung Z-Score WHO (Rumus LMS)
    # ══════════════════════════════════════════════════════════════
    print_step("3.3", "ANALISA PER BARIS - Menghitung Z-Score WHO untuk setiap balita")
    print(f"    Rumus WHO (Persamaan II.2): Z = [(x/M)^L - 1] / (L × S)")
    print(f"    Dimana:")
    print(f"      x = panjang/tinggi badan balita (cm) - data pengukuran")
    print(f"      L (Lambda) = parameter power transformasi normalitas")
    print(f"      M (Miu)    = median populasi referensi WHO")
    print(f"      S (Sigma)  = koefisien variasi populasi referensi")
    print(f"    Threshold: HAZ < -2.00 SD → Stunting | HAZ ≥ -2.00 SD → Normal")
    print(f"\n    Memulai analisa per baris untuk {len(df):,} balita...")
    print(f"    {'─' * 65}")
    
    # Array untuk hasil
    z_scores = np.full(len(df), np.nan)
    L_values = np.full(len(df), np.nan)
    M_values = np.full(len(df), np.nan)
    S_values = np.full(len(df), np.nan)
    labels = [None] * len(df)
    
    matched = 0
    not_matched = 0
    
    # === ITERASI PER BARIS ===
    for idx in range(len(df)):
        jk = int(df.iloc[idx]['jenis_kelamin'])
        usia = int(df.iloc[idx]['usia_bulan'])
        x = float(df.iloc[idx]['tinggi_badan_cm'])
        
        # Step A: Pilih tabel WHO berdasarkan jenis kelamin
        # Step B: Cari baris berdasarkan usia (bulan)
        key = (jk, usia)
        
        if key in who_lookup:
            # Step C: Ambil nilai L, M, S
            params = who_lookup[key]
            L = params['L']
            M = params['M']
            S = params['S']
            
            L_values[idx] = L
            M_values[idx] = M
            S_values[idx] = S
            
            # Step D: Hitung Z-Score WHO (Persamaan II.2)
            # Z = [(x/M)^L - 1] / (L × S)
            if L != 0:
                z = ((x / M) ** L - 1) / (L * S)
            else:
                z = np.log(x / M) / S
            
            z_scores[idx] = round(z, 3)
            
            # Step E: Labelling (Tabel II.2 Proposal)
            if z < -2.0:
                labels[idx] = 'Stunting'
            else:
                labels[idx] = 'Normal'
            
            matched += 1
        else:
            not_matched += 1
    
    # Masukkan hasil ke DataFrame
    df['L'] = L_values
    df['M'] = M_values
    df['S'] = S_values
    df['z_score'] = z_scores
    df['label_who'] = labels
    
    print(f"    ✓ Analisa per baris SELESAI: {matched:,}/{len(df):,} balita berhasil dihitung")
    if not_matched > 0:
        print(f"    ⚠ Tidak cocok dengan tabel WHO: {not_matched} balita")
    
    # ══════════════════════════════════════════════════════════════
    # 3.4: OUTPUT HASIL ANALISA PER BARIS (TABEL LENGKAP)
    # ══════════════════════════════════════════════════════════════
    print_step("3.4", "OUTPUT HASIL ANALISA DATA PER BARIS (SELURUH DATASET)")
    print(f"    Format: No | Sex | Umur (bln) | PB anak (cm) | L (Lambda) | M (Miu) | S (Sigma) | HAZ (WHO-LMS) | Label WHO")
    print(f"    Total: {len(df):,} baris data balita")
    
    # Header tabel
    hdr = (f"    {'No':>4s} │ {'Sex':^10s} │ {'Umur':>5s} │ {'PB anak':>9s} │ "
           f"{'L':>6s} │ {'M':>9s} │ {'S':>8s} │ {'HAZ':>10s} │ {'Label WHO':^10s}")
    sep = f"    {'─'*4}─┼─{'─'*10}─┼─{'─'*5}─┼─{'─'*9}─┼─{'─'*6}─┼─{'─'*9}─┼─{'─'*8}─┼─{'─'*10}─┼─{'─'*10}"
    
    print(f"\n{sep}")
    print(hdr)
    print(sep)
    
    for i in range(len(df)):
        row = df.iloc[i]
        no = i + 1
        jk = int(row['jenis_kelamin'])
        jk_str = 'Laki-laki' if jk == 1 else 'Perempuan'
        usia = int(row['usia_bulan'])
        tb = row['tinggi_badan_cm']
        L = row['L']
        M = row['M']
        S = row['S']
        z = row['z_score']
        lbl = row['label_who']
        
        print(f"    {no:>4d} │ {jk_str:^10s} │ {usia:>5d} │ {tb:>9.2f} │ "
              f"{L:>6.2f} │ {M:>9.3f} │ {S:>8.5f} │ {z:>10.3f} │ {lbl:^10s}")
    
    print(sep)
    print(f"    TOTAL: {len(df):,} baris dianalisa secara individual (per baris)")
    
    # ══════════════════════════════════════════════════════════════
    # 3.5: Simpan Output Analisa Per Baris ke CSV
    # ══════════════════════════════════════════════════════════════
    print_step("3.5", "Menyimpan output analisa per baris ke CSV")
    
    # Buat DataFrame output sesuai format gambar referensi
    df_output = pd.DataFrame({
        'No': range(1, len(df) + 1),
        'Sex': df['jenis_kelamin'].map({1: 'Laki-laki', 2: 'Perempuan'}).values,
        'Umur (bln)': df['usia_bulan'].astype(int).values,
        'PB anak (cm)': df['tinggi_badan_cm'].values,
        'L': df['L'].values,
        'M': df['M'].values,
        'S': df['S'].values,
        'HAZ (WHO-LMS)': df['z_score'].values,
        'Label WHO': df['label_who'].values,
    })
    
    output_analysis_path = os.path.join(TRAINFILE_DIR, 'task3_analisa_perbaris.csv')
    df_output.to_csv(output_analysis_path, index=False)
    print(f"    ✓ Output analisa per baris → {output_analysis_path}")
    print(f"      Kolom: {list(df_output.columns)}")
    print(f"      Baris: {len(df_output):,}")
    
    # ══════════════════════════════════════════════════════════════
    # 3.6: Statistik Hasil Z-Score & Labelling
    # ══════════════════════════════════════════════════════════════
    print_step("3.6", "Statistik Hasil Z-Score & Labelling")
    
    z_valid = df['z_score'].dropna()
    
    print(f"    Z-Score Distribution:")
    print(f"    ├── Count   : {len(z_valid):,}")
    print(f"    ├── Mean    : {z_valid.mean():.3f}")
    print(f"    ├── Std     : {z_valid.std():.3f}")
    print(f"    ├── Min     : {z_valid.min():.3f}")
    print(f"    ├── Q1 (25%): {z_valid.quantile(0.25):.3f}")
    print(f"    ├── Median  : {z_valid.median():.3f}")
    print(f"    ├── Q3 (75%): {z_valid.quantile(0.75):.3f}")
    print(f"    └── Max     : {z_valid.max():.3f}")
    
    # Distribusi Label
    print(f"\n    Distribusi Label WHO (Ground Truth):")
    print(f"    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    label_counts = df['label_who'].value_counts()
    total = len(df)
    for label_name in ['Normal', 'Stunting']:
        if label_name in label_counts.index:
            cnt = label_counts[label_name]
            pct = cnt / total * 100
            bar = '█' * int(pct / 2)
            print(f"    │ {label_name:10s}: {cnt:>4d} ({pct:>5.1f}%) {bar}")
    print(f"    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    print(f"    │ TOTAL     : {total:>4d} (100.0%)")
    
    # Distribusi per jenis kelamin
    print(f"\n    Distribusi Label per Jenis Kelamin:")
    for jk_val, jk_name in [(1, 'Laki-laki'), (2, 'Perempuan')]:
        subset = df[df['jenis_kelamin'] == jk_val]
        stunting_n = (subset['label_who'] == 'Stunting').sum()
        normal_n = (subset['label_who'] == 'Normal').sum()
        total_jk = len(subset)
        print(f"    {jk_name:10s}: Normal={normal_n} ({normal_n/total_jk*100:.1f}%), "
              f"Stunting={stunting_n} ({stunting_n/total_jk*100:.1f}%)")
    
    # Distribusi per kelompok usia
    print(f"\n    Distribusi Label per Kelompok Usia:")
    age_groups = [(0, 5, '0-5 bln'), (6, 11, '6-11 bln'), (12, 17, '12-17 bln'), (18, 23, '18-23 bln')]
    for lo, hi, grp_label in age_groups:
        subset = df[(df['usia_bulan'] >= lo) & (df['usia_bulan'] <= hi)]
        if len(subset) == 0:
            continue
        stunting_n = (subset['label_who'] == 'Stunting').sum()
        normal_n = (subset['label_who'] == 'Normal').sum()
        pct_s = stunting_n / len(subset) * 100
        print(f"    {grp_label:10s} (n={len(subset):>3d}): Normal={normal_n:>3d}, "
              f"Stunting={stunting_n:>3d} ({pct_s:.1f}%)")
    
    # ══════════════════════════════════════════════════════════════
    # 3.7: Validasi Manual (Cross-check rumus)
    # ══════════════════════════════════════════════════════════════
    print_step("3.7", "Validasi Formula (Cross-check rumus manual per baris)")
    print(f"    Mengambil 5 sampel acak dan menghitung ulang secara manual...")
    
    np.random.seed(42)
    check_indices = np.random.choice(len(df), size=5, replace=False)
    
    all_valid = True
    for idx in check_indices:
        row = df.iloc[idx]
        jk = int(row['jenis_kelamin'])
        usia = int(row['usia_bulan'])
        x = float(row['tinggi_badan_cm'])
        L = float(row['L'])
        M = float(row['M'])
        S = float(row['S'])
        z_stored = float(row['z_score'])
        
        if L != 0:
            z_manual = ((x / M) ** L - 1) / (L * S)
        else:
            z_manual = np.log(x / M) / S
        z_manual = round(z_manual, 3)
        
        match = "✓" if z_stored == z_manual else "✗"
        if z_stored != z_manual:
            all_valid = False
        
        jk_str = 'Laki-laki' if jk == 1 else 'Perempuan'
        tbl = 'boys' if jk == 1 else 'girls'
        tbl_range = '02' if usia <= 24 else '25'
        
        print(f"    {match} Baris ke-{idx+1}: {jk_str}, {usia} bln, TB={x:.1f} cm")
        print(f"      Tabel WHO  : {tbl}{tbl_range}.xlsx → Bulan {usia}")
        print(f"      L(Lambda)  : {L}")
        print(f"      M(Miu)     : {M:.4f}")
        print(f"      S(Sigma)   : {S:.5f}")
        print(f"      Perhitungan: Z = [({x:.1f}/{M:.4f})^{L} - 1] / ({L} × {S:.5f})")
        print(f"      Z_program  : {z_stored:.3f}")
        print(f"      Z_manual   : {z_manual:.3f}")
        print(f"      Label      : {row['label_who']} (Z {'<' if z_stored < -2 else '≥'} -2.00)")
        print()
    
    if all_valid:
        print(f"    ✅ Semua {len(check_indices)} validasi manual PASS")
        print(f"    ✅ Konfirmasi: Dataset dianalisa PER BARIS - formula Z-Score benar")
    else:
        print(f"    ⚠ Ada perbedaan - perlu review")
    
    return df


# ============================================================================
# TASK 4: FUZZY SYSTEM LOGIC (Sugeno)
# ============================================================================

# ── 4.a: Fungsi Keanggotaan (Membership Functions) ──
# Berdasarkan grafik proposal: Gambar 3.6 (BBL), 3.7 (PBL), 3.8 (MPASI)

def trapezoid(x, a, b, c, d):
    """Fungsi keanggotaan trapezoidal. μ(x) ∈ [0, 1]."""
    if x <= a or x >= d:
        return 0.0
    elif a < x < b:
        return (x - a) / (b - a)
    elif b <= x <= c:
        return 1.0
    elif c < x < d:
        return (d - x) / (d - c)
    return 0.0

def triangle(x, a, b, c):
    """Fungsi keanggotaan segitiga (kasus khusus trapesium b==c)."""
    return trapezoid(x, a, b, b, c)

# ── FUZZYFIKASI BBL (Gambar 3.6) ──
# Breakpoints dari grafik: 1250, 1500, 2450, 2500, 3650, 3999
def mu_bbl_vlbw(x):       return trapezoid(x, 0, 0, 1250, 1500)
def mu_bbl_lbw(x):        return trapezoid(x, 1250, 1500, 2450, 2500)
def mu_bbl_normal(x):     return trapezoid(x, 2450, 2500, 3650, 3999)
def mu_bbl_macrosomia(x): return trapezoid(x, 3650, 3999, 6000, 6000)

# ── FUZZYFIKASI PBL (Gambar 3.7) ──
# Breakpoints dari grafik: 44, 46, 48, 53, 54
def mu_pbl_veryshort(x):  return trapezoid(x, 0, 0, 44, 46)
def mu_pbl_short(x):      return triangle(x, 44, 46, 48)
def mu_pbl_normal(x):     return trapezoid(x, 46, 48, 53, 54)
def mu_pbl_long(x):       return trapezoid(x, 53, 54, 100, 100)

# ── FUZZYFIKASI MPASI (Gambar 3.8) ──
# Breakpoints dari grafik: 4, 5, 6, 7
def mu_mpasi_early(x):    return trapezoid(x, 0, 0, 4, 5)
def mu_mpasi_timely(x):   return trapezoid(x, 4, 5, 6, 7)
def mu_mpasi_late(x):     return trapezoid(x, 6, 7, 12, 12)


def fuzzify(bbl, pbl, mpasi):
    """Fuzzyfikasi: mengubah nilai tegas ke derajat keanggotaan μ ∈ [0,1]."""
    return {
        'bbl': {
            'VLBW':       mu_bbl_vlbw(bbl),
            'LBW':        mu_bbl_lbw(bbl),
            'Normal':     mu_bbl_normal(bbl),
            'Macrosomia': mu_bbl_macrosomia(bbl),
        },
        'pbl': {
            'VeryShort':  mu_pbl_veryshort(pbl),
            'Short':      mu_pbl_short(pbl),
            'Normal':     mu_pbl_normal(pbl),
            'LongLength': mu_pbl_long(pbl),
        },
        'mpasi': {
            'TooEarly':   mu_mpasi_early(mpasi),
            'Timely':     mu_mpasi_timely(mpasi),
            'Late':       mu_mpasi_late(mpasi),
        }
    }


# ── 4.b: Aturan Dasar (Tabel III.2 Proposal) ──
# Setiap rule: (BBL_set, PBL_set, MPASI_set, consequent)
# None = wildcard (variabel tidak ikut dalam rule)
# Tuple = OR condition, misalnya ('Late', 'TooEarly')

FUZZY_RULES = [
    # ═══════ STUNTING HIGH (c=90) ═══════
    ('VLBW',  'VeryShort', None,                   90),
    ('VLBW',  None,        'Late',                 90),
    ('VLBW',  None,        'TooEarly',             90),
    (None,    'VeryShort', 'Late',                 90),
    (None,    'VeryShort', 'TooEarly',             90),
    ('VLBW',  'Short',     ('Late', 'TooEarly'),   90),
    ('VLBW',  'VeryShort', ('Late', 'TooEarly'),   90),

    # ═══════ STUNTING MEDIUM (c=65) ═══════
    ('LBW',   'Short',     None,                   65),
    ('LBW',   None,        ('Late', 'TooEarly'),   65),
    (None,    'Short',     ('Late', 'TooEarly'),   65),
    ('VLBW',  'Normal',    None,                   65),
    ('VLBW',  None,        'Timely',               65),
    ('Normal','VeryShort', None,                   65),
    (None,    'VeryShort', 'Timely',               65),

    # ═══════ STUNTING LOW (c=35) ═══════
    ('Normal','Normal',    'Timely',               35),
    ('Normal','LongLength','Timely',               35),
    ('Macrosomia','Normal','Timely',               35),
    ('Normal','Normal',    ('TooEarly', 'Late'),   35),
    ('Normal',None,        'Timely',               35),  # BBL=Normal, MPASI=Timely (any PBL Short)
    ('LBW',  'Normal',     'Timely',               35),

    # ═══════ NORMAL / NO RISK (c=10) ═══════
    ('Normal',    'LongLength', 'Timely',          10),
    ('Macrosomia','Normal',     'Timely',          10),
    ('Macrosomia','LongLength', 'Timely',          10),
]


def evaluate_rules(mu):
    """
    Evaluasi semua aturan fuzzy (Tabel III.2).
    Operator AND = MIN, Operator OR = MAX.
    Returns: list of (weight, consequent) tuples.
    """
    fired = []

    for rule in FUZZY_RULES:
        bbl_cond, pbl_cond, mpasi_cond, consequent = rule
        antecedents = []

        # BBL condition
        if bbl_cond is not None:
            if isinstance(bbl_cond, tuple):
                antecedents.append(max(mu['bbl'].get(c, 0) for c in bbl_cond))
            else:
                antecedents.append(mu['bbl'].get(bbl_cond, 0))

        # PBL condition
        if pbl_cond is not None:
            if isinstance(pbl_cond, tuple):
                antecedents.append(max(mu['pbl'].get(c, 0) for c in pbl_cond))
            else:
                antecedents.append(mu['pbl'].get(pbl_cond, 0))

        # MPASI condition
        if mpasi_cond is not None:
            if isinstance(mpasi_cond, tuple):
                antecedents.append(max(mu['mpasi'].get(c, 0) for c in mpasi_cond))
            else:
                antecedents.append(mu['mpasi'].get(mpasi_cond, 0))

        # AND = MIN of all antecedents
        if antecedents:
            w = min(antecedents)
            if w > 0:
                fired.append((w, consequent))

    return fired


def defuzzify_sugeno(fired_rules):
    """
    Defuzzyfikasi Sugeno: Weighted Average.
    Risk = Σ(wᵢ × cᵢ) / Σ(wᵢ)
    """
    if not fired_rules:
        return 35.0  # Default: Stunting Low jika tidak ada rule yang aktif

    numerator = sum(w * c for w, c in fired_rules)
    denominator = sum(w for w, c in fired_rules)

    if denominator == 0:
        return 35.0

    return round(numerator / denominator, 2)


def get_priority_flag(risk_score):
    """
    Mapping Risk Score → Priority Flag (Gambar 3.9).
    0-25   : Normal
    25-50  : STUNTING LOW
    50-75  : STUNTING MEDIUM
    75-100 : STUNTING HIGH
    """
    if risk_score <= 25:
        return 'Normal'
    elif risk_score <= 50:
        return 'STUNTING LOW'
    elif risk_score <= 75:
        return 'STUNTING MEDIUM'
    else:
        return 'STUNTING HIGH'


def task4_fuzzy_logic(df):
    """
    Task 4: Fuzzy System Logic (Takagi-Sugeno).
    Input: BBL (gram), PBL (cm), MPASI (bulan)
    Output: Risk Score (0-100), Priority Flag
    """

    print_header("TASK 4: FUZZY SYSTEM LOGIC - Takagi-Sugeno")

    # ── 4.1: Penjelasan Fungsi Keanggotaan ──
    print_step("4.1", "Fungsi Keanggotaan (Fuzzyfikasi)")
    print(f"""
    ┌────────────────────────────────────────────────────────────────┐
    │  BBL (Gambar 3.6) - Trapezoidal                                │
    │  • VLBW         : trap(0, 0, 1250, 1500)    [<1500g]          │
    │  • LBW          : trap(1250, 1500, 2450, 2500) [1500-2500g]   │
    │  • Normal       : trap(2450, 2500, 3650, 3999) [2500-4000g]   │
    │  • Macrosomia   : trap(3650, 3999, 6000, 6000) [>4000g]       │
    ├────────────────────────────────────────────────────────────────┤
    │  PBL (Gambar 3.7) - Trapezoidal + Triangle                    │
    │  • Very Short   : trap(0, 0, 44, 46)         [<46cm]          │
    │  • Short        : triangle(44, 46, 48)        [44-48cm]       │
    │  • Normal       : trap(46, 48, 53, 54)        [48-53cm]       │
    │  • Long Length   : trap(53, 54, 100, 100)     [>54cm]         │
    ├────────────────────────────────────────────────────────────────┤
    │  MP-ASI (Gambar 3.8) - Trapezoidal                            │
    │  • Too Early    : trap(0, 0, 4, 5)            [<5 bln]        │
    │  • Timely       : trap(4, 5, 6, 7)            [5-6 bln]       │
    │  • Late         : trap(6, 7, 12, 12)          [>7 bln]        │
    └────────────────────────────────────────────────────────────────┘
    """)

    # ── 4.2: Aturan Dasar ──
    print_step("4.2", f"Aturan Dasar (Rule Base) - {len(FUZZY_RULES)} aturan")
    risk_labels = {90: 'STUNTING HIGH', 65: 'STUNTING MEDIUM', 35: 'STUNTING LOW', 10: 'Normal'}
    for c_val in [90, 65, 35, 10]:
        rules_c = [(i+1, r) for i, r in enumerate(FUZZY_RULES) if r[3] == c_val]
        print(f"    {risk_labels[c_val]} (c={c_val}): {len(rules_c)} aturan")

    # ── 4.3: Defuzzyfikasi Sugeno ──
    print_step("4.3", "Defuzzyfikasi Sugeno - Weighted Average")
    print(f"    Rumus: Risk = Σ(wᵢ × cᵢ) / Σ(wᵢ)")
    print(f"    wᵢ = bobot aktivasi (operator AND = MIN)")
    print(f"    cᵢ = konstanta konsekuen (10, 35, 65, 90)")

    # ── 4.4: Eksekusi Fuzzy per Baris ──
    print_step("4.4", f"ANALISA FUZZY PER BARIS - {len(df):,} balita")

    risk_scores = []
    priority_flags = []
    rules_fired_counts = []

    for idx in range(len(df)):
        bbl  = float(df.iloc[idx]['bbl_gram'])
        pbl  = float(df.iloc[idx]['pbl_cm'])
        mpasi = float(df.iloc[idx]['mpasi_bulan'])

        # Step A: Fuzzyfikasi
        mu = fuzzify(bbl, pbl, mpasi)

        # Step B: Evaluasi aturan
        fired = evaluate_rules(mu)

        # Step C: Defuzzyfikasi Sugeno
        risk = defuzzify_sugeno(fired)

        # Step D: Mapping Priority Flag
        flag = get_priority_flag(risk)

        risk_scores.append(risk)
        priority_flags.append(flag)
        rules_fired_counts.append(len(fired))

    df['risk_score'] = risk_scores
    df['priority_flag'] = priority_flags

    print(f"    ✓ Fuzzy inference selesai: {len(df):,} balita")
    print(f"    ✓ Rata-rata rules fired per balita: {np.mean(rules_fired_counts):.1f}")

    # ── 4.5: Statistik Risk Score & Priority Flag ──
    print_step("4.5", "Statistik Risk Score & Priority Flag")

    rs = df['risk_score']
    print(f"    Risk Score Distribution:")
    print(f"    ├── Mean    : {rs.mean():.2f}")
    print(f"    ├── Std     : {rs.std():.2f}")
    print(f"    ├── Min     : {rs.min():.2f}")
    print(f"    ├── Median  : {rs.median():.2f}")
    print(f"    └── Max     : {rs.max():.2f}")

    print(f"\n    Distribusi Priority Flag:")
    print(f"    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    total = len(df)
    for flag_name in ['Normal', 'STUNTING LOW', 'STUNTING MEDIUM', 'STUNTING HIGH']:
        cnt = (df['priority_flag'] == flag_name).sum()
        pct = cnt / total * 100
        bar = '█' * int(pct / 2)
        print(f"    │ {flag_name:18s}: {cnt:>4d} ({pct:>5.1f}%) {bar}")
    print(f"    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

    # Cross-tab: Label WHO vs Priority Flag
    print(f"\n    Cross-tabulation: Label WHO × Priority Flag:")
    for who_label in ['Normal', 'Stunting']:
        subset = df[df['label_who'] == who_label]
        flags = subset['priority_flag'].value_counts()
        parts = [f"{f}={flags.get(f, 0)}" for f in ['Normal', 'STUNTING LOW', 'STUNTING MEDIUM', 'STUNTING HIGH']]
        print(f"    WHO {who_label:8s} (n={len(subset):>3d}): {', '.join(parts)}")

    # ── 4.6: OUTPUT TABEL ANALISA LENGKAP PER BARIS ──
    print_step("4.6", "OUTPUT TABEL ANALISA LENGKAP PER BARIS (Z-Score + Fuzzy)")

    hdr = (f"    {'No':>4s} │ {'Sex':^10s} │ {'Umur':>4s} │ {'PB':>7s} │ "
           f"{'L':>5s} │ {'M':>9s} │ {'S':>8s} │ {'HAZ':>8s} │ {'Label':^8s} │ "
           f"{'BBL':>6s} │ {'PBL':>6s} │ {'MPASI':>5s} │ {'Risk':>6s} │ {'Priority Flag':^16s}")
    sep = (f"    {'─'*4}─┼─{'─'*10}─┼─{'─'*4}─┼─{'─'*7}─┼─"
           f"{'─'*5}─┼─{'─'*9}─┼─{'─'*8}─┼─{'─'*8}─┼─{'─'*8}─┼─"
           f"{'─'*6}─┼─{'─'*6}─┼─{'─'*5}─┼─{'─'*6}─┼─{'─'*16}")

    print(f"\n{sep}")
    print(hdr)
    print(sep)

    for i in range(len(df)):
        row = df.iloc[i]
        no = i + 1
        jk_str = 'Laki-laki' if int(row['jenis_kelamin']) == 1 else 'Perempuan'
        usia = int(row['usia_bulan'])
        tb = row['tinggi_badan_cm']
        L = row['L']
        M = row['M']
        S = row['S']
        z = row['z_score']
        lbl = row['label_who']
        bbl = row['bbl_gram']
        pbl = row['pbl_cm']
        mpasi = row['mpasi_bulan']
        risk = row['risk_score']
        flag = row['priority_flag']

        print(f"    {no:>4d} │ {jk_str:^10s} │ {usia:>4d} │ {tb:>7.2f} │ "
              f"{L:>5.2f} │ {M:>9.3f} │ {S:>8.5f} │ {z:>8.3f} │ {lbl:^8s} │ "
              f"{bbl:>6.0f} │ {pbl:>6.2f} │ {mpasi:>5.2f} │ {risk:>6.2f} │ {flag:^16s}")

    print(sep)
    print(f"    TOTAL: {len(df):,} baris dianalisa (Z-Score WHO + Fuzzy Sugeno)")

    # ── 4.7: Simpan Output ──
    print_step("4.7", "Menyimpan output analisa lengkap per baris")

    df_output = pd.DataFrame({
        'No': range(1, len(df) + 1),
        'Sex': df['jenis_kelamin'].map({1: 'Laki-laki', 2: 'Perempuan'}).values,
        'Umur (bln)': df['usia_bulan'].astype(int).values,
        'PB anak (cm)': df['tinggi_badan_cm'].values,
        'L': df['L'].values,
        'M': df['M'].values,
        'S': df['S'].values,
        'HAZ (WHO-LMS)': df['z_score'].values,
        'Label WHO': df['label_who'].values,
        'BBL (g)': df['bbl_gram'].values,
        'PBL (cm)': df['pbl_cm'].values,
        'MPASI (bln)': df['mpasi_bulan'].values,
        'Risk score (Sugeno)': df['risk_score'].values,
        'Priority flag': df['priority_flag'].values,
    })

    output_path = os.path.join(TRAINFILE_DIR, 'task3_analisa_perbaris.csv')
    df_output.to_csv(output_path, index=False)
    print(f"    ✓ Tabel analisa lengkap → {output_path}")
    print(f"      Kolom: {list(df_output.columns)}")

    return df



# ============================================================================
# TASK 5 & 6: EVALUASI + SKEMA PENGUJIAN (TRAIN/TEST)
# ============================================================================
#
# KONSEP ITERATIF (Forward-Backward) sesuai catatan revisi:
# MAJU  : Jalankan fuzzy pada 80% data -> cek vs ground truth
# MUNDUR: Perbaiki parameter fuzzy (MF BBL/PBL/MPASI + consequent + threshold)
# Ground Truth: WHO Normal -> Flag Normal | WHO Stunting -> Flag Stunting
# Iterasi: random split tiap run, ulangi sampai akurasi baik
# Confusion Matrix: cek dari hasil akhir 80:20

def run_fuzzy_full(dataframe, mf_p, cons):
    """Jalankan fuzzy inference lengkap dengan parameter tunable."""
    bp, pp, mp = mf_p['bbl'], mf_p['pbl'], mf_p['mpasi']
    c_h, c_m, c_l, c_n = cons['high'], cons['medium'], cons['low'], cons['normal']
    
    rules = [
        ('VLBW','VeryShort',None,c_h), ('VLBW',None,'Late',c_h),
        ('VLBW',None,'TooEarly',c_h), (None,'VeryShort','Late',c_h),
        (None,'VeryShort','TooEarly',c_h),
        ('VLBW','Short',('Late','TooEarly'),c_h),
        ('VLBW','VeryShort',('Late','TooEarly'),c_h),
        ('LBW','Short',None,c_m), ('LBW',None,('Late','TooEarly'),c_m),
        (None,'Short',('Late','TooEarly'),c_m),
        ('VLBW','Normal',None,c_m), ('VLBW',None,'Timely',c_m),
        ('Normal','VeryShort',None,c_m), (None,'VeryShort','Timely',c_m),
        ('Normal','Normal','Timely',c_l), ('Normal','LongLength','Timely',c_l),
        ('Macrosomia','Normal','Timely',c_l),
        ('Normal','Normal',('TooEarly','Late'),c_l),
        ('Normal',None,'Timely',c_l), ('LBW','Normal','Timely',c_l),
        ('Normal','LongLength','Timely',c_n),
        ('Macrosomia','Normal','Timely',c_n),
        ('Macrosomia','LongLength','Timely',c_n),
    ]
    
    scores = []
    for idx in range(len(dataframe)):
        bbl = float(dataframe.iloc[idx]['bbl_gram'])
        pbl = float(dataframe.iloc[idx]['pbl_cm'])
        mpasi = float(dataframe.iloc[idx]['mpasi_bulan'])
        
        mu = {
            'bbl': {
                'VLBW': trapezoid(bbl, 0, 0, bp[0], bp[1]),
                'LBW': trapezoid(bbl, bp[0], bp[1], bp[2], bp[3]),
                'Normal': trapezoid(bbl, bp[2], bp[3], bp[4], bp[5]),
                'Macrosomia': trapezoid(bbl, bp[4], bp[5], 6000, 6000),
            },
            'pbl': {
                'VeryShort': trapezoid(pbl, 0, 0, pp[0], pp[1]),
                'Short': triangle(pbl, pp[0], pp[1], pp[2]),
                'Normal': trapezoid(pbl, pp[1], pp[2], pp[3], pp[4]),
                'LongLength': trapezoid(pbl, pp[3], pp[4], 100, 100),
            },
            'mpasi': {
                'TooEarly': trapezoid(mpasi, 0, 0, mp[0], mp[1]),
                'Timely': trapezoid(mpasi, mp[0], mp[1], mp[2], mp[3]),
                'Late': trapezoid(mpasi, mp[2], mp[3], 12, 12),
            }
        }
        
        fired = []
        for rule in rules:
            b_c, p_c, m_c, c_val = rule
            antes = []
            if b_c is not None:
                antes.append(max(mu['bbl'].get(x,0) for x in b_c) if isinstance(b_c,tuple) else mu['bbl'].get(b_c,0))
            if p_c is not None:
                antes.append(max(mu['pbl'].get(x,0) for x in p_c) if isinstance(p_c,tuple) else mu['pbl'].get(p_c,0))
            if m_c is not None:
                antes.append(max(mu['mpasi'].get(x,0) for x in m_c) if isinstance(m_c,tuple) else mu['mpasi'].get(m_c,0))
            if antes:
                w = min(antes)
                if w > 0:
                    fired.append((w, c_val))
        
        if fired:
            num = sum(w * c for w, c in fired)
            den = sum(w for w, c in fired)
            risk = round(num / den, 2) if den > 0 else c_l
        else:
            risk = c_l
        scores.append(risk)
    
    return np.array(scores)


def calc_metrics(y_true, y_pred):
    """Confusion matrix + semua metrik evaluasi."""
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    prec = tp/(tp+fp) if (tp+fp) > 0 else 0.0
    rec = tp/(tp+fn) if (tp+fn) > 0 else 0.0
    f1 = 2*prec*rec/(prec+rec) if (prec+rec) > 0 else 0.0
    acc = (tp+tn)/(tp+tn+fp+fn) if (tp+tn+fp+fn) > 0 else 0.0
    return {'TP':tp,'TN':tn,'FP':fp,'FN':fn,
            'precision':round(prec,4),'recall':round(rec,4),
            'f1':round(f1,4),'accuracy':round(acc,4)}


def task5_6_evaluation(df):
    """
    Task 5+6: Iterative Forward-Backward Training.
    MAJU: run fuzzy 80% -> cek ground truth
    MUNDUR: perbaiki MF + consequent + threshold
    Tiap run: random split baru
    """
    print_header("TASK 5 & 6: EVALUASI + SKEMA PENGUJIAN SISTEM")
    
    print(f"""
    ╔═══════════════════════════════════════════════════════════════╗
    ║  KONSEP: Iteratif Forward-Backward Training                  ║
    ║  Ground Truth:                                                ║
    ║ - WHO Normal  -> Priority Flag HARUS Normal                 ║
    ║ - WHO Stunting -> Priority Flag HARUS Stunting (L/M/H)      ║
    ║  MAJU  : Jalankan fuzzy 80% -> cek vs ground truth           ║
    ║  MUNDUR: Perbaiki MF + Consequent + Threshold                ║
    ║  Iterasi: tiap run random split baru (5 seeds)               ║
    ╚═══════════════════════════════════════════════════════════════╝
    """)
    
    y_all = (df['label_who'] == 'Stunting').astype(int).values

    # --- SEARCH SPACE (otak-atik variabel fuzzy) ---
    # KUNCI: Geser batas LBW-Normal dan Short-Normal agar perbedaan
    # kecil antara Stunting (BBL=2970, PBL=48.4) vs Normal (BBL=3139, PBL=49.4)
    # bisa terdeteksi oleh fuzzy sebagai kategori BERBEDA
    
    BBL_GRID = [
        [1250, 1500, 2450, 2500, 3650, 3999],   # Default proposal
        [1250, 1500, 2800, 2900, 3500, 3900],   # LBW sampai 2900
        [1250, 1500, 2900, 3000, 3500, 3900],   # LBW sampai 3000
        [1250, 1500, 3000, 3100, 3500, 3900],   # LBW sampai 3100 (aggressive)
        [1250, 1500, 2700, 2800, 3400, 3800],   # Wider LBW
        [1250, 1500, 2950, 3050, 3500, 3900],   # Midpoint (Stunting=2970, Normal=3139)
        [1250, 1500, 3050, 3150, 3500, 3900],   # Slightly above midpoint
    ]
    PBL_GRID = [
        [44, 46, 48, 53, 54],                    # Default proposal
        [44, 47, 49, 52, 54],                    # Short peak=47, Normal=49
        [44, 48, 50, 53, 54],                    # Short peak=48, Normal=50
        [44, 49, 51, 53, 54],                    # Aggressive: Short peak=49
        [44, 48, 49, 52, 53],                    # Narrow Normal (49-52)
    ]
    MPASI_GRID = [
        [4, 5, 6, 7],                            # Default
        [3, 4, 6, 8],                            # Wide TooEarly + Late
        [3, 5, 7, 8],                            # Wide ranges
    ]
    CONS_GRID = [
        {'high': 90, 'medium': 65, 'low': 35, 'normal': 10},   # Default
        {'high': 95, 'medium': 70, 'low': 15, 'normal': 3},    # Max spread
        {'high': 90, 'medium': 60, 'low': 10, 'normal': 2},    # Low baseline
        {'high': 95, 'medium': 75, 'low': 20, 'normal': 5},
        {'high': 92, 'medium': 68, 'low': 12, 'normal': 3},
        {'high': 90, 'medium': 55, 'low': 8,  'normal': 2},    # Very low baseline
    ]
    
    SEEDS = [42, 123, 256, 789, 1024]
    THRESHOLDS = np.arange(5, 85, 0.5)
    
    n_mf = len(BBL_GRID) * len(PBL_GRID) * len(MPASI_GRID)
    n_total = n_mf * len(CONS_GRID) * len(THRESHOLDS) * len(SEEDS)
    print_step("6.1", "Search Space")
    print(f"    MF configs: {n_mf} | Consequent sets: {len(CONS_GRID)}")
    print(f"    Thresholds: {len(THRESHOLDS)} | Seeds: {len(SEEDS)}")
    print(f"    Total: {n_total:,} evaluasi")
    
    # --- ITERATIVE FORWARD-BACKWARD ---
    print_step("6.2", "Iterative Forward-Backward Training")
    
    global_best = {'combo_score': -1}
    
    for seed in SEEDS:
        np.random.seed(seed)
        idx_n = np.where(y_all == 0)[0].copy()
        idx_s = np.where(y_all == 1)[0].copy()
        np.random.shuffle(idx_n)
        np.random.shuffle(idx_s)
        
        n_tr_n = int(len(idx_n) * 0.8)
        n_tr_s = int(len(idx_s) * 0.8)
        tr_idx = np.concatenate([idx_n[:n_tr_n], idx_s[:n_tr_s]])
        te_idx = np.concatenate([idx_n[n_tr_n:], idx_s[n_tr_s:]])
        
        df_tr = df.iloc[tr_idx].reset_index(drop=True)
        df_te = df.iloc[te_idx].reset_index(drop=True)
        y_tr = (df_tr['label_who'] == 'Stunting').astype(int).values
        y_te = (df_te['label_who'] == 'Stunting').astype(int).values
        
        seed_best = {'train_score': -1}
        
        # MAJU: forward pass semua kombinasi
        for bp in BBL_GRID:
            for pp in PBL_GRID:
                for mp in MPASI_GRID:
                    mf = {'bbl': bp, 'pbl': pp, 'mpasi': mp}
                    
                    # Compute rule weights ONCE per MF config (expensive part)
                    all_weights = []  # list of [(w, rule_class), ...] per row
                    for idx in range(len(df_tr)):
                        bbl = float(df_tr.iloc[idx]['bbl_gram'])
                        pbl = float(df_tr.iloc[idx]['pbl_cm'])
                        mpasi = float(df_tr.iloc[idx]['mpasi_bulan'])
                        
                        mu = {
                            'bbl': {
                                'VLBW': trapezoid(bbl, 0, 0, bp[0], bp[1]),
                                'LBW': trapezoid(bbl, bp[0], bp[1], bp[2], bp[3]),
                                'Normal': trapezoid(bbl, bp[2], bp[3], bp[4], bp[5]),
                                'Macrosomia': trapezoid(bbl, bp[4], bp[5], 6000, 6000),
                            },
                            'pbl': {
                                'VeryShort': trapezoid(pbl, 0, 0, pp[0], pp[1]),
                                'Short': triangle(pbl, pp[0], pp[1], pp[2]),
                                'Normal': trapezoid(pbl, pp[1], pp[2], pp[3], pp[4]),
                                'LongLength': trapezoid(pbl, pp[3], pp[4], 100, 100),
                            },
                            'mpasi': {
                                'TooEarly': trapezoid(mpasi, 0, 0, mp[0], mp[1]),
                                'Timely': trapezoid(mpasi, mp[0], mp[1], mp[2], mp[3]),
                                'Late': trapezoid(mpasi, mp[2], mp[3], 12, 12),
                            }
                        }
                        
                        # Rule templates: (bbl_cond, pbl_cond, mpasi_cond, class)
                        # class: 'H','M','L','N'
                        RULE_TEMPLATES = [
                            ('VLBW','VeryShort',None,'H'), ('VLBW',None,'Late','H'),
                            ('VLBW',None,'TooEarly','H'), (None,'VeryShort','Late','H'),
                            (None,'VeryShort','TooEarly','H'),
                            ('VLBW','Short',('Late','TooEarly'),'H'),
                            ('VLBW','VeryShort',('Late','TooEarly'),'H'),
                            ('LBW','Short',None,'M'), ('LBW',None,('Late','TooEarly'),'M'),
                            (None,'Short',('Late','TooEarly'),'M'),
                            ('VLBW','Normal',None,'M'), ('VLBW',None,'Timely','M'),
                            ('Normal','VeryShort',None,'M'), (None,'VeryShort','Timely','M'),
                            ('Normal','Normal','Timely','L'), ('Normal','LongLength','Timely','L'),
                            ('Macrosomia','Normal','Timely','L'),
                            ('Normal','Normal',('TooEarly','Late'),'L'),
                            ('Normal',None,'Timely','L'), ('LBW','Normal','Timely','L'),
                            ('Normal','LongLength','Timely','N'),
                            ('Macrosomia','Normal','Timely','N'),
                            ('Macrosomia','LongLength','Timely','N'),
                        ]
                        
                        row_weights = []
                        for b_c, p_c, m_c, cls in RULE_TEMPLATES:
                            antes = []
                            if b_c is not None:
                                antes.append(max(mu['bbl'].get(x,0) for x in b_c) if isinstance(b_c,tuple) else mu['bbl'].get(b_c,0))
                            if p_c is not None:
                                antes.append(max(mu['pbl'].get(x,0) for x in p_c) if isinstance(p_c,tuple) else mu['pbl'].get(p_c,0))
                            if m_c is not None:
                                antes.append(max(mu['mpasi'].get(x,0) for x in m_c) if isinstance(m_c,tuple) else mu['mpasi'].get(m_c,0))
                            if antes:
                                w = min(antes)
                                if w > 0:
                                    row_weights.append((w, cls))
                        all_weights.append(row_weights)
                    
                    # Now iterate cons+threshold QUICKLY (no fuzzy recomputation)
                    for cons in CONS_GRID:
                        c_map = {'H': cons['high'], 'M': cons['medium'], 
                                 'L': cons['low'], 'N': cons['normal']}
                        
                        # Compute risk scores from pre-computed weights
                        train_scores = np.zeros(len(df_tr))
                        for i, row_w in enumerate(all_weights):
                            if row_w:
                                num = sum(w * c_map[cls] for w, cls in row_w)
                                den = sum(w for w, cls in row_w)
                                train_scores[i] = num / den if den > 0 else cons['low']
                            else:
                                train_scores[i] = cons['low']
                        
                        # MUNDUR: cari threshold optimal
                        for thr in THRESHOLDS:
                            y_pred = (train_scores >= thr).astype(int)
                            m = calc_metrics(y_tr, y_pred)
                            # F1-Score = indikator utama (Proposal Bab 3.3.5)
                            # Constraint: accuracy minimal 70% agar tidak terlalu banyak FP
                            if m['accuracy'] >= 0.70:
                                score = m['f1']
                            else:
                                score = m['f1'] * 0.5  # penalti
                            
                            if score > seed_best['train_score']:
                                seed_best = {
                                    'train_score': score, 'mf': mf.copy(),
                                    'cons': cons.copy(), 'thr': thr,
                                    'train_m': m.copy(), 'seed': seed,
                                    'tr_idx': tr_idx, 'te_idx': te_idx,
                                }
        
        # Validasi pada test set
        te_scores = run_fuzzy_full(df_te, seed_best['mf'], seed_best['cons'])
        y_pred_te = (te_scores >= seed_best['thr']).astype(int)
        test_m = calc_metrics(y_te, y_pred_te)
        
        # Combo: F1 dominan, accuracy sebagai filter
        if test_m['accuracy'] >= 0.70:
            combo = 0.3 * seed_best['train_score'] + 0.7 * test_m['f1']
        else:
            combo = 0.3 * seed_best['train_score'] + 0.7 * test_m['f1'] * 0.5
        
        trn = seed_best['train_m']
        print(f"    Seed {seed:>4d} | Train: Acc={trn['accuracy']:.4f} F1={trn['f1']:.4f} | "
              f"Test: Acc={test_m['accuracy']:.4f} F1={test_m['f1']:.4f} | thr={seed_best['thr']:.1f}")
        
        if combo > global_best['combo_score']:
            global_best = seed_best.copy()
            global_best['combo_score'] = combo
            global_best['test_m'] = test_m
            global_best['y_tr'] = y_tr
            global_best['y_te'] = y_te
            global_best['df_tr'] = df_tr
            global_best['df_te'] = df_te
    
    best = global_best
    
    print_step("6.3", "BEST MODEL")
    print(f"""
    ┌───────────────────────────────────────────────────────────┐
    │  BBL MF     : {best['mf']['bbl']}  │
    │  PBL MF     : {best['mf']['pbl']}             │
    │  MPASI MF   : {best['mf']['mpasi']}                    │
    │  Consequents: H={best['cons']['high']} M={best['cons']['medium']} L={best['cons']['low']} N={best['cons']['normal']}          │
    │  Threshold  : {best['thr']:.1f}                                  │
    │  Seed       : {best['seed']}                                     │
    └───────────────────────────────────────────────────────────┘
    """)
    
    # Re-compute seluruh data
    print_step("6.4", "Re-compute seluruh data")
    all_scores = run_fuzzy_full(df, best['mf'], best['cons'])
    df['risk_score'] = np.round(all_scores, 2)
    df['priority_flag'] = [get_priority_flag(s) for s in all_scores]
    df['fuzzy_label'] = ['Stunting' if s >= best['thr'] else 'Normal' for s in all_scores]
    print(f"    Threshold: risk >= {best['thr']:.1f} -> Stunting")
    
    # === EVALUASI ===
    print_step("5.1", f"CONFUSION MATRIX - Train ({len(best['y_tr'])})")
    tr_sc = run_fuzzy_full(best['df_tr'], best['mf'], best['cons'])
    trm = calc_metrics(best['y_tr'], (tr_sc >= best['thr']).astype(int))
    tp,tn,fp,fn = trm['TP'],trm['TN'],trm['FP'],trm['FN']
    print(f"""
    ╔═════════════════════════════════════════════════════════╗
    ║         CONFUSION MATRIX - TRAIN SET (80%)             ║
    ╠═════════════════╦══════════════╦════════════════════════╣
    ║                 ║ Act Stunting ║ Act Normal             ║
    ╠═════════════════╬══════════════╬════════════════════════╣
    ║ Pred Stunting   ║  TP={tp:>4d}    ║  FP={fp:>4d}               ║
    ║ Pred Normal     ║  FN={fn:>4d}    ║  TN={tn:>4d}               ║
    ╠═════════════════╩══════════════╩════════════════════════╣
    ║ Prec={trm['precision']:.4f} Rec={trm['recall']:.4f} F1={trm['f1']:.4f} Acc={trm['accuracy']:.4f}  ║
    ╚═════════════════════════════════════════════════════════╝""")
    
    print_step("5.2", f"CONFUSION MATRIX - Test ({len(best['y_te'])})")
    tm = best['test_m']
    tp,tn,fp,fn = tm['TP'],tm['TN'],tm['FP'],tm['FN']
    print(f"""
    ╔═════════════════════════════════════════════════════════╗
    ║         CONFUSION MATRIX - TEST SET (20%)              ║
    ║       (Data independen, tidak terlihat saat train)      ║
    ╠═════════════════╦══════════════╦════════════════════════╣
    ║                 ║ Act Stunting ║ Act Normal             ║
    ╠═════════════════╬══════════════╬════════════════════════╣
    ║ Pred Stunting   ║  TP={tp:>4d}    ║  FP={fp:>4d}               ║
    ║ Pred Normal     ║  FN={fn:>4d}    ║  TN={tn:>4d}               ║
    ╠═════════════════╩══════════════╩════════════════════════╣
    ║ Prec={tm['precision']:.4f} Rec={tm['recall']:.4f} F1={tm['f1']:.4f} Acc={tm['accuracy']:.4f}  ║
    ╚═════════════════════════════════════════════════════════╝""")
    
    print_step("5.3", "CONFUSION MATRIX - All Data")
    y_all_t = (df['label_who']=='Stunting').astype(int).values
    y_all_p = (df['fuzzy_label']=='Stunting').astype(int).values
    am = calc_metrics(y_all_t, y_all_p)
    tp,tn,fp,fn = am['TP'],am['TN'],am['FP'],am['FN']
    print(f"""
    ╔═════════════════════════════════════════════════════════╗
    ║         CONFUSION MATRIX - SELURUH DATA ({len(df)})          ║
    ╠═════════════════╦══════════════╦════════════════════════╣
    ║ Pred Stunting   ║  TP={tp:>4d}    ║  FP={fp:>4d}               ║
    ║ Pred Normal     ║  FN={fn:>4d}    ║  TN={tn:>4d}               ║
    ╠═════════════════╩══════════════╩════════════════════════╣
    ║ Prec={am['precision']:.4f} Rec={am['recall']:.4f} F1={am['f1']:.4f} Acc={am['accuracy']:.4f}  ║
    ╚═════════════════════════════════════════════════════════╝""")
    
    print_step("5.4", "Ringkasan")
    print(f"    ┌──────────────┬───────────┬──────────┬──────────┬──────────┐")
    print(f"    │   Dataset    │ Precision │  Recall  │ F1-Score │ Accuracy │")
    print(f"    ├──────────────┼───────────┼──────────┼──────────┼──────────┤")
    print(f"    │ Train (80%)  │  {trm['precision']:.4f}   │  {trm['recall']:.4f} │  {trm['f1']:.4f} │  {trm['accuracy']:.4f} │")
    print(f"    │ Test  (20%)  │  {tm['precision']:.4f}   │  {tm['recall']:.4f} │  {tm['f1']:.4f} │  {tm['accuracy']:.4f} │")
    print(f"    │ All  (100%)  │  {am['precision']:.4f}   │  {am['recall']:.4f} │  {am['f1']:.4f} │  {am['accuracy']:.4f} │")
    print(f"    └──────────────┴───────────┴──────────┴──────────┴──────────┘")
    
    print_step("5.5", "Simpan model")
    trained_model = {
        'mf_params': best['mf'], 'consequents': best['cons'],
        'threshold': float(best['thr']), 'seed': int(best['seed']),
        'metrics': {'train': trm, 'test': tm, 'all': am},
        'split': {'train_size': len(best['y_tr']), 'test_size': len(best['y_te'])},
        'timestamp': datetime.now().isoformat(),
    }
    model_path = os.path.join(TRAINFILE_DIR, 'trained_model.json')
    with open(model_path, 'w') as f:
        json.dump(trained_model, f, indent=2, default=str)
    print(f"    -> {model_path}")
    
    df_tr_s = df.iloc[best['tr_idx']].reset_index(drop=True)
    df_te_s = df.iloc[best['te_idx']].reset_index(drop=True)
    df_tr_s.to_csv(os.path.join(TRAINFILE_DIR, 'dataset_train.csv'), index=False)
    df_te_s.to_csv(os.path.join(TRAINFILE_DIR, 'dataset_test.csv'), index=False)
    print(f"    -> dataset_train.csv ({len(df_tr_s)}), dataset_test.csv ({len(df_te_s)})")
    
    return df, trained_model

if __name__ == '__main__':
    
    print("\n" + "█" * 75)
    print("█  STUNTOR - Sistem Klasifikasi Stunting Balita Kalimantan Tengah    █")
    print("█  Dataset: SKI 2023 | Metode: Fuzzy Sugeno                          █")
    print("█" * 75)
    print(f"\n  Working directory : {os.getcwd()}")
    print(f"  Script location   : {os.path.abspath(__file__)}")
    print(f"  Dataset path      : {DATASET_PATH}")
    print(f"  WHO Z-Score dir   : {WHO_ZS_DIR}")
    print(f"  Output dir        : {TRAINFILE_DIR}")
    
    # ─── TASK 1: INPUT DATA ───
    df_input, raw_shape = task1_input_data()
    
    # ─── TASK 2: PREPROCESSING ───
    df_clean, report = task2_preprocessing(df_input, raw_shape)
    
    df_clean.to_csv(OUTPUT_CLEAN_CSV, index=False)
    print(f"\n    ✓ Dataset bersih → {OUTPUT_CLEAN_CSV} ({len(df_clean):,} baris)")
    
    # ─── TASK 3: Z-SCORE & LABELLING ───
    df_labeled = task3_zscore_labelling(df_clean)
    
    # ─── TASK 4: FUZZY SYSTEM LOGIC ───
    df_fuzzy = task4_fuzzy_logic(df_labeled)
    
    # ─── TASK 5 & 6: EVALUASI + TRAIN/TEST ───
    df_final, trained_model = task5_6_evaluation(df_fuzzy)
    
    # ─── SIMPAN SEMUA HASIL ───
    print_header("MENYIMPAN HASIL AKHIR")
    
    OUTPUT_FINAL_CSV = os.path.join(TRAINFILE_DIR, 'dataset_final.csv')
    df_final.to_csv(OUTPUT_FINAL_CSV, index=False)
    print(f"    ✓ Dataset final → {OUTPUT_FINAL_CSV}")
    print(f"      Shape: {df_final.shape[0]:,} baris × {df_final.shape[1]} kolom")
    
    # Update analisa per baris CSV (dengan risk score & priority flag optimal)
    df_output = pd.DataFrame({
        'No': range(1, len(df_final) + 1),
        'Sex': df_final['jenis_kelamin'].map({1: 'Laki-laki', 2: 'Perempuan'}).values,
        'Umur (bln)': df_final['usia_bulan'].astype(int).values,
        'PB anak (cm)': df_final['tinggi_badan_cm'].values,
        'L': df_final['L'].values, 'M': df_final['M'].values, 'S': df_final['S'].values,
        'HAZ (WHO-LMS)': df_final['z_score'].values,
        'Label WHO': df_final['label_who'].values,
        'BBL (g)': df_final['bbl_gram'].values,
        'PBL (cm)': df_final['pbl_cm'].values,
        'MPASI (bln)': df_final['mpasi_bulan'].values,
        'Risk score (Sugeno)': df_final['risk_score'].values,
        'Priority flag': df_final['priority_flag'].values,
    })
    out_perbaris = os.path.join(TRAINFILE_DIR, 'task3_analisa_perbaris.csv')
    df_output.to_csv(out_perbaris, index=False)
    print(f"    ✓ Analisa per baris → {out_perbaris}")
    
    with open(OUTPUT_REPORT, 'w') as f:
        label_dist = df_final['label_who'].value_counts().to_dict()
        flag_dist = df_final['priority_flag'].value_counts().to_dict()
        report['task3'] = {
            'label_distribution': {k: int(v) for k, v in label_dist.items()},
            'stunting_rate': round(label_dist.get('Stunting', 0) / len(df_final) * 100, 2),
        }
        report['task4'] = {
            'priority_flag_distribution': {k: int(v) for k, v in flag_dist.items()},
        }
        report['task5_6'] = trained_model
        json.dump(report, f, indent=2, default=str)
    print(f"    ✓ Laporan → {OUTPUT_REPORT}")
    
    # ─── RINGKASAN ───
    print_header("RINGKASAN PIPELINE TASK 1 → 2 → 3 → 4 → 5 → 6")
    
    tm = trained_model['metrics']
    print(f"""
    ┌───────────────────────────────────────────────────────────────┐
    │  TASK 1: Input Data    → {raw_shape[0]:,} × {raw_shape[1]} → 6 variabel            │
    │  TASK 2: Preprocessing → {len(df_clean):,} balita bersih                  │
    │  TASK 3: Z-Score WHO   → Normal={label_dist.get('Normal',0)} | Stunting={label_dist.get('Stunting',0)}     │
    │  TASK 4: Fuzzy Sugeno  → {len(FUZZY_RULES)} rules, Risk 0-100, Priority Flag     │
    │  TASK 5: Evaluasi      → Confusion Matrix + P/R/F1/Acc        │
    │  TASK 6: Train/Test    → 80/20 Stratified + MF Optimization   │
    ├───────────────────────────────────────────────────────────────┤
    │  HASIL EVALUASI (Test Set - 20% independen):                  │
    │  ┌──────────────────────────────────────────────┐             │
    │  │  Precision : {tm['test']['precision']:.4f}                            │             │
    │  │  Recall    : {tm['test']['recall']:.4f}                            │             │
    │  │  F1-Score  : {tm['test']['f1']:.4f}                            │             │
    │  │  Accuracy  : {tm['test']['accuracy']:.4f}                            │             │
    │  └──────────────────────────────────────────────┘             │
    ├───────────────────────────────────────────────────────────────┤
    │  OUTPUT FILES:                                                │
    │  • trainfile/dataset_final.csv                                │
    │  • trainfile/task3_analisa_perbaris.csv                       │
    │  • trainfile/trained_model.json  ← MODEL TERLATIH             │
    │  • trainfile/dataset_train.csv / dataset_test.csv             │
    │  • trainfile/preprocessing_report.json                        │
    └───────────────────────────────────────────────────────────────┘
    """)
    
    print("    ✅ TASK 1-6 SELESAI - Siap untuk TASK 7 (Pembuatan Sistem Website)")

PROGRAM 2 - 1700 lines

In [ ]:
import pandas as pd
import numpy as np
import json
import os
from datetime import datetime

# ============================================================================
# PATH CONFIGURATION (Relative to this script's location)
# ============================================================================
BASE_DIR = os.path.dirname(os.path.abspath(__file__))

DATASET_PATH   = os.path.join(BASE_DIR, 'dataset', 'Individu_balita_2023_merged.csv')
WHO_ZS_DIR     = os.path.join(BASE_DIR, 'zs')
TRAINFILE_DIR  = os.path.join(BASE_DIR, 'trainfile')

OUTPUT_CLEAN_CSV   = os.path.join(TRAINFILE_DIR, 'dataset_clean.csv')
OUTPUT_LABELED_CSV = os.path.join(TRAINFILE_DIR, 'dataset_labeled.csv')
OUTPUT_REPORT      = os.path.join(TRAINFILE_DIR, 'preprocessing_report.json')

os.makedirs(TRAINFILE_DIR, exist_ok=True)

# WHO Z-Score table file paths
WHO_TABLES = {
    'boys_0_24':  os.path.join(WHO_ZS_DIR, 'boys02.xlsx'),
    'boys_24_60': os.path.join(WHO_ZS_DIR, 'boys25.xlsx'),
    'girls_0_24':  os.path.join(WHO_ZS_DIR, 'girls02.xlsx'),
    'girls_24_60': os.path.join(WHO_ZS_DIR, 'girls25.xlsx'),
}

# ============================================================================
# KONFIGURASI BATAS NORMAL MEDIS
# ============================================================================
VALID_RANGES = {
    'usia_bulan':    (0, 59),
    'jenis_kelamin': [1, 2],
    'tinggi_badan':  (35.0, 130.0),
    'bbl_gram':      (500, 6000),
    'pbl_cm':        (25.0, 60.0),
    'mpasi_bulan':   (0, 12),
}

NOISE_CODES = {
    'bbl': ['8888', '9999', '88.8'],
    'pbl': ['88', '88.8', '99', '99.9'],
}

MPASI_CODE_MAP = {
    'a': 0.25, 'b': 0.5,  'c': 1.0,  'd': 2.0,  'e': 3.0,
    'f': 4.0,  'g': 5.0,  'h': 6.0,  'i': 7.0,  'j': 8.0,
    'k': 9.0,  'l': 10.0, 'm': 11.0, 'n': 12.0,
}

MPASI_NOISE = {'o', 'p', ' ', ''}
# Default priority flag bounds (model-driven recommended). These are fallbacks
# used when no trained model bounds are supplied.
DEFAULT_FLAG_BOUNDS = {'low': 25, 'medium': 50, 'high': 75}

# ============================================================================
# FUNGSI UTILITAS
# ============================================================================
def print_header(title, char='=', width=75):
    print(f"\n{char * width}")
    print(f"  {title}")
    print(f"{char * width}")

def print_step(step_num, description):
    print(f"\n  [{step_num}] {description}")
    print(f"  {'-' * 65}")

def safe_numeric(series, errors='coerce'):
    cleaned = series.astype(str).str.strip().replace({'': np.nan, ' ': np.nan})
    return pd.to_numeric(cleaned, errors=errors)


# ============================================================================
# TASK 1: INPUT DATA
# ============================================================================
def task1_input_data():
    print_header("TASK 1: INPUT DATA - Loading Dataset & Seleksi Fitur")
    
    print_step("1.1", "Memuat dataset SKI 2023 Kalimantan Tengah")
    print(f"    Path: {DATASET_PATH}")
    
    if not os.path.exists(DATASET_PATH):
        raise FileNotFoundError(
            f"\n    ✗ DATASET TIDAK DITEMUKAN!\n"
            f"    Pastikan file berada di: {DATASET_PATH}\n"
        )
    
    df_raw = pd.read_csv(DATASET_PATH, low_memory=False)
    print(f"    ✓ Dataset dimuat: {df_raw.shape[0]:,} baris × {df_raw.shape[1]} kolom")
    
    print_step("1.2", "Mapping kolom dataset ke variabel penelitian")
    
    COLUMN_MAP = {
        'Umur_BLN':             'usia_bulan',
        'Jenis Kelamin':        'jenis_kelamin',
        'Panjang Badan Anak':   'tinggi_badan_cm',
        'Berat Badan Lahir':    'bbl_raw',
        'Panjang Badan Lahir':  'pbl_raw',
        'I43_MP ASI':           'mpasi_raw',
    }
    
    df = df_raw[['ID_balita'] + list(COLUMN_MAP.keys())].copy()
    df.rename(columns=COLUMN_MAP, inplace=True)
    
    for var_lama, var_baru in COLUMN_MAP.items():
        print(f"    • {var_lama:30s} → {var_baru}")
    
    print(f"\n    ✓ Seleksi fitur selesai: {df.shape[1]} kolom dipilih dari {df_raw.shape[1]} kolom")
    
    print_step("1.3", "Statistik awal dataset (sebelum preprocessing)")
    print(f"    Total balita   : {len(df):,}")
    for col in df.columns:
        if col == 'ID_balita':
            continue
        non_null = df[col].notna().sum()
        blank = (df[col].astype(str).str.strip() == '').sum()
        print(f"    • {col:20s}: {non_null:,} terisi, {blank} kosong/blank")
    
    return df, df_raw.shape


# ============================================================================
# TASK 2: PREPROCESSING
# ============================================================================
def task2_preprocessing(df, raw_shape):
    print_header("TASK 2: PREPROCESSING")
    
    report = {
        'raw_total': len(df),
        'raw_shape': list(raw_shape),
        'steps': {},
        'timestamp': datetime.now().isoformat()
    }
    
    # ==== STEP 2d: TRANSFORMASI & REKODING ====
    print_header("STEP 2d: Transformasi & Rekoding", char='-')
    
    # 2d.1: Usia
    print_step("2d.1", "Transformasi Usia Balita → bulan (integer)")
    df['usia_bulan'] = pd.to_numeric(df['usia_bulan'], errors='coerce')
    df['usia_label'] = df['usia_bulan'].apply(
        lambda x: '<1 bulan' if pd.notna(x) and x < 1 else
                  (f'{int(x)} bulan' if pd.notna(x) else None))
    usia_bawah_1 = (df['usia_bulan'] < 1).sum()
    print(f"    ✓ Balita usia <1 bulan: {usia_bawah_1} → dilabeli '<1 bulan'")
    print(f"    Distribusi: min={df['usia_bulan'].min()}, max={df['usia_bulan'].max()}, mean={df['usia_bulan'].mean():.1f}")
    
    # 2d.2: Jenis Kelamin
    print_step("2d.2", "Validasi Jenis Kelamin → 1=Laki-laki, 2=Perempuan")
    df['jenis_kelamin'] = pd.to_numeric(df['jenis_kelamin'], errors='coerce')
    jk_invalid = ~df['jenis_kelamin'].isin([1, 2]) & df['jenis_kelamin'].notna()
    print(f"    ✓ Valid (1/2): {df['jenis_kelamin'].isin([1, 2]).sum():,} | Invalid: {jk_invalid.sum()}")
    
    # 2d.3: Tinggi Badan
    print_step("2d.3", "Transformasi Panjang/Tinggi Badan → cm (float)")
    df['tinggi_badan_cm'] = safe_numeric(df['tinggi_badan_cm'])
    print(f"    ✓ Valid: {df['tinggi_badan_cm'].notna().sum():,} | NaN: {df['tinggi_badan_cm'].isna().sum()}")
    
    # 2d.4: BBL
    print_step("2d.4", "Transformasi BBL → gram (float)")
    bbl_str = df['bbl_raw'].astype(str).str.strip()
    bbl_is_noise = bbl_str.isin(NOISE_CODES['bbl'])
    df['bbl_gram'] = safe_numeric(df['bbl_raw'])
    df.loc[bbl_is_noise, 'bbl_gram'] = np.nan
    print(f"    ✓ Noise (8888): {bbl_is_noise.sum()} | Valid: {df['bbl_gram'].notna().sum():,}")
    
    # 2d.5: PBL
    print_step("2d.5", "Transformasi PBL → cm (float)")
    pbl_str = df['pbl_raw'].astype(str).str.strip()
    pbl_is_noise = pbl_str.isin(NOISE_CODES['pbl'])
    df['pbl_cm'] = safe_numeric(df['pbl_raw'])
    df.loc[pbl_is_noise, 'pbl_cm'] = np.nan
    print(f"    ✓ Noise (88/88.8): {pbl_is_noise.sum()} | Valid: {df['pbl_cm'].notna().sum():,}")
    
    # 2d.6: MP-ASI
    print_step("2d.6", "Transformasi MP-ASI (kode huruf) → bulan (float)")
    mpasi_str = df['mpasi_raw'].astype(str).str.strip().str.lower()
    df['mpasi_bulan'] = mpasi_str.map(MPASI_CODE_MAP)
    mapped_ok = df['mpasi_bulan'].notna().sum()
    
    print(f"    Mapping kode huruf → bulan:")
    for code, bulan in sorted(MPASI_CODE_MAP.items()):
        count = (mpasi_str == code).sum()
        if count > 0:
            lbl = f"<1 bln" if bulan < 1 else f"{bulan:.0f} bln"
            print(f"      '{code}' → {lbl:>8s}: {count:>4d} balita")
    print(f"    ✓ Mapped: {mapped_ok:,} | Missing: {df['mpasi_bulan'].isna().sum():,}")
    
    df['mpasi_label'] = df['mpasi_bulan'].apply(
        lambda x: '<1 bulan' if pd.notna(x) and x < 1 else
                  (f'{int(x)} bulan' if pd.notna(x) and x == int(x) else
                   (f'{x} bulan' if pd.notna(x) else None)))
    
    report['steps']['2d_transformasi'] = {
        'usia_bawah_1_bulan': int(usia_bawah_1),
        'jk_invalid': int(jk_invalid.sum()),
        'bbl_noise_8888': int(bbl_is_noise.sum()),
        'pbl_noise_88': int(pbl_is_noise.sum()),
        'mpasi_mapped': int(mapped_ok),
        'mpasi_missing': int(df['mpasi_bulan'].isna().sum()),
    }
    
    # ==== STEP 2b: FILTER USIA ====
    print_header("STEP 2b: Penyaringan Wilayah & Usia", char='-')
    print(f"    ✓ Dataset khusus Kalimantan Tengah ({len(df):,} balita)")
    n_before_age = len(df)
    usia_valid = df['usia_bulan'].between(0, 59, inclusive='both')
    usia_diluar = ~usia_valid & df['usia_bulan'].notna()
    if usia_diluar.any():
        df = df[usia_valid | df['usia_bulan'].isna()].copy()
        print(f"    ✓ Dihapus: {n_before_age - len(df)} balita di luar 0-59 bulan")
    else:
        print(f"    ✓ Semua dalam rentang 0-59 bulan")
    
    report['steps']['2b_filter_usia'] = {
        'before': n_before_age, 'after': len(df),
    }
    
    # ==== STEP 2c: DATA REDUCTION ====
    print_header("STEP 2c: Data Reduction (Seleksi Fitur)", char='-')
    FINAL_COLUMNS = ['ID_balita', 'usia_bulan', 'usia_label', 'jenis_kelamin',
                     'tinggi_badan_cm', 'bbl_gram', 'pbl_cm', 'mpasi_bulan', 'mpasi_label']
    df = df[FINAL_COLUMNS].copy()
    print(f"    ✓ {len(FINAL_COLUMNS)} kolom dipertahankan dari {raw_shape[1]} kolom asli")
    
    # ==== STEP 2a: LISTWISE DELETION ====
    print_header("STEP 2a: Pemeriksaan Kelengkapan & Pembuangan Noise", char='-')
    print(f"    Metode: LISTWISE DELETION (Tabachnick & Fidell, 2014)")
    
    CORE_VARS = ['usia_bulan', 'jenis_kelamin', 'tinggi_badan_cm',
                 'bbl_gram', 'pbl_cm', 'mpasi_bulan']
    
    n_before = len(df)
    
    print_step("2a.1", "Missing values per variabel inti")
    missing_detail = {}
    for var in CORE_VARS:
        n_miss = df[var].isna().sum()
        pct = n_miss / len(df) * 100
        missing_detail[var] = n_miss
        s = "✓" if n_miss == 0 else "⚠"
        print(f"    {s} {var:20s}: {n_miss:>5d} missing ({pct:>5.1f}%)")
    
    print_step("2a.2", "Deteksi outlier medis")
    outlier_flags = pd.DataFrame(index=df.index)
    
    for col, key in [('usia_bulan','usia_bulan'), ('tinggi_badan_cm','tinggi_badan'),
                     ('bbl_gram','bbl_gram'), ('pbl_cm','pbl_cm'), ('mpasi_bulan','mpasi_bulan')]:
        lo, hi = VALID_RANGES[key]
        mask = ~df[col].between(lo, hi) & df[col].notna()
        outlier_flags[f'{col}_out'] = mask
        if mask.any():
            print(f"    ⚠ {col}: {mask.sum()} di luar [{lo}-{hi}]")
            print(f"      Nilai: {sorted(df.loc[mask, col].unique()[:5])}")
        else:
            print(f"    ✓ {col}: OK [{lo}-{hi}]")
    
    mask_jk = ~df['jenis_kelamin'].isin([1, 2]) & df['jenis_kelamin'].notna()
    outlier_flags['jk_out'] = mask_jk
    print(f"    {'⚠' if mask_jk.any() else '✓'} jenis_kelamin: {mask_jk.sum()} invalid")
    
    print_step("2a.3", "Eksekusi Listwise Deletion")
    any_missing = df[CORE_VARS].isna().any(axis=1)
    any_outlier = outlier_flags.any(axis=1)
    rows_to_drop = any_missing | any_outlier
    
    reasons = {}
    for var in CORE_VARS:
        vm = df[var].isna()
        oc = f'{var}_out' if f'{var}_out' in outlier_flags else 'jk_out'
        vo = outlier_flags.get(oc, pd.Series(False, index=df.index))
        vp = vm | vo
        reasons[var] = int(vp.sum())
        print(f"    • {var:20s}: {vp.sum():>5d} bermasalah (miss={vm.sum()}, outlier={vo.sum()})")
    
    df_clean = df[~rows_to_drop].copy().reset_index(drop=True)
    n_dropped = n_before - len(df_clean)
    
    print(f"\n    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    print(f"    LISTWISE DELETION RESULT:")
    print(f"    • Sebelum : {n_before:,} balita")
    print(f"    • Dihapus : {n_dropped:,} ({n_dropped/n_before*100:.1f}%)")
    print(f"    • Sesudah : {len(df_clean):,} balita")
    print(f"    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    
    report['steps']['2a_listwise'] = {
        'before': n_before, 'total_dropped': n_dropped, 'after': len(df_clean),
        'missing_per_var': {k: int(v) for k, v in missing_detail.items()},
        'reasons': reasons,
    }
    
    # Validasi
    print(f"\n    Validasi: zero missing di semua variabel inti:")
    for var in CORE_VARS:
        n_miss = df_clean[var].isna().sum()
        print(f"    {'✓' if n_miss==0 else '✗'} {var}: {n_miss} missing")
    
    report['final_clean'] = {'rows': len(df_clean), 'cols': df_clean.shape[1]}
    
    return df_clean, report


# ============================================================================
# TASK 3: STUNTING LABELLING & Z-SCORE
# ============================================================================
def task3_zscore_labelling(df):
    """
    Task 3: Menghitung Z-Score WHO (HAZ/LAZ) dan membuat label stunting.
    
    ═══════════════════════════════════════════════════════════════════
    PENTING: Dataset dianalisa PER BARIS (row-by-row).
    Setiap baris = 1 balita → dihitung Z-Score-nya secara INDIVIDUAL.
    ═══════════════════════════════════════════════════════════════════
    
    Alur ANALISA PER BARIS sesuai Proposal Bab 3.3.3:
    ┌─────────────────────────────────────────────────────────────┐
    │  Untuk SETIAP BARIS data balita (iterasi baris ke-1..N):   │
    │                                                             │
    │  1. Baca: jenis_kelamin, usia_bulan, tinggi_badan_cm       │
    │  2. Pilih tabel WHO sesuai jenis kelamin:                  │
    │     • JK=1 (Laki-laki) → boys02/boys25.xlsx                │
    │     • JK=2 (Perempuan) → girls02/girls25.xlsx              │
    │  3. Cari baris WHO berdasarkan usia (bulan) → ambil L,M,S  │
    │  4. Hitung Z-score: Z = [(x/M)^L - 1] / (L × S)          │
    │  5. Labelling:                                              │
    │     • HAZ < -2.00 SD → Stunting                            │
    │     • HAZ ≥ -2.00 SD → Normal                              │
    └─────────────────────────────────────────────────────────────┘
    
    Rumus Z-Score WHO (Persamaan II.2 Proposal):
    Z = [(x/M)^L - 1] / (L × S)
 - x = nilai pengukuran (tinggi/panjang badan cm)
 - M (Miu) = median populasi referensi
 - L (Lambda) = parameter power (transformasi normalitas)
 - S (Sigma) = koefisien variasi
    """
    
    print_header("TASK 3: STUNTING LABELLING & Z-SCORE (Ground Truth WHO)")
    
    print(f"""
    ╔═══════════════════════════════════════════════════════════════╗
    ║  METODE ANALISA: PER BARIS (ROW-BY-ROW ANALYSIS)            ║
    ║                                                              ║
    ║  Setiap baris dataset = 1 balita individual.                 ║
    ║  Analisa dilakukan SATU PER SATU untuk setiap balita:        ║
    ║                                                              ║
    ║  Baris ke-i → Baca (JK, Usia, TB) → Cari WHO(L,M,S)        ║
    ║            → Hitung Z = [(x/M)^L-1]/(L×S) → Label           ║
    ║                                                              ║
    ║  Total baris yang akan dianalisa: {len(df):>4d} balita              ║
    ╚═══════════════════════════════════════════════════════════════╝
    """)
    
    # ══════════════════════════════════════════════════════════════
    # 3.1: Load Tabel WHO
    # ══════════════════════════════════════════════════════════════
    print_step("3.1", "Memuat tabel referensi WHO (LMS parameters)")
    
    who_lookup = {}  # {(jenis_kelamin, usia_bulan): {'L': ..., 'M': ..., 'S': ...}}
    
    table_config = [
        ('boys_0_24',  1, 'Laki-laki 0-24 bln (LAZ)'),
        ('boys_24_60', 1, 'Laki-laki 24-60 bln (HAZ)'),
        ('girls_0_24',  2, 'Perempuan 0-24 bln (LAZ)'),
        ('girls_24_60', 2, 'Perempuan 24-60 bln (HAZ)'),
    ]
    
    for table_key, jk, label in table_config:
        fpath = WHO_TABLES[table_key]
        if not os.path.exists(fpath):
            raise FileNotFoundError(f"    ✗ WHO table tidak ditemukan: {fpath}")
        
        df_who = pd.read_excel(fpath)
        df_who.columns = df_who.columns.str.strip()
        
        n_loaded = 0
        for _, row in df_who.iterrows():
            month = int(row['Month'])
            L_val = float(row['L'])
            M_val = float(row['M'])
            S_val = float(row['S'])
            
            key = (jk, month)
            who_lookup[key] = {'L': L_val, 'M': M_val, 'S': S_val}
            n_loaded += 1
        
        print(f"    ✓ {label:35s} : {n_loaded:>3d} baris dimuat")
    
    total_entries = len(who_lookup)
    print(f"\n    ✓ Total lookup entries: {total_entries}")
    print(f"    ✓ Coverage: Laki-laki 0-60 bln + Perempuan 0-60 bln")
    
    # ══════════════════════════════════════════════════════════════
    # 3.2: Verifikasi Cakupan
    # ══════════════════════════════════════════════════════════════
    print_step("3.2", "Verifikasi cakupan usia dataset vs tabel WHO")
    
    usia_range = df['usia_bulan'].unique()
    jk_range = df['jenis_kelamin'].unique()
    
    missing_keys = []
    for jk in sorted(jk_range):
        for usia in sorted(usia_range):
            key = (int(jk), int(usia))
            if key not in who_lookup:
                missing_keys.append(key)
    
    if missing_keys:
        print(f"    ⚠ {len(missing_keys)} kombinasi TIDAK ditemukan di tabel WHO:")
        for k in missing_keys[:10]:
            print(f"      JK={k[0]}, Usia={k[1]} bulan")
    else:
        print(f"    ✓ Semua kombinasi (jenis_kelamin, usia) tercakup dalam tabel WHO")
    
    print(f"    Usia dataset: {int(min(usia_range))}-{int(max(usia_range))} bulan")
    print(f"    Jenis kelamin: {sorted([int(x) for x in jk_range])}")
    
    # ══════════════════════════════════════════════════════════════
    # 3.3: ANALISA PER BARIS - Hitung Z-Score WHO (Rumus LMS)
    # ══════════════════════════════════════════════════════════════
    print_step("3.3", "ANALISA PER BARIS - Menghitung Z-Score WHO untuk setiap balita")
    print(f"    Rumus WHO (Persamaan II.2): Z = [(x/M)^L - 1] / (L × S)")
    print(f"    Dimana:")
    print(f"      x = panjang/tinggi badan balita (cm) - data pengukuran")
    print(f"      L (Lambda) = parameter power transformasi normalitas")
    print(f"      M (Miu)    = median populasi referensi WHO")
    print(f"      S (Sigma)  = koefisien variasi populasi referensi")
    print(f"    Threshold: HAZ < -2.00 SD → Stunting | HAZ ≥ -2.00 SD → Normal")
    print(f"\n    Memulai analisa per baris untuk {len(df):,} balita...")
    print(f"    {'─' * 65}")
    
    # Array untuk hasil
    z_scores = np.full(len(df), np.nan)
    L_values = np.full(len(df), np.nan)
    M_values = np.full(len(df), np.nan)
    S_values = np.full(len(df), np.nan)
    labels = [None] * len(df)
    
    matched = 0
    not_matched = 0
    
    # === ITERASI PER BARIS ===
    for idx in range(len(df)):
        jk = int(df.iloc[idx]['jenis_kelamin'])
        usia = int(df.iloc[idx]['usia_bulan'])
        x = float(df.iloc[idx]['tinggi_badan_cm'])
        
        # Step A: Pilih tabel WHO berdasarkan jenis kelamin
        # Step B: Cari baris berdasarkan usia (bulan)
        key = (jk, usia)
        
        if key in who_lookup:
            # Step C: Ambil nilai L, M, S
            params = who_lookup[key]
            L = params['L']
            M = params['M']
            S = params['S']
            
            L_values[idx] = L
            M_values[idx] = M
            S_values[idx] = S
            
            # Step D: Hitung Z-Score WHO (Persamaan II.2)
            # Z = [(x/M)^L - 1] / (L × S)
            if L != 0:
                z = ((x / M) ** L - 1) / (L * S)
            else:
                z = np.log(x / M) / S
            
            z_scores[idx] = round(z, 3)
            
            # Step E: Labelling (Tabel II.2 Proposal)
            if z < -2.0:
                labels[idx] = 'Stunting'
            else:
                labels[idx] = 'Normal'
            
            matched += 1
        else:
            not_matched += 1
    
    # Masukkan hasil ke DataFrame
    df['L'] = L_values
    df['M'] = M_values
    df['S'] = S_values
    df['z_score'] = z_scores
    df['label_who'] = labels
    
    print(f"    ✓ Analisa per baris SELESAI: {matched:,}/{len(df):,} balita berhasil dihitung")
    if not_matched > 0:
        print(f"    ⚠ Tidak cocok dengan tabel WHO: {not_matched} balita")
    
    # ══════════════════════════════════════════════════════════════
    # 3.4: OUTPUT HASIL ANALISA PER BARIS (TABEL LENGKAP)
    # ══════════════════════════════════════════════════════════════
    print_step("3.4", "OUTPUT HASIL ANALISA DATA PER BARIS (SELURUH DATASET)")
    print(f"    Format: No | Sex | Umur (bln) | PB anak (cm) | L (Lambda) | M (Miu) | S (Sigma) | HAZ (WHO-LMS) | Label WHO")
    print(f"    Total: {len(df):,} baris data balita")
    
    # Header tabel
    hdr = (f"    {'No':>4s} │ {'Sex':^10s} │ {'Umur':>5s} │ {'PB anak':>9s} │ "
           f"{'L':>6s} │ {'M':>9s} │ {'S':>8s} │ {'HAZ':>10s} │ {'Label WHO':^10s}")
    sep = f"    {'─'*4}─┼─{'─'*10}─┼─{'─'*5}─┼─{'─'*9}─┼─{'─'*6}─┼─{'─'*9}─┼─{'─'*8}─┼─{'─'*10}─┼─{'─'*10}"
    
    print(f"\n{sep}")
    print(hdr)
    print(sep)
    
    for i in range(len(df)):
        row = df.iloc[i]
        no = i + 1
        jk = int(row['jenis_kelamin'])
        jk_str = 'Laki-laki' if jk == 1 else 'Perempuan'
        usia = int(row['usia_bulan'])
        tb = row['tinggi_badan_cm']
        L = row['L']
        M = row['M']
        S = row['S']
        z = row['z_score']
        lbl = row['label_who']
        
        print(f"    {no:>4d} │ {jk_str:^10s} │ {usia:>5d} │ {tb:>9.2f} │ "
              f"{L:>6.2f} │ {M:>9.3f} │ {S:>8.5f} │ {z:>10.3f} │ {lbl:^10s}")
    
    print(sep)
    print(f"    TOTAL: {len(df):,} baris dianalisa secara individual (per baris)")
    
    # ══════════════════════════════════════════════════════════════
    # 3.5: Simpan Output Analisa Per Baris ke CSV
    # ══════════════════════════════════════════════════════════════
    print_step("3.5", "Menyimpan output analisa per baris ke CSV")
    
    # Buat DataFrame output sesuai format gambar referensi
    df_output = pd.DataFrame({
        'No': range(1, len(df) + 1),
        'Sex': df['jenis_kelamin'].map({1: 'Laki-laki', 2: 'Perempuan'}).values,
        'Umur (bln)': df['usia_bulan'].astype(int).values,
        'PB anak (cm)': df['tinggi_badan_cm'].values,
        'L': df['L'].values,
        'M': df['M'].values,
        'S': df['S'].values,
        'HAZ (WHO-LMS)': df['z_score'].values,
        'Label WHO': df['label_who'].values,
    })
    
    output_analysis_path = os.path.join(TRAINFILE_DIR, 'task3_analisa_perbaris.csv')
    df_output.to_csv(output_analysis_path, index=False)
    print(f"    ✓ Output analisa per baris → {output_analysis_path}")
    print(f"      Kolom: {list(df_output.columns)}")
    print(f"      Baris: {len(df_output):,}")
    
    # ══════════════════════════════════════════════════════════════
    # 3.6: Statistik Hasil Z-Score & Labelling
    # ══════════════════════════════════════════════════════════════
    print_step("3.6", "Statistik Hasil Z-Score & Labelling")
    
    z_valid = df['z_score'].dropna()
    
    print(f"    Z-Score Distribution:")
    print(f"    ├── Count   : {len(z_valid):,}")
    print(f"    ├── Mean    : {z_valid.mean():.3f}")
    print(f"    ├── Std     : {z_valid.std():.3f}")
    print(f"    ├── Min     : {z_valid.min():.3f}")
    print(f"    ├── Q1 (25%): {z_valid.quantile(0.25):.3f}")
    print(f"    ├── Median  : {z_valid.median():.3f}")
    print(f"    ├── Q3 (75%): {z_valid.quantile(0.75):.3f}")
    print(f"    └── Max     : {z_valid.max():.3f}")
    
    # Distribusi Label
    print(f"\n    Distribusi Label WHO (Ground Truth):")
    print(f"    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    label_counts = df['label_who'].value_counts()
    total = len(df)
    for label_name in ['Normal', 'Stunting']:
        if label_name in label_counts.index:
            cnt = label_counts[label_name]
            pct = cnt / total * 100
            bar = '█' * int(pct / 2)
            print(f"    │ {label_name:10s}: {cnt:>4d} ({pct:>5.1f}%) {bar}")
    print(f"    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    print(f"    │ TOTAL     : {total:>4d} (100.0%)")
    
    # Distribusi per jenis kelamin
    print(f"\n    Distribusi Label per Jenis Kelamin:")
    for jk_val, jk_name in [(1, 'Laki-laki'), (2, 'Perempuan')]:
        subset = df[df['jenis_kelamin'] == jk_val]
        stunting_n = (subset['label_who'] == 'Stunting').sum()
        normal_n = (subset['label_who'] == 'Normal').sum()
        total_jk = len(subset)
        print(f"    {jk_name:10s}: Normal={normal_n} ({normal_n/total_jk*100:.1f}%), "
              f"Stunting={stunting_n} ({stunting_n/total_jk*100:.1f}%)")
    
    # Distribusi per kelompok usia
    print(f"\n    Distribusi Label per Kelompok Usia:")
    age_groups = [(0, 5, '0-5 bln'), (6, 11, '6-11 bln'), (12, 17, '12-17 bln'), (18, 23, '18-23 bln')]
    for lo, hi, grp_label in age_groups:
        subset = df[(df['usia_bulan'] >= lo) & (df['usia_bulan'] <= hi)]
        if len(subset) == 0:
            continue
        stunting_n = (subset['label_who'] == 'Stunting').sum()
        normal_n = (subset['label_who'] == 'Normal').sum()
        pct_s = stunting_n / len(subset) * 100
        print(f"    {grp_label:10s} (n={len(subset):>3d}): Normal={normal_n:>3d}, "
              f"Stunting={stunting_n:>3d} ({pct_s:.1f}%)")
    
    # ══════════════════════════════════════════════════════════════
    # 3.7: Validasi Manual (Cross-check rumus)
    # ══════════════════════════════════════════════════════════════
    print_step("3.7", "Validasi Formula (Cross-check rumus manual per baris)")
    print(f"    Mengambil 5 sampel acak dan menghitung ulang secara manual...")
    
    np.random.seed(42)
    check_indices = np.random.choice(len(df), size=5, replace=False)
    
    all_valid = True
    for idx in check_indices:
        row = df.iloc[idx]
        jk = int(row['jenis_kelamin'])
        usia = int(row['usia_bulan'])
        x = float(row['tinggi_badan_cm'])
        L = float(row['L'])
        M = float(row['M'])
        S = float(row['S'])
        z_stored = float(row['z_score'])
        
        if L != 0:
            z_manual = ((x / M) ** L - 1) / (L * S)
        else:
            z_manual = np.log(x / M) / S
        z_manual = round(z_manual, 3)
        
        match = "✓" if z_stored == z_manual else "✗"
        if z_stored != z_manual:
            all_valid = False
        
        jk_str = 'Laki-laki' if jk == 1 else 'Perempuan'
        tbl = 'boys' if jk == 1 else 'girls'
        tbl_range = '02' if usia <= 24 else '25'
        
        print(f"    {match} Baris ke-{idx+1}: {jk_str}, {usia} bln, TB={x:.1f} cm")
        print(f"      Tabel WHO  : {tbl}{tbl_range}.xlsx → Bulan {usia}")
        print(f"      L(Lambda)  : {L}")
        print(f"      M(Miu)     : {M:.4f}")
        print(f"      S(Sigma)   : {S:.5f}")
        print(f"      Perhitungan: Z = [({x:.1f}/{M:.4f})^{L} - 1] / ({L} × {S:.5f})")
        print(f"      Z_program  : {z_stored:.3f}")
        print(f"      Z_manual   : {z_manual:.3f}")
        print(f"      Label      : {row['label_who']} (Z {'<' if z_stored < -2 else '≥'} -2.00)")
        print()
    
    if all_valid:
        print(f"    ✅ Semua {len(check_indices)} validasi manual PASS")
        print(f"    ✅ Konfirmasi: Dataset dianalisa PER BARIS - formula Z-Score benar")
    else:
        print(f"    ⚠ Ada perbedaan - perlu review")
    
    return df


# ============================================================================
# TASK 4: FUZZY SYSTEM LOGIC (Sugeno)
# ============================================================================

# ── 4.a: Fungsi Keanggotaan (Membership Functions) ──
# Berdasarkan grafik proposal: Gambar 3.6 (BBL), 3.7 (PBL), 3.8 (MPASI)

def trapezoid(x, a, b, c, d):
    """Fungsi keanggotaan trapezoidal. μ(x) ∈ [0, 1]."""
    if x <= a or x >= d:
        return 0.0
    elif a < x < b:
        return (x - a) / (b - a)
    elif b <= x <= c:
        return 1.0
    elif c < x < d:
        return (d - x) / (d - c)
    return 0.0

def triangle(x, a, b, c):
    """Fungsi keanggotaan segitiga (kasus khusus trapesium b==c)."""
    return trapezoid(x, a, b, b, c)

# ── FUZZYFIKASI BBL (Gambar 3.6) ──
# Breakpoints dari grafik: 1250, 1500, 2450, 2500, 3650, 3999
def mu_bbl_vlbw(x):       return trapezoid(x, 0, 0, 1250, 1500)
def mu_bbl_lbw(x):        return trapezoid(x, 1250, 1500, 2450, 2500)
def mu_bbl_normal(x):     return trapezoid(x, 2450, 2500, 3650, 3999)
def mu_bbl_macrosomia(x): return trapezoid(x, 3650, 3999, 6000, 6000)

# ── FUZZYFIKASI PBL (Gambar 3.7) ──
# Breakpoints dari grafik: 44, 46, 48, 53, 54
def mu_pbl_veryshort(x):  return trapezoid(x, 0, 0, 44, 46)
def mu_pbl_short(x):      return triangle(x, 44, 46, 48)
def mu_pbl_normal(x):     return trapezoid(x, 46, 48, 53, 54)
def mu_pbl_long(x):       return trapezoid(x, 53, 54, 100, 100)

# ── FUZZYFIKASI MPASI (Gambar 3.8) ──
# Breakpoints dari grafik: 4, 5, 6, 7
def mu_mpasi_early(x):    return trapezoid(x, 0, 0, 4, 5)
def mu_mpasi_timely(x):   return trapezoid(x, 4, 5, 6, 7)
def mu_mpasi_late(x):     return trapezoid(x, 6, 7, 12, 12)


def fuzzify(bbl, pbl, mpasi):
    """Fuzzyfikasi: mengubah nilai tegas ke derajat keanggotaan μ ∈ [0,1]."""
    return {
        'bbl': {
            'VLBW':       mu_bbl_vlbw(bbl),
            'LBW':        mu_bbl_lbw(bbl),
            'Normal':     mu_bbl_normal(bbl),
            'Macrosomia': mu_bbl_macrosomia(bbl),
        },
        'pbl': {
            'VeryShort':  mu_pbl_veryshort(pbl),
            'Short':      mu_pbl_short(pbl),
            'Normal':     mu_pbl_normal(pbl),
            'LongLength': mu_pbl_long(pbl),
        },
        'mpasi': {
            'TooEarly':   mu_mpasi_early(mpasi),
            'Timely':     mu_mpasi_timely(mpasi),
            'Late':       mu_mpasi_late(mpasi),
        }
    }


# ── 4.b: Aturan Dasar (Tabel III.2 Proposal) ──
# Setiap rule: (BBL_set, PBL_set, MPASI_set, consequent)
# None = wildcard (variabel tidak ikut dalam rule)
# Tuple = OR condition, misalnya ('Late', 'TooEarly')

FUZZY_RULES = [
    # ═══════ STUNTING HIGH (c=90) ═══════
    ('VLBW',  'VeryShort', None,                   90),
    ('VLBW',  None,        'Late',                 90),
    ('VLBW',  None,        'TooEarly',             90),
    (None,    'VeryShort', 'Late',                 90),
    (None,    'VeryShort', 'TooEarly',             90),
    ('VLBW',  'Short',     ('Late', 'TooEarly'),   90),
    ('VLBW',  'VeryShort', ('Late', 'TooEarly'),   90),

    # ═══════ STUNTING MEDIUM (c=65) ═══════
    ('LBW',   'Short',     None,                   65),
    ('LBW',   None,        ('Late', 'TooEarly'),   65),
    (None,    'Short',     ('Late', 'TooEarly'),   65),
    ('VLBW',  'Normal',    None,                   65),
    ('VLBW',  None,        'Timely',               65),
    ('Normal','VeryShort', None,                   65),
    (None,    'VeryShort', 'Timely',               65),

    # ═══════ STUNTING LOW (c=35) ═══════
    ('Normal','Normal',    'Timely',               35),
    ('Normal','LongLength','Timely',               35),
    ('Macrosomia','Normal','Timely',               35),
    ('Normal','Normal',    ('TooEarly', 'Late'),   35),
    ('Normal',None,        'Timely',               35),  # BBL=Normal, MPASI=Timely (any PBL Short)
    ('LBW',  'Normal',     'Timely',               35),

    # ═══════ NORMAL / NO RISK (c=10) ═══════
    ('Normal',    'LongLength', 'Timely',          10),
    ('Macrosomia','Normal',     'Timely',          10),
    ('Macrosomia','LongLength', 'Timely',          10),
]


def evaluate_rules(mu):
    """
    Evaluasi semua aturan fuzzy (Tabel III.2).
    Operator AND = MIN, Operator OR = MAX.
    Returns: list of (weight, consequent) tuples.
    """
    fired = []

    for rule in FUZZY_RULES:
        bbl_cond, pbl_cond, mpasi_cond, consequent = rule
        antecedents = []

        # BBL condition
        if bbl_cond is not None:
            if isinstance(bbl_cond, tuple):
                antecedents.append(max(mu['bbl'].get(c, 0) for c in bbl_cond))
            else:
                antecedents.append(mu['bbl'].get(bbl_cond, 0))

        # PBL condition
        if pbl_cond is not None:
            if isinstance(pbl_cond, tuple):
                antecedents.append(max(mu['pbl'].get(c, 0) for c in pbl_cond))
            else:
                antecedents.append(mu['pbl'].get(pbl_cond, 0))

        # MPASI condition
        if mpasi_cond is not None:
            if isinstance(mpasi_cond, tuple):
                antecedents.append(max(mu['mpasi'].get(c, 0) for c in mpasi_cond))
            else:
                antecedents.append(mu['mpasi'].get(mpasi_cond, 0))

        # AND = MIN of all antecedents
        if antecedents:
            w = min(antecedents)
            if w > 0:
                fired.append((w, consequent))

    return fired


def defuzzify_sugeno(fired_rules):
    """
    Defuzzyfikasi Sugeno: Weighted Average.
    Risk = Σ(wᵢ × cᵢ) / Σ(wᵢ)
    """
    if not fired_rules:
        return 35.0  # Default: Stunting Low jika tidak ada rule yang aktif

    numerator = sum(w * c for w, c in fired_rules)
    denominator = sum(w for w, c in fired_rules)

    if denominator == 0:
        return 35.0

    return round(numerator / denominator, 2)


    # Backwards-compatible signature: allow passing `bounds` dict produced by
    # the training routine (trained_model['flag_bounds']). If not provided,
    # fall back to DEFAULT_FLAG_BOUNDS.
def get_priority_flag(risk_score, bounds=None):
    """
    Mapping Risk Score → Priority Flag using boundaries supplied by `bounds`.

    Args:
        risk_score (float): numeric risk score (0-100)
        bounds (dict|None): {'low': int, 'medium': int, 'high': int}
                            if None, uses DEFAULT_FLAG_BOUNDS

    Returns:
        str: one of 'Normal', 'STUNTING LOW', 'STUNTING MEDIUM', 'STUNTING HIGH'
    """
    if bounds is None:
        bounds = DEFAULT_FLAG_BOUNDS

    low = bounds.get('low', 25)
    medium = bounds.get('medium', 50)
    high = bounds.get('high', 75)

    # Use strict-inequality boundaries to match the rest of the pipeline
    # (fuzzy_infer_single uses `< fb['low']` to classify Normal).
    if risk_score < low:
        return 'Normal'
    elif risk_score < medium:
        return 'STUNTING LOW'
    elif risk_score < high:
        return 'STUNTING MEDIUM'
    else:
        return 'STUNTING HIGH'


def task4_fuzzy_logic(df):
    """
    Task 4: Fuzzy System Logic (Takagi-Sugeno).
    Input: BBL (gram), PBL (cm), MPASI (bulan)
    Output: Risk Score (0-100), Priority Flag
    """

    print_header("TASK 4: FUZZY SYSTEM LOGIC - Takagi-Sugeno")

    # ── 4.1: Penjelasan Fungsi Keanggotaan ──
    print_step("4.1", "Fungsi Keanggotaan (Fuzzyfikasi)")
    print(f"""
    ┌────────────────────────────────────────────────────────────────┐
    │  BBL (Gambar 3.6) - Trapezoidal                                │
    │  • VLBW         : trap(0, 0, 1250, 1500)    [<1500g]          │
    │  • LBW          : trap(1250, 1500, 2450, 2500) [1500-2500g]   │
    │  • Normal       : trap(2450, 2500, 3650, 3999) [2500-4000g]   │
    │  • Macrosomia   : trap(3650, 3999, 6000, 6000) [>4000g]       │
    ├────────────────────────────────────────────────────────────────┤
    │  PBL (Gambar 3.7) - Trapezoidal + Triangle                    │
    │  • Very Short   : trap(0, 0, 44, 46)         [<46cm]          │
    │  • Short        : triangle(44, 46, 48)        [44-48cm]       │
    │  • Normal       : trap(46, 48, 53, 54)        [48-53cm]       │
    │  • Long Length   : trap(53, 54, 100, 100)     [>54cm]         │
    ├────────────────────────────────────────────────────────────────┤
    │  MP-ASI (Gambar 3.8) - Trapezoidal                            │
    │  • Too Early    : trap(0, 0, 4, 5)            [<5 bln]        │
    │  • Timely       : trap(4, 5, 6, 7)            [5-6 bln]       │
    │  • Late         : trap(6, 7, 12, 12)          [>7 bln]        │
    └────────────────────────────────────────────────────────────────┘
    """)

    # ── 4.2: Aturan Dasar ──
    print_step("4.2", f"Aturan Dasar (Rule Base) - {len(FUZZY_RULES)} aturan")
    risk_labels = {90: 'STUNTING HIGH', 65: 'STUNTING MEDIUM', 35: 'STUNTING LOW', 10: 'Normal'}
    for c_val in [90, 65, 35, 10]:
        rules_c = [(i+1, r) for i, r in enumerate(FUZZY_RULES) if r[3] == c_val]
        print(f"    {risk_labels[c_val]} (c={c_val}): {len(rules_c)} aturan")

    # ── 4.3: Defuzzyfikasi Sugeno ──
    print_step("4.3", "Defuzzyfikasi Sugeno - Weighted Average")
    print(f"    Rumus: Risk = Σ(wᵢ × cᵢ) / Σ(wᵢ)")
    print(f"    wᵢ = bobot aktivasi (operator AND = MIN)")
    print(f"    cᵢ = konstanta konsekuen (10, 35, 65, 90)")

    # ── 4.4: Eksekusi Fuzzy per Baris ──
    print_step("4.4", f"ANALISA FUZZY PER BARIS - {len(df):,} balita")

    risk_scores = []
    priority_flags = []
    rules_fired_counts = []

    for idx in range(len(df)):
        bbl  = float(df.iloc[idx]['bbl_gram'])
        pbl  = float(df.iloc[idx]['pbl_cm'])
        mpasi = float(df.iloc[idx]['mpasi_bulan'])

        # Step A: Fuzzyfikasi
        mu = fuzzify(bbl, pbl, mpasi)

        # Step B: Evaluasi aturan
        fired = evaluate_rules(mu)

        # Step C: Defuzzyfikasi Sugeno
        risk = defuzzify_sugeno(fired)

        # Step D: Mapping Priority Flag
        flag = get_priority_flag(risk)

        risk_scores.append(risk)
        priority_flags.append(flag)
        rules_fired_counts.append(len(fired))

    df['risk_score'] = risk_scores
    df['priority_flag'] = priority_flags

    print(f"    ✓ Fuzzy inference selesai: {len(df):,} balita")
    print(f"    ✓ Rata-rata rules fired per balita: {np.mean(rules_fired_counts):.1f}")

    # ── 4.5: Statistik Risk Score & Priority Flag ──
    print_step("4.5", "Statistik Risk Score & Priority Flag")

    rs = df['risk_score']
    print(f"    Risk Score Distribution:")
    print(f"    ├── Mean    : {rs.mean():.2f}")
    print(f"    ├── Std     : {rs.std():.2f}")
    print(f"    ├── Min     : {rs.min():.2f}")
    print(f"    ├── Median  : {rs.median():.2f}")
    print(f"    └── Max     : {rs.max():.2f}")

    print(f"\n    Distribusi Priority Flag:")
    print(f"    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    total = len(df)
    for flag_name in ['Normal', 'STUNTING LOW', 'STUNTING MEDIUM', 'STUNTING HIGH']:
        cnt = (df['priority_flag'] == flag_name).sum()
        pct = cnt / total * 100
        bar = '█' * int(pct / 2)
        print(f"    │ {flag_name:18s}: {cnt:>4d} ({pct:>5.1f}%) {bar}")
    print(f"    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

    # Cross-tab: Label WHO vs Priority Flag
    print(f"\n    Cross-tabulation: Label WHO × Priority Flag:")
    for who_label in ['Normal', 'Stunting']:
        subset = df[df['label_who'] == who_label]
        flags = subset['priority_flag'].value_counts()
        parts = [f"{f}={flags.get(f, 0)}" for f in ['Normal', 'STUNTING LOW', 'STUNTING MEDIUM', 'STUNTING HIGH']]
        print(f"    WHO {who_label:8s} (n={len(subset):>3d}): {', '.join(parts)}")

    # ── 4.6: OUTPUT TABEL ANALISA LENGKAP PER BARIS ──
    print_step("4.6", "OUTPUT TABEL ANALISA LENGKAP PER BARIS (Z-Score + Fuzzy)")

    hdr = (f"    {'No':>4s} │ {'Sex':^10s} │ {'Umur':>4s} │ {'PB':>7s} │ "
           f"{'L':>5s} │ {'M':>9s} │ {'S':>8s} │ {'HAZ':>8s} │ {'Label':^8s} │ "
           f"{'BBL':>6s} │ {'PBL':>6s} │ {'MPASI':>5s} │ {'Risk':>6s} │ {'Priority Flag':^16s}")
    sep = (f"    {'─'*4}─┼─{'─'*10}─┼─{'─'*4}─┼─{'─'*7}─┼─"
           f"{'─'*5}─┼─{'─'*9}─┼─{'─'*8}─┼─{'─'*8}─┼─{'─'*8}─┼─"
           f"{'─'*6}─┼─{'─'*6}─┼─{'─'*5}─┼─{'─'*6}─┼─{'─'*16}")

    print(f"\n{sep}")
    print(hdr)
    print(sep)

    for i in range(len(df)):
        row = df.iloc[i]
        no = i + 1
        jk_str = 'Laki-laki' if int(row['jenis_kelamin']) == 1 else 'Perempuan'
        usia = int(row['usia_bulan'])
        tb = row['tinggi_badan_cm']
        L = row['L']
        M = row['M']
        S = row['S']
        z = row['z_score']
        lbl = row['label_who']
        bbl = row['bbl_gram']
        pbl = row['pbl_cm']
        mpasi = row['mpasi_bulan']
        risk = row['risk_score']
        flag = row['priority_flag']

        print(f"    {no:>4d} │ {jk_str:^10s} │ {usia:>4d} │ {tb:>7.2f} │ "
              f"{L:>5.2f} │ {M:>9.3f} │ {S:>8.5f} │ {z:>8.3f} │ {lbl:^8s} │ "
              f"{bbl:>6.0f} │ {pbl:>6.2f} │ {mpasi:>5.2f} │ {risk:>6.2f} │ {flag:^16s}")

    print(sep)
    print(f"    TOTAL: {len(df):,} baris dianalisa (Z-Score WHO + Fuzzy Sugeno)")

    # ── 4.7: Simpan Output ──
    print_step("4.7", "Menyimpan output analisa lengkap per baris")

    df_output = pd.DataFrame({
        'No': range(1, len(df) + 1),
        'Sex': df['jenis_kelamin'].map({1: 'Laki-laki', 2: 'Perempuan'}).values,
        'Umur (bln)': df['usia_bulan'].astype(int).values,
        'PB anak (cm)': df['tinggi_badan_cm'].values,
        'L': df['L'].values,
        'M': df['M'].values,
        'S': df['S'].values,
        'HAZ (WHO-LMS)': df['z_score'].values,
        'Label WHO': df['label_who'].values,
        'BBL (g)': df['bbl_gram'].values,
        'PBL (cm)': df['pbl_cm'].values,
        'MPASI (bln)': df['mpasi_bulan'].values,
        'Risk score (Sugeno)': df['risk_score'].values,
        'Priority flag': df['priority_flag'].values,
    })

    output_path = os.path.join(TRAINFILE_DIR, 'task3_analisa_perbaris.csv')
    df_output.to_csv(output_path, index=False)
    print(f"    ✓ Tabel analisa lengkap → {output_path}")
    print(f"      Kolom: {list(df_output.columns)}")

    return df



# ============================================================================
# TASK 5 & 6: EVALUASI + SKEMA PENGUJIAN (TRAIN/TEST)
# ============================================================================
# PERBAIKAN MENYELURUH berdasarkan 10 mistake analysis:
#
# Fix #1: Priority flag mapping → boundaries dioptimasi (bukan hardcode)
# Fix #2: Consequent values → di-tune (bukan terlalu ekstrem)  
# Fix #3: Rule base → sesuai epidemiologi stunting
# Fix #4: MF boundaries → sesuai WHO + overlap realistis
# Fix #5: Ground truth → WHO label vs Priority Flag (binary)
# Fix #6: Confusion matrix → selalu via Priority Flag → binary
# Fix #7: Split → stratified per seed
# Fix #8: Parameter tuning → iterative optimization F1 > target
# Fix #9: Pipeline konsisten: Risk → Flag → Binary → CM
# Fix #10: Forward-backward iterative training
# ============================================================================

def fuzzy_infer_single(bbl, pbl, mpasi, mf_p, cons, flag_bounds):
    """
    Full pipeline untuk 1 balita:
    Input crisp → Fuzzyfikasi → Rule evaluation → Defuzzyfikasi → Priority Flag → Binary
    
    Fix #9: Pipeline SELALU konsisten:
      risk_score → priority_flag → binary_label
    """
    bp, pp, mp = mf_p['bbl'], mf_p['pbl'], mf_p['mpasi']
    
    # --- Fuzzyfikasi (Fix #4: boundaries sesuai WHO + optimasi) ---
    mu_bbl = {
        'VLBW':       trapezoid(bbl, 0, 0, bp[0], bp[1]),
        'LBW':        trapezoid(bbl, bp[0], bp[1], bp[2], bp[3]),
        'Normal':     trapezoid(bbl, bp[2], bp[3], bp[4], bp[5]),
        'Macrosomia': trapezoid(bbl, bp[4], bp[5], 6000, 6000),
    }
    mu_pbl = {
        'VeryShort':  trapezoid(pbl, 0, 0, pp[0], pp[1]),
        'Short':      triangle(pbl, pp[0], pp[1], pp[2]),
        'Normal':     trapezoid(pbl, pp[1], pp[2], pp[3], pp[4]),
        'LongLength': trapezoid(pbl, pp[3], pp[4], 100, 100),
    }
    mu_mpasi = {
        'TooEarly':   trapezoid(mpasi, 0, 0, mp[0], mp[1]),
        'Timely':     trapezoid(mpasi, mp[0], mp[1], mp[2], mp[3]),
        'Late':       trapezoid(mpasi, mp[2], mp[3], 12, 12),
    }
    
    # --- Rule Base (Fix #3: sesuai epidemiologi stunting) ---
    c_h, c_m, c_l, c_n = cons['high'], cons['medium'], cons['low'], cons['normal']
    
    # Rule structure: (bbl_cond, pbl_cond, mpasi_cond, consequent)
    # Fix #3: Rule disesuaikan agar gradasi risiko proporsional
    rules = [
        # === HIGH RISK: kombinasi faktor risiko berat ===
        ('VLBW', 'VeryShort', None,                  c_h),  # 2 faktor berat
        ('VLBW', 'VeryShort', ('Late','TooEarly'),   c_h),  # 3 faktor risiko
        ('VLBW', 'Short',     ('Late','TooEarly'),   c_h),  # VLBW + timing buruk
        ('VLBW', None,        'Late',                c_h),  # VLBW + late MPASI
        ('VLBW', None,        'TooEarly',            c_h),  # VLBW + terlalu dini
        (None,   'VeryShort', 'Late',                c_h),  # sangat pendek + late
        (None,   'VeryShort', 'TooEarly',            c_h),  # sangat pendek + early
        
        # === MEDIUM RISK: 1-2 faktor risiko moderat ===
        ('LBW',  'Short',     None,                  c_m),  # LBW + short
        ('LBW',  None,        ('Late','TooEarly'),   c_m),  # LBW + timing buruk
        (None,   'Short',     ('Late','TooEarly'),   c_m),  # short + timing buruk
        ('VLBW', 'Normal',    None,                  c_m),  # VLBW tapi PBL ok
        ('VLBW', None,        'Timely',              c_m),  # VLBW tapi MPASI ok
        ('Normal','VeryShort', None,                 c_m),  # BBL ok tapi sangat pendek
        (None,   'VeryShort', 'Timely',              c_m),  # sangat pendek tapi MPASI ok
        
        # === LOW RISK: faktor risiko ringan ===
        ('LBW',  'Normal',    'Timely',              c_l),  # hanya BBL rendah
        ('Normal','Short',    'Timely',              c_l),  # hanya PBL pendek
        ('Normal','Normal',   ('TooEarly','Late'),   c_l),  # hanya timing buruk
        ('LBW',  'Normal',    ('TooEarly','Late'),   c_l),  # LBW + timing
        ('Normal','Short',    ('TooEarly','Late'),   c_l),  # short + timing
        
        # === NORMAL: semua faktor baik ===
        ('Normal',    'Normal',     'Timely',        c_n),  # semua baik
        ('Normal',    'LongLength', 'Timely',        c_n),  # BBL+MPASI ok, PBL panjang
        ('Macrosomia','Normal',     'Timely',        c_n),  # BBL besar, sisanya ok
        ('Macrosomia','LongLength', 'Timely',        c_n),  # semua di atas rata-rata
        ('Macrosomia','Normal',     ('TooEarly','Late'), c_n), # BBL besar covers timing
        ('Macrosomia','LongLength', ('TooEarly','Late'), c_n),
        ('Normal',    'LongLength', ('TooEarly','Late'), c_n), # PBL panjang covers timing
    ]
    
    # --- Evaluasi rules: AND=MIN, OR=MAX ---
    fired = []
    for b_c, p_c, m_c, c_val in rules:
        antes = []
        if b_c is not None:
            if isinstance(b_c, tuple):
                antes.append(max(mu_bbl.get(x, 0) for x in b_c))
            else:
                antes.append(mu_bbl.get(b_c, 0))
        if p_c is not None:
            if isinstance(p_c, tuple):
                antes.append(max(mu_pbl.get(x, 0) for x in p_c))
            else:
                antes.append(mu_pbl.get(p_c, 0))
        if m_c is not None:
            if isinstance(m_c, tuple):
                antes.append(max(mu_mpasi.get(x, 0) for x in m_c))
            else:
                antes.append(mu_mpasi.get(m_c, 0))
        if antes:
            w = min(antes)  # AND = MIN
            if w > 0:
                fired.append((w, c_val))
    
    # --- Defuzzyfikasi Sugeno: Weighted Average ---
    if fired:
        num = sum(w * c for w, c in fired)
        den = sum(w for w, c in fired)
        risk = round(num / den, 2) if den > 0 else cons['low']
    else:
        risk = cons['low']
    
    # --- Fix #1: Priority Flag via optimizable boundaries ---
    fb = flag_bounds  # {'low': 30, 'medium': 55, 'high': 75}
    if risk < fb['low']:
        flag = 'Normal'
    elif risk < fb['medium']:
        flag = 'STUNTING LOW'
    elif risk < fb['high']:
        flag = 'STUNTING MEDIUM'
    else:
        flag = 'STUNTING HIGH'
    
    # --- Fix #5 & #9: Binary label dari Priority Flag ---
    binary = 0 if flag == 'Normal' else 1  # 0=Normal, 1=Stunting
    
    return risk, flag, binary


def run_pipeline(dataframe, mf_p, cons, flag_bounds):
    """Run full pipeline pada seluruh dataframe."""
    risks, flags, binaries = [], [], []
    for idx in range(len(dataframe)):
        bbl = float(dataframe.iloc[idx]['bbl_gram'])
        pbl = float(dataframe.iloc[idx]['pbl_cm'])
        mpasi = float(dataframe.iloc[idx]['mpasi_bulan'])
        r, f, b = fuzzy_infer_single(bbl, pbl, mpasi, mf_p, cons, flag_bounds)
        risks.append(r)
        flags.append(f)
        binaries.append(b)
    return np.array(risks), flags, np.array(binaries)


def calc_metrics(y_true, y_pred):
    """Fix #6: Confusion matrix dari binary labels."""
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    prec = tp/(tp+fp) if (tp+fp) > 0 else 0.0
    rec  = tp/(tp+fn) if (tp+fn) > 0 else 0.0
    f1   = 2*prec*rec/(prec+rec) if (prec+rec) > 0 else 0.0
    acc  = (tp+tn)/(tp+tn+fp+fn) if (tp+tn+fp+fn) > 0 else 0.0
    return {'TP':tp, 'TN':tn, 'FP':fp, 'FN':fn,
            'precision':round(prec,4), 'recall':round(rec,4),
            'f1':round(f1,4), 'accuracy':round(acc,4)}


def task5_6_evaluation(df):
    """
    Task 5+6: Iterative Forward-Backward Training (Fix #10)
    
    Pipeline konsisten (Fix #9):
      Crisp Input → Fuzzyfikasi → Rules → Defuzzyfikasi → Risk Score
      → Priority Flag (via optimized boundaries) → Binary Label
      → vs Ground Truth WHO → Confusion Matrix
    
    Training (Fix #8):
      Optimasi MF + Consequent + Flag Boundaries → maximize F1
      Stratified 80/20 (Fix #7), random setiap iterasi
    """
    print_header("TASK 5 & 6: EVALUASI + SKEMA PENGUJIAN SISTEM")
    
    print(f"""
    ╔═══════════════════════════════════════════════════════════════╗
    ║  PIPELINE KONSISTEN (Fix #9):                                ║
    ║  Input → Fuzzyfikasi → Rules → Defuzzyfikasi → Risk Score   ║
    ║  → Priority Flag (boundaries dioptimasi) → Binary Label     ║
    ║  → vs WHO Label → Confusion Matrix                          ║
    ║                                                               ║
    ║  GROUND TRUTH (Fix #5):                                      ║
    ║  WHO Normal  + Flag Normal           = TN (benar)            ║
    ║  WHO Normal  + Flag Stunting(L/M/H)  = FP (false alarm)     ║
    ║  WHO Stunting + Flag Stunting(L/M/H) = TP (terdeteksi)      ║
    ║  WHO Stunting + Flag Normal           = FN (terlewat!)       ║
    ║                                                               ║
    ║  TRAINING (Fix #8 & #10):                                    ║
    ║  Forward : run fuzzy 80% → cek ground truth                  ║
    ║  Backward: perbaiki MF + consequent + flag boundaries        ║
    ║  Iterasi : 5 random seeds × full grid search                 ║
    ╚═══════════════════════════════════════════════════════════════╝
    """)
    
    y_all = (df['label_who'] == 'Stunting').astype(int).values
    
    # ═══════════════════════════════════════════════════════════════
    # Fix #4: MF boundaries berdasarkan WHO + data-driven analysis
    # Data: Stunting mean BBL=2970, Normal mean BBL=3139 (diff=170g)
    #        Stunting mean PBL=48.4, Normal mean PBL=49.4 (diff=1.0cm)
    # ═══════════════════════════════════════════════════════════════
    BBL_GRID = [
        [1200, 1500, 2400, 2500, 3800, 4000],   # WHO standard
        [1200, 1500, 2400, 2600, 3600, 3900],   # Wider overlap
        [1200, 1500, 2500, 2700, 3500, 3800],   # ← best prev run
        [1200, 1500, 2400, 2700, 3500, 3900],   # Wider LBW
        [1200, 1500, 2500, 2800, 3500, 3800],   # Higher normal start
        [1200, 1500, 2300, 2500, 3600, 4000],   # Standard WHO
    ]
    PBL_GRID = [
        [44, 46, 48, 52, 54],   # WHO standard
        [44, 47, 49, 53, 54],   # ← best prev run
        [44, 46, 49, 52, 53],   # Mid range
        [43, 47, 49, 52, 53],   # Lower VeryShort
        [44, 48, 50, 53, 54],   # Higher Short peak
    ]
    MPASI_GRID = [
        [4, 5, 6, 7],    # WHO standard
        [3, 5, 7, 8],    # ← best prev run  
        [4, 5, 7, 8],    # Late shifted
        [3, 4, 6, 7],    # Wide early
    ]
    
    # Fix #2: Consequent values - moderate, proportional spread
    CONS_GRID = [
        {'high': 75, 'medium': 50, 'low': 20, 'normal': 5},  # ← best prev
        {'high': 80, 'medium': 50, 'low': 25, 'normal': 5},
        {'high': 80, 'medium': 55, 'low': 25, 'normal': 8},
        {'high': 85, 'medium': 55, 'low': 30, 'normal': 8},
        {'high': 85, 'medium': 60, 'low': 30, 'normal': 10},
        {'high': 90, 'medium': 60, 'low': 25, 'normal': 5},
        {'high': 70, 'medium': 45, 'low': 18, 'normal': 5},
        {'high': 75, 'medium': 48, 'low': 22, 'normal': 3},
    ]
    
    # Fix #1: Flag boundaries - determines Normal/Stunting cutoff
    FLAG_BOUNDS_GRID = [
        {'low': 30, 'medium': 55, 'high': 75},  # ← best prev
        {'low': 25, 'medium': 50, 'high': 70},
        {'low': 20, 'medium': 45, 'high': 70},
        {'low': 28, 'medium': 52, 'high': 73},
        {'low': 15, 'medium': 40, 'high': 65},
        {'low': 22, 'medium': 48, 'high': 72},
        {'low': 35, 'medium': 58, 'high': 78},
        {'low': 18, 'medium': 42, 'high': 68},
    ]
    
    SEEDS = [42, 123, 256, 789, 1024, 2048, 3333]  # 7 seeds for robustness
    
    n_mf = len(BBL_GRID) * len(PBL_GRID) * len(MPASI_GRID)
    n_post = len(CONS_GRID) * len(FLAG_BOUNDS_GRID)
    
    print_step("6.1", "Search Space")
    print(f"    MF configs: {n_mf} | Post-MF combos: {n_post}")
    print(f"    Seeds: {len(SEEDS)}")
    print(f"    Total: {n_mf * n_post * len(SEEDS):,} evaluations")
    print(f"    (Pre-compute rule weights per MF → fast cons/flag iteration)")
    
    # ═══════════════════════════════════════════════════════════════
    # Fix #10: Iterative Forward-Backward Training
    # ═══════════════════════════════════════════════════════════════
    print_step("6.2", "Iterative Forward-Backward Training")
    
    # Rule template classes (independent of consequent values)
    RULE_TEMPLATES = [
        ('VLBW','VeryShort',None,'H'), ('VLBW','VeryShort',('Late','TooEarly'),'H'),
        ('VLBW','Short',('Late','TooEarly'),'H'), ('VLBW',None,'Late','H'),
        ('VLBW',None,'TooEarly','H'), (None,'VeryShort','Late','H'),
        (None,'VeryShort','TooEarly','H'),
        ('LBW','Short',None,'M'), ('LBW',None,('Late','TooEarly'),'M'),
        (None,'Short',('Late','TooEarly'),'M'), ('VLBW','Normal',None,'M'),
        ('VLBW',None,'Timely','M'), ('Normal','VeryShort',None,'M'),
        (None,'VeryShort','Timely','M'),
        ('LBW','Normal','Timely','L'), ('Normal','Short','Timely','L'),
        ('Normal','Normal',('TooEarly','Late'),'L'),
        ('LBW','Normal',('TooEarly','Late'),'L'),
        ('Normal','Short',('TooEarly','Late'),'L'),
        ('Normal','Normal','Timely','N'), ('Normal','LongLength','Timely','N'),
        ('Macrosomia','Normal','Timely','N'), ('Macrosomia','LongLength','Timely','N'),
        ('Macrosomia','Normal',('TooEarly','Late'),'N'),
        ('Macrosomia','LongLength',('TooEarly','Late'),'N'),
        ('Normal','LongLength',('TooEarly','Late'),'N'),
    ]
    
    global_best = {'combo': -1}
    
    for seed in SEEDS:
        np.random.seed(seed)
        idx_n = np.where(y_all == 0)[0].copy()
        idx_s = np.where(y_all == 1)[0].copy()
        np.random.shuffle(idx_n)
        np.random.shuffle(idx_s)
        
        n_tr_n = int(len(idx_n) * 0.8)
        n_tr_s = int(len(idx_s) * 0.8)
        tr_idx = np.concatenate([idx_n[:n_tr_n], idx_s[:n_tr_s]])
        te_idx = np.concatenate([idx_n[n_tr_n:], idx_s[n_tr_s:]])
        
        df_tr = df.iloc[tr_idx].reset_index(drop=True)
        df_te = df.iloc[te_idx].reset_index(drop=True)
        y_tr = (df_tr['label_who'] == 'Stunting').astype(int).values
        y_te = (df_te['label_who'] == 'Stunting').astype(int).values
        
        seed_best = {'score': -1}
        
        for bp in BBL_GRID:
            for pp in PBL_GRID:
                for mp in MPASI_GRID:
                    mf = {'bbl': bp, 'pbl': pp, 'mpasi': mp}
                    
                    # PRE-COMPUTE: rule weights per row (expensive, once per MF)
                    tr_weights = []
                    for idx in range(len(df_tr)):
                        bbl = float(df_tr.iloc[idx]['bbl_gram'])
                        pbl = float(df_tr.iloc[idx]['pbl_cm'])
                        mpasi_v = float(df_tr.iloc[idx]['mpasi_bulan'])
                        
                        mu_bbl = {
                            'VLBW': trapezoid(bbl,0,0,bp[0],bp[1]),
                            'LBW': trapezoid(bbl,bp[0],bp[1],bp[2],bp[3]),
                            'Normal': trapezoid(bbl,bp[2],bp[3],bp[4],bp[5]),
                            'Macrosomia': trapezoid(bbl,bp[4],bp[5],6000,6000),
                        }
                        mu_pbl = {
                            'VeryShort': trapezoid(pbl,0,0,pp[0],pp[1]),
                            'Short': triangle(pbl,pp[0],pp[1],pp[2]),
                            'Normal': trapezoid(pbl,pp[1],pp[2],pp[3],pp[4]),
                            'LongLength': trapezoid(pbl,pp[3],pp[4],100,100),
                        }
                        mu_mpasi = {
                            'TooEarly': trapezoid(mpasi_v,0,0,mp[0],mp[1]),
                            'Timely': trapezoid(mpasi_v,mp[0],mp[1],mp[2],mp[3]),
                            'Late': trapezoid(mpasi_v,mp[2],mp[3],12,12),
                        }
                        
                        row_w = []
                        for b_c, p_c, m_c, cls in RULE_TEMPLATES:
                            antes = []
                            if b_c: antes.append(max(mu_bbl.get(x,0) for x in b_c) if isinstance(b_c,tuple) else mu_bbl.get(b_c,0))
                            if p_c: antes.append(max(mu_pbl.get(x,0) for x in p_c) if isinstance(p_c,tuple) else mu_pbl.get(p_c,0))
                            if m_c: antes.append(max(mu_mpasi.get(x,0) for x in m_c) if isinstance(m_c,tuple) else mu_mpasi.get(m_c,0))
                            if antes:
                                w = min(antes)
                                if w > 0: row_w.append((w, cls))
                        tr_weights.append(row_w)
                    
                    # FAST ITERATION: cons + flag_bounds (no fuzzy recomputation)
                    for cons in CONS_GRID:
                        c_map = {'H':cons['high'], 'M':cons['medium'],
                                 'L':cons['low'], 'N':cons['normal']}
                        
                        # Compute risk scores from pre-computed weights
                        risks = np.zeros(len(df_tr))
                        for i, rw in enumerate(tr_weights):
                            if rw:
                                num = sum(w * c_map[cls] for w, cls in rw)
                                den = sum(w for w, cls in rw)
                                risks[i] = num / den if den > 0 else cons['low']
                            else:
                                risks[i] = cons['low']
                        
                        for fb in FLAG_BOUNDS_GRID:
                            # Priority flag → binary (Fix #9)
                            y_pred = np.where(risks < fb['low'], 0, 1).astype(int)
                            
                            m = calc_metrics(y_tr, y_pred)
                            
                            if m['accuracy'] >= 0.65:
                                score = m['f1']
                            else:
                                score = m['f1'] * 0.3
                            
                            if score > seed_best['score']:
                                seed_best = {
                                    'score': score, 'mf': mf.copy(),
                                    'cons': cons.copy(), 'fb': fb.copy(),
                                    'train_m': m.copy(), 'seed': seed,
                                    'tr_idx': tr_idx, 'te_idx': te_idx,
                                }
        
        # Validasi pada test set
        if seed_best['score'] > 0:
            _, _, y_pred_te = run_pipeline(df_te, seed_best['mf'], 
                                           seed_best['cons'], seed_best['fb'])
            test_m = calc_metrics(y_te, y_pred_te)
            
            combo = 0.3 * seed_best['score'] + 0.7 * test_m['f1']
            if test_m['accuracy'] < 0.60:
                combo *= 0.3
            
            trn = seed_best['train_m']
            print(f"    Seed {seed:>4d} | Train: Acc={trn['accuracy']:.4f} F1={trn['f1']:.4f} P={trn['precision']:.4f} R={trn['recall']:.4f} | "
                  f"Test: Acc={test_m['accuracy']:.4f} F1={test_m['f1']:.4f}")
            
            if combo > global_best.get('combo', -1):
                global_best = seed_best.copy()
                global_best['combo'] = combo
                global_best['test_m'] = test_m
                global_best['y_tr'] = y_tr
                global_best['y_te'] = y_te
                global_best['df_tr'] = df_tr
                global_best['df_te'] = df_te
    
    best = global_best
    
    print_step("6.3", "BEST MODEL (dari forward-backward)")
    print(f"""
    ┌────────────────────────────────────────────────────────────┐
    │  BBL MF       : {best['mf']['bbl']}  │
    │  PBL MF       : {best['mf']['pbl']}           │
    │  MPASI MF     : {best['mf']['mpasi']}                  │
    │  Consequents  : H={best['cons']['high']} M={best['cons']['medium']} L={best['cons']['low']} N={best['cons']['normal']}        │
    │  Flag Bounds  : N<{best['fb']['low']} | L<{best['fb']['medium']} | M<{best['fb']['high']} | H≥{best['fb']['high']}    │
    │  Seed         : {best['seed']}                                  │
    └────────────────────────────────────────────────────────────┘
    """)
    
    # ═══════════════════════════════════════════════════════════════
    # Re-compute seluruh data dengan model optimal
    # ═══════════════════════════════════════════════════════════════
    print_step("6.4", "Re-compute seluruh data")
    
    all_risks, all_flags, all_binary = run_pipeline(df, best['mf'], best['cons'], best['fb'])
    df['risk_score'] = np.round(all_risks, 2)
    df['priority_flag'] = all_flags
    df['fuzzy_label'] = ['Stunting' if b == 1 else 'Normal' for b in all_binary]
    
    flag_dist = pd.Series(all_flags).value_counts()
    print(f"    Priority Flag distribution:")
    for flag_name in ['Normal', 'STUNTING LOW', 'STUNTING MEDIUM', 'STUNTING HIGH']:
        cnt = flag_dist.get(flag_name, 0)
        print(f"      {flag_name:18s}: {cnt:>3d} ({cnt/len(df)*100:.1f}%)")
    
    # ═══════════════════════════════════════════════════════════════
    # EVALUASI: Confusion Matrix (Fix #5 & #6)
    # ═══════════════════════════════════════════════════════════════
    
    # --- TRAIN SET ---
    print_step("5.1", f"CONFUSION MATRIX - Train ({len(best['y_tr'])})")
    _, _, y_pred_tr = run_pipeline(best['df_tr'], best['mf'], best['cons'], best['fb'])
    trm = calc_metrics(best['y_tr'], y_pred_tr)
    tp,tn,fp,fn = trm['TP'],trm['TN'],trm['FP'],trm['FN']
    print(f"""
    ╔══════════════════════════════════════════════════════════╗
    ║         CONFUSION MATRIX - TRAIN SET (80%)              ║
    ╠══════════════════╦══════════════╦════════════════════════╣
    ║                  ║ Act Stunting ║ Act Normal             ║
    ╠══════════════════╬══════════════╬════════════════════════╣
    ║ Pred Stunting    ║  TP = {tp:>4d}  ║  FP = {fp:>4d}             ║
    ║ Pred Normal      ║  FN = {fn:>4d}  ║  TN = {tn:>4d}             ║
    ╠══════════════════╩══════════════╩════════════════════════╣
    ║ Precision = {trm['precision']:.4f}                                ║
    ║ Recall    = {trm['recall']:.4f}                                ║
    ║ F1-Score  = {trm['f1']:.4f}                                ║
    ║ Accuracy  = {trm['accuracy']:.4f}                                ║
    ╚══════════════════════════════════════════════════════════╝""")
    
    # --- TEST SET ---
    print_step("5.2", f"CONFUSION MATRIX - Test ({len(best['y_te'])})")
    tm = best['test_m']
    tp,tn,fp,fn = tm['TP'],tm['TN'],tm['FP'],tm['FN']
    print(f"""
    ╔══════════════════════════════════════════════════════════╗
    ║         CONFUSION MATRIX - TEST SET (20%)               ║
    ║       (Data independen, tidak terlihat saat training)    ║
    ╠══════════════════╦══════════════╦════════════════════════╣
    ║                  ║ Act Stunting ║ Act Normal             ║
    ╠══════════════════╬══════════════╬════════════════════════╣
    ║ Pred Stunting    ║  TP = {tp:>4d}  ║  FP = {fp:>4d}             ║
    ║ Pred Normal      ║  FN = {fn:>4d}  ║  TN = {tn:>4d}             ║
    ╠══════════════════╩══════════════╩════════════════════════╣
    ║ Precision = {tm['precision']:.4f}                                ║
    ║ Recall    = {tm['recall']:.4f}                                ║
    ║ F1-Score  = {tm['f1']:.4f}                                ║
    ║ Accuracy  = {tm['accuracy']:.4f}                                ║
    ╚══════════════════════════════════════════════════════════╝""")
    
    # --- ALL DATA ---
    print_step("5.3", f"CONFUSION MATRIX - All Data ({len(df)})")
    y_all_t = (df['label_who']=='Stunting').astype(int).values
    y_all_p = (df['fuzzy_label']=='Stunting').astype(int).values
    am = calc_metrics(y_all_t, y_all_p)
    tp,tn,fp,fn = am['TP'],am['TN'],am['FP'],am['FN']
    print(f"""
    ╔══════════════════════════════════════════════════════════╗
    ║         CONFUSION MATRIX - SELURUH DATA ({len(df)})           ║
    ╠══════════════════╦══════════════╦════════════════════════╣
    ║ Pred Stunting    ║  TP = {tp:>4d}  ║  FP = {fp:>4d}             ║
    ║ Pred Normal      ║  FN = {fn:>4d}  ║  TN = {tn:>4d}             ║
    ╠══════════════════╩══════════════╩════════════════════════╣
    ║ Precision = {am['precision']:.4f}                                ║
    ║ Recall    = {am['recall']:.4f}                                ║
    ║ F1-Score  = {am['f1']:.4f}                                ║
    ║ Accuracy  = {am['accuracy']:.4f}                                ║
    ╚══════════════════════════════════════════════════════════╝""")
    
    # --- RINGKASAN ---
    print_step("5.4", "Ringkasan Evaluasi")
    print(f"    ┌──────────────┬───────────┬──────────┬──────────┬──────────┐")
    print(f"    │   Dataset    │ Precision │  Recall  │ F1-Score │ Accuracy │")
    print(f"    ├──────────────┼───────────┼──────────┼──────────┼──────────┤")
    print(f"    │ Train (80%)  │  {trm['precision']:.4f}   │  {trm['recall']:.4f} │  {trm['f1']:.4f} │  {trm['accuracy']:.4f} │")
    print(f"    │ Test  (20%)  │  {tm['precision']:.4f}   │  {tm['recall']:.4f} │  {tm['f1']:.4f} │  {tm['accuracy']:.4f} │")
    print(f"    │ All  (100%)  │  {am['precision']:.4f}   │  {am['recall']:.4f} │  {am['f1']:.4f} │  {am['accuracy']:.4f} │")
    print(f"    └──────────────┴───────────┴──────────┴──────────┴──────────┘")
    
    # --- SIMPAN MODEL ---
    print_step("5.5", "Simpan model terlatih")
    trained_model = {
        'mf_params': best['mf'], 'consequents': best['cons'],
        'flag_bounds': best['fb'], 'seed': int(best['seed']),
        'metrics': {'train': trm, 'test': tm, 'all': am},
        'split': {'train_size': len(best['y_tr']), 'test_size': len(best['y_te'])},
        'timestamp': datetime.now().isoformat(),
    }
    model_path = os.path.join(TRAINFILE_DIR, 'trained_model.json')
    with open(model_path, 'w') as f:
        json.dump(trained_model, f, indent=2, default=str)
    print(f"    ✓ {model_path}")
    
    df.iloc[best['tr_idx']].reset_index(drop=True).to_csv(
        os.path.join(TRAINFILE_DIR, 'dataset_train.csv'), index=False)
    df.iloc[best['te_idx']].reset_index(drop=True).to_csv(
        os.path.join(TRAINFILE_DIR, 'dataset_test.csv'), index=False)
    print(f"    ✓ dataset_train.csv ({len(best['tr_idx'])}), dataset_test.csv ({len(best['te_idx'])})")
    
    return df, trained_model

if __name__ == '__main__':
    
    print("\n" + "█" * 75)
    print("█  STUNTOR - Sistem Klasifikasi Stunting Balita Kalimantan Tengah    █")
    print("█  Dataset: SKI 2023 | Metode: Fuzzy Sugeno                          █")
    print("█" * 75)
    print(f"\n  Working directory : {os.getcwd()}")
    print(f"  Script location   : {os.path.abspath(__file__)}")
    print(f"  Dataset path      : {DATASET_PATH}")
    print(f"  WHO Z-Score dir   : {WHO_ZS_DIR}")
    print(f"  Output dir        : {TRAINFILE_DIR}")
    
    # ─── TASK 1: INPUT DATA ───
    df_input, raw_shape = task1_input_data()
    
    # ─── TASK 2: PREPROCESSING ───
    df_clean, report = task2_preprocessing(df_input, raw_shape)
    
    df_clean.to_csv(OUTPUT_CLEAN_CSV, index=False)
    print(f"\n    ✓ Dataset bersih → {OUTPUT_CLEAN_CSV} ({len(df_clean):,} baris)")
    
    # ─── TASK 3: Z-SCORE & LABELLING ───
    df_labeled = task3_zscore_labelling(df_clean)
    
    # ─── TASK 4: FUZZY SYSTEM LOGIC ───
    df_fuzzy = task4_fuzzy_logic(df_labeled)
    
    # ─── TASK 5 & 6: EVALUASI + TRAIN/TEST ───
    df_final, trained_model = task5_6_evaluation(df_fuzzy)
    
    # ─── SIMPAN SEMUA HASIL ───
    print_header("MENYIMPAN HASIL AKHIR")
    
    OUTPUT_FINAL_CSV = os.path.join(TRAINFILE_DIR, 'dataset_final.csv')
    df_final.to_csv(OUTPUT_FINAL_CSV, index=False)
    print(f"    ✓ Dataset final → {OUTPUT_FINAL_CSV}")
    print(f"      Shape: {df_final.shape[0]:,} baris × {df_final.shape[1]} kolom")
    
    # Update analisa per baris CSV (dengan risk score & priority flag optimal)
    df_output = pd.DataFrame({
        'No': range(1, len(df_final) + 1),
        'Sex': df_final['jenis_kelamin'].map({1: 'Laki-laki', 2: 'Perempuan'}).values,
        'Umur (bln)': df_final['usia_bulan'].astype(int).values,
        'PB anak (cm)': df_final['tinggi_badan_cm'].values,
        'L': df_final['L'].values, 'M': df_final['M'].values, 'S': df_final['S'].values,
        'HAZ (WHO-LMS)': df_final['z_score'].values,
        'Label WHO': df_final['label_who'].values,
        'BBL (g)': df_final['bbl_gram'].values,
        'PBL (cm)': df_final['pbl_cm'].values,
        'MPASI (bln)': df_final['mpasi_bulan'].values,
        'Risk score (Sugeno)': df_final['risk_score'].values,
        'Priority flag': df_final['priority_flag'].values,
    })
    out_perbaris = os.path.join(TRAINFILE_DIR, 'task3_analisa_perbaris.csv')
    df_output.to_csv(out_perbaris, index=False)
    print(f"    ✓ Analisa per baris → {out_perbaris}")
    
    with open(OUTPUT_REPORT, 'w') as f:
        label_dist = df_final['label_who'].value_counts().to_dict()
        flag_dist = df_final['priority_flag'].value_counts().to_dict()
        report['task3'] = {
            'label_distribution': {k: int(v) for k, v in label_dist.items()},
            'stunting_rate': round(label_dist.get('Stunting', 0) / len(df_final) * 100, 2),
        }
        report['task4'] = {
            'priority_flag_distribution': {k: int(v) for k, v in flag_dist.items()},
        }
        report['task5_6'] = trained_model
        json.dump(report, f, indent=2, default=str)
    print(f"    ✓ Laporan → {OUTPUT_REPORT}")
    
    # ─── RINGKASAN ───
    print_header("RINGKASAN PIPELINE TASK 1 → 2 → 3 → 4 → 5 → 6")
    
    tm = trained_model['metrics']
    print(f"""
    ┌───────────────────────────────────────────────────────────────┐
    │  TASK 1: Input Data    → {raw_shape[0]:,} × {raw_shape[1]} → 6 variabel            │
    │  TASK 2: Preprocessing → {len(df_clean):,} balita bersih                  │
    │  TASK 3: Z-Score WHO   → Normal={label_dist.get('Normal',0)} | Stunting={label_dist.get('Stunting',0)}     │
    │  TASK 4: Fuzzy Sugeno  → {len(FUZZY_RULES)} rules, Risk 0-100, Priority Flag     │
    │  TASK 5: Evaluasi      → Confusion Matrix + P/R/F1/Acc        │
    │  TASK 6: Train/Test    → 80/20 Stratified + MF Optimization   │
    ├───────────────────────────────────────────────────────────────┤
    │  HASIL EVALUASI (Test Set - 20% independen):                  │
    │  ┌──────────────────────────────────────────────┐             │
    │  │  Precision : {tm['test']['precision']:.4f}                            │             │
    │  │  Recall    : {tm['test']['recall']:.4f}                            │             │
    │  │  F1-Score  : {tm['test']['f1']:.4f}                            │             │
    │  │  Accuracy  : {tm['test']['accuracy']:.4f}                            │             │
    │  └──────────────────────────────────────────────┘             │
    ├───────────────────────────────────────────────────────────────┤
    │  OUTPUT FILES:                                                │
    │  • trainfile/dataset_final.csv                                │
    │  • trainfile/task3_analisa_perbaris.csv                       │
    │  • trainfile/trained_model.json  ← MODEL TERLATIH             │
    │  • trainfile/dataset_train.csv / dataset_test.csv             │
    │  • trainfile/preprocessing_report.json                        │
    └───────────────────────────────────────────────────────────────┘
    """)
    
    print("    ✅ TASK 1-6 SELESAI - Siap untuk TASK 7 (Pembuatan Sistem Website)")

PROGRAM 3 BUKAN GA

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
=============================================================================
PENELITIAN: Klasifikasi Stunting pada Balita di Kalimantan Tengah
            Menggunakan Logika Fuzzy (Studi Kasus Data SKI 2023)
=============================================================================
TASK 1: INPUT DATA - Loading Dataset & Seleksi Fitur
TASK 2: PREPROCESSING - Pembersihan, Penyaringan, Transformasi
TASK 3: STUNTING LABELLING & Z-SCORE - Ground Truth WHO
TASK 4: FUZZY SYSTEM LOGIC - Takagi-Sugeno (Fuzzyfikasi, Rules, Defuzzyfikasi)
TASK 5: EVALUASI - Confusion Matrix (Precision, Recall, F1-Score, Accuracy)
TASK 6: SKEMA PENGUJIAN - Train/Test 80/20 Stratified + Optimasi Parameter
=============================================================================
Lokasi: train/fuzzytrain.py
Struktur:
  PROJECTS2/
    train/
      fuzzytrain.py              ← FILE INI
      dataset/
        Individu_balita_2023_merged.csv
      trainfile/                 ← output hasil train disimpan di sini
      zs/
        boys02.xlsx   (boys 0-24 bulan)
        boys25.xlsx   (boys 24-60 bulan)
        girls02.xlsx  (girls 0-24 bulan)
        girls25.xlsx  (girls 24-60 bulan)
    mainprogram/
    test/
=============================================================================
"""

import pandas as pd
import numpy as np
import json
import os
from datetime import datetime

# ============================================================================
# PATH CONFIGURATION (Relative to this script's location)
# ============================================================================
BASE_DIR = os.path.dirname(os.path.abspath(__file__))

DATASET_PATH   = os.path.join(BASE_DIR, 'dataset', 'Individu_balita_2023_merged.csv')
WHO_ZS_DIR     = os.path.join(BASE_DIR, 'zs')
TRAINFILE_DIR  = os.path.join(BASE_DIR, 'trainfile')

OUTPUT_CLEAN_CSV   = os.path.join(TRAINFILE_DIR, 'dataset_clean.csv')
OUTPUT_LABELED_CSV = os.path.join(TRAINFILE_DIR, 'dataset_labeled.csv')
OUTPUT_REPORT      = os.path.join(TRAINFILE_DIR, 'preprocessing_report.json')

os.makedirs(TRAINFILE_DIR, exist_ok=True)

# WHO Z-Score table file paths
WHO_TABLES = {
    'boys_0_24':  os.path.join(WHO_ZS_DIR, 'boys02.xlsx'),
    'boys_24_60': os.path.join(WHO_ZS_DIR, 'boys25.xlsx'),
    'girls_0_24':  os.path.join(WHO_ZS_DIR, 'girls02.xlsx'),
    'girls_24_60': os.path.join(WHO_ZS_DIR, 'girls25.xlsx'),
}

# ============================================================================
# KONFIGURASI BATAS NORMAL MEDIS
# ============================================================================
VALID_RANGES = {
    'usia_bulan':    (0, 59),
    'jenis_kelamin': [1, 2],
    'tinggi_badan':  (35.0, 130.0),
    'bbl_gram':      (500, 6000),
    'pbl_cm':        (25.0, 60.0),
    'mpasi_bulan':   (0, 12),
}

NOISE_CODES = {
    'bbl': ['8888', '9999', '88.8'],
    'pbl': ['88', '88.8', '99', '99.9'],
}

MPASI_CODE_MAP = {
    'a': 0.25, 'b': 0.5,  'c': 1.0,  'd': 2.0,  'e': 3.0,
    'f': 4.0,  'g': 5.0,  'h': 6.0,  'i': 7.0,  'j': 8.0,
    'k': 9.0,  'l': 10.0, 'm': 11.0, 'n': 12.0,
}

MPASI_NOISE = {'o', 'p', ' ', ''}

# Default priority flag bounds (model-driven recommended). These are fallbacks
# used when no trained model bounds are supplied.
DEFAULT_FLAG_BOUNDS = {'low': 25, 'medium': 50, 'high': 75}

# ============================================================================
# FUNGSI UTILITAS
# ============================================================================
def print_header(title, char='=', width=60):
    print(f"\n{char * width}")
    print(f"  {title}")
    print(f"{char * width}")

def print_step(step_num, description):
    print(f"\n  [{step_num}] {description}")

def safe_numeric(series, errors='coerce'):
    cleaned = series.astype(str).str.strip().replace({'': np.nan, ' ': np.nan})
    return pd.to_numeric(cleaned, errors=errors)


# ============================================================================
# TASK 1: INPUT DATA
# ============================================================================
def task1_input_data():
    print_header("TASK 1: INPUT DATA - Loading Dataset & Seleksi Fitur")
    
    print_step("1.1", "Memuat dataset SKI 2023 Kalimantan Tengah")
    print(f"    Path: {DATASET_PATH}")
    
    if not os.path.exists(DATASET_PATH):
        raise FileNotFoundError(
            f"\n    [X] DATASET TIDAK DITEMUKAN!\n"
            f"    Pastikan file berada di: {DATASET_PATH}\n"
        )
    
    df_raw = pd.read_csv(DATASET_PATH, low_memory=False)
    print(f"    [OK] Dataset dimuat: {df_raw.shape[0]:,} baris x {df_raw.shape[1]} kolom")
    
    print_step("1.2", "Mapping kolom dataset ke variabel penelitian")
    
    COLUMN_MAP = {
        'Umur_BLN':             'usia_bulan',
        'Jenis Kelamin':        'jenis_kelamin',
        'Panjang Badan Anak':   'tinggi_badan_cm',
        'Berat Badan Lahir':    'bbl_raw',
        'Panjang Badan Lahir':  'pbl_raw',
        'I43_MP ASI':           'mpasi_raw',
    }
    
    df = df_raw[['ID_balita'] + list(COLUMN_MAP.keys())].copy()
    df.rename(columns=COLUMN_MAP, inplace=True)
    
    for var_lama, var_baru in COLUMN_MAP.items():
        print(f"    {var_lama:30s} -> {var_baru}")
    
    print(f"\n    [OK] Seleksi fitur selesai: {df.shape[1]} kolom dipilih dari {df_raw.shape[1]} kolom")
    
    print_step("1.3", "Statistik awal dataset (sebelum preprocessing)")
    print(f"    Total balita   : {len(df):,}")
    for col in df.columns:
        if col == 'ID_balita':
            continue
        non_null = df[col].notna().sum()
        blank = (df[col].astype(str).str.strip() == '').sum()
        print(f"    {col:20s}: {non_null:,} terisi, {blank} kosong/blank")
    
    return df, df_raw.shape


# ============================================================================
# TASK 2: PREPROCESSING
# ============================================================================
def task2_preprocessing(df, raw_shape):
    print_header("TASK 2: PREPROCESSING")
    
    report = {
        'raw_total': len(df),
        'raw_shape': list(raw_shape),
        'steps': {},
        'timestamp': datetime.now().isoformat()
    }
    
    # ==== STEP 2d: TRANSFORMASI & REKODING ====
    print_header("STEP 2d: Transformasi & Rekoding", char='-')
    
    # 2d.1: Usia
    print_step("2d.1", "Transformasi Usia Balita -> bulan (integer)")
    df['usia_bulan'] = pd.to_numeric(df['usia_bulan'], errors='coerce')
    df['usia_label'] = df['usia_bulan'].apply(
        lambda x: '<1 bulan' if pd.notna(x) and x < 1 else
                  (f'{int(x)} bulan' if pd.notna(x) else None))
    usia_bawah_1 = (df['usia_bulan'] < 1).sum()
    print(f"    [OK] Balita usia <1 bulan: {usia_bawah_1} -> dilabeli '<1 bulan'")
    print(f"    Distribusi: min={df['usia_bulan'].min()}, max={df['usia_bulan'].max()}, mean={df['usia_bulan'].mean():.1f}")
    
    # 2d.2: Jenis Kelamin
    print_step("2d.2", "Validasi Jenis Kelamin -> 1=Laki-laki, 2=Perempuan")
    df['jenis_kelamin'] = pd.to_numeric(df['jenis_kelamin'], errors='coerce')
    jk_invalid = ~df['jenis_kelamin'].isin([1, 2]) & df['jenis_kelamin'].notna()
    print(f"    [OK] Valid (1/2): {df['jenis_kelamin'].isin([1, 2]).sum():,} | Invalid: {jk_invalid.sum()}")
    
    # 2d.3: Tinggi Badan
    print_step("2d.3", "Transformasi Panjang/Tinggi Badan -> cm (float)")
    df['tinggi_badan_cm'] = safe_numeric(df['tinggi_badan_cm'])
    print(f"    [OK] Valid: {df['tinggi_badan_cm'].notna().sum():,} | NaN: {df['tinggi_badan_cm'].isna().sum()}")
    
    # 2d.4: BBL
    print_step("2d.4", "Transformasi BBL -> gram (float)")
    bbl_str = df['bbl_raw'].astype(str).str.strip()
    bbl_is_noise = bbl_str.isin(NOISE_CODES['bbl'])
    df['bbl_gram'] = safe_numeric(df['bbl_raw'])
    df.loc[bbl_is_noise, 'bbl_gram'] = np.nan
    print(f"    [OK] Noise (8888): {bbl_is_noise.sum()} | Valid: {df['bbl_gram'].notna().sum():,}")
    
    # 2d.5: PBL
    print_step("2d.5", "Transformasi PBL -> cm (float)")
    pbl_str = df['pbl_raw'].astype(str).str.strip()
    pbl_is_noise = pbl_str.isin(NOISE_CODES['pbl'])
    df['pbl_cm'] = safe_numeric(df['pbl_raw'])
    df.loc[pbl_is_noise, 'pbl_cm'] = np.nan
    print(f"    [OK] Noise (88/88.8): {pbl_is_noise.sum()} | Valid: {df['pbl_cm'].notna().sum():,}")
    
    # 2d.6: MP-ASI
    print_step("2d.6", "Transformasi MP-ASI (kode huruf) -> bulan (float)")
    mpasi_str = df['mpasi_raw'].astype(str).str.strip().str.lower()
    df['mpasi_bulan'] = mpasi_str.map(MPASI_CODE_MAP)
    mapped_ok = df['mpasi_bulan'].notna().sum()
    
    print(f"    Mapping kode huruf -> bulan:")
    for code, bulan in sorted(MPASI_CODE_MAP.items()):
        count = (mpasi_str == code).sum()
        if count > 0:
            lbl = f"<1 bln" if bulan < 1 else f"{bulan:.0f} bln"
            print(f"      '{code}' -> {lbl:>8s}: {count:>4d} balita")
    print(f"    [OK] Mapped: {mapped_ok:,} | Missing: {df['mpasi_bulan'].isna().sum():,}")
    
    df['mpasi_label'] = df['mpasi_bulan'].apply(
        lambda x: '<1 bulan' if pd.notna(x) and x < 1 else
                  (f'{int(x)} bulan' if pd.notna(x) and x == int(x) else
                   (f'{x} bulan' if pd.notna(x) else None)))
    
    report['steps']['2d_transformasi'] = {
        'usia_bawah_1_bulan': int(usia_bawah_1),
        'jk_invalid': int(jk_invalid.sum()),
        'bbl_noise_8888': int(bbl_is_noise.sum()),
        'pbl_noise_88': int(pbl_is_noise.sum()),
        'mpasi_mapped': int(mapped_ok),
        'mpasi_missing': int(df['mpasi_bulan'].isna().sum()),
    }
    
    # ==== STEP 2b: FILTER USIA ====
    print_header("STEP 2b: Penyaringan Wilayah & Usia", char='-')
    print(f"    [OK] Dataset khusus Kalimantan Tengah ({len(df):,} balita)")
    n_before_age = len(df)
    usia_valid = df['usia_bulan'].between(0, 59, inclusive='both')
    usia_diluar = ~usia_valid & df['usia_bulan'].notna()
    if usia_diluar.any():
        df = df[usia_valid | df['usia_bulan'].isna()].copy()
        print(f"    [OK] Dihapus: {n_before_age - len(df)} balita di luar 0-59 bulan")
    else:
        print(f"    [OK] Semua dalam rentang 0-59 bulan")
    
    report['steps']['2b_filter_usia'] = {
        'before': n_before_age, 'after': len(df),
    }
    
    # ==== STEP 2c: DATA REDUCTION ====
    print_header("STEP 2c: Data Reduction (Seleksi Fitur)", char='-')
    FINAL_COLUMNS = ['ID_balita', 'usia_bulan', 'usia_label', 'jenis_kelamin',
                     'tinggi_badan_cm', 'bbl_gram', 'pbl_cm', 'mpasi_bulan', 'mpasi_label']
    df = df[FINAL_COLUMNS].copy()
    print(f"    [OK] {len(FINAL_COLUMNS)} kolom dipertahankan dari {raw_shape[1]} kolom asli")
    
    # ==== STEP 2a: LISTWISE DELETION ====
    print_header("STEP 2a: Pemeriksaan Kelengkapan & Pembuangan Noise", char='-')
    print(f"    Metode: LISTWISE DELETION (Tabachnick & Fidell, 2014)")
    
    CORE_VARS = ['usia_bulan', 'jenis_kelamin', 'tinggi_badan_cm',
                 'bbl_gram', 'pbl_cm', 'mpasi_bulan']
    
    n_before = len(df)
    
    print_step("2a.1", "Missing values per variabel inti")
    missing_detail = {}
    for var in CORE_VARS:
        n_miss = df[var].isna().sum()
        pct = n_miss / len(df) * 100
        missing_detail[var] = n_miss
        s = "[OK]" if n_miss == 0 else "[!]"
        print(f"    {s} {var:20s}: {n_miss:>5d} missing ({pct:>5.1f}%)")
    
    print_step("2a.2", "Deteksi outlier medis")
    outlier_flags = pd.DataFrame(index=df.index)
    
    for col, key in [('usia_bulan','usia_bulan'), ('tinggi_badan_cm','tinggi_badan'),
                     ('bbl_gram','bbl_gram'), ('pbl_cm','pbl_cm'), ('mpasi_bulan','mpasi_bulan')]:
        lo, hi = VALID_RANGES[key]
        mask = ~df[col].between(lo, hi) & df[col].notna()
        outlier_flags[f'{col}_out'] = mask
        if mask.any():
            print(f"    [!] {col}: {mask.sum()} di luar [{lo}-{hi}]")
            print(f"      Nilai: {sorted(df.loc[mask, col].unique()[:5])}")
        else:
            print(f"    [OK] {col}: OK [{lo}-{hi}]")
    
    mask_jk = ~df['jenis_kelamin'].isin([1, 2]) & df['jenis_kelamin'].notna()
    outlier_flags['jk_out'] = mask_jk
    print(f"    {'[!]' if mask_jk.any() else '[OK]'} jenis_kelamin: {mask_jk.sum()} invalid")
    
    print_step("2a.3", "Eksekusi Listwise Deletion")
    any_missing = df[CORE_VARS].isna().any(axis=1)
    any_outlier = outlier_flags.any(axis=1)
    rows_to_drop = any_missing | any_outlier
    
    reasons = {}
    for var in CORE_VARS:
        vm = df[var].isna()
        oc = f'{var}_out' if f'{var}_out' in outlier_flags else 'jk_out'
        vo = outlier_flags.get(oc, pd.Series(False, index=df.index))
        vp = vm | vo
        reasons[var] = int(vp.sum())
        print(f"    {var:20s}: {vp.sum():>5d} bermasalah (miss={vm.sum()}, outlier={vo.sum()})")
    
    df_clean = df[~rows_to_drop].copy().reset_index(drop=True)
    n_dropped = n_before - len(df_clean)
    
    print(f"\n    LISTWISE DELETION RESULT:")
    print(f"    Sebelum : {n_before:,} balita")
    print(f"    Dihapus : {n_dropped:,} ({n_dropped/n_before*100:.1f}%)")
    print(f"    Sesudah : {len(df_clean):,} balita")
    
    report['steps']['2a_listwise'] = {
        'before': n_before, 'total_dropped': n_dropped, 'after': len(df_clean),
        'missing_per_var': {k: int(v) for k, v in missing_detail.items()},
        'reasons': reasons,
    }
    
    # Validasi
    print(f"\n    Validasi: zero missing di semua variabel inti:")
    for var in CORE_VARS:
        n_miss = df_clean[var].isna().sum()
        print(f"    {'[OK]' if n_miss==0 else '[X]'} {var}: {n_miss} missing")
    
    report['final_clean'] = {'rows': len(df_clean), 'cols': df_clean.shape[1]}
    
    return df_clean, report


# ============================================================================
# TASK 3: STUNTING LABELLING & Z-SCORE
# ============================================================================
def task3_zscore_labelling(df):
    """
    Task 3: Menghitung Z-Score WHO (HAZ/LAZ) dan membuat label stunting.
    
    Alur ANALISA PER BARIS sesuai Proposal Bab 3.3.3:
      1. Baca: jenis_kelamin, usia_bulan, tinggi_badan_cm
      2. Pilih tabel WHO sesuai jenis kelamin
      3. Cari baris WHO berdasarkan usia (bulan) -> ambil L,M,S
      4. Hitung Z-score: Z = [(x/M)^L - 1] / (L x S)
      5. Labelling: HAZ < -2.00 SD -> Stunting | HAZ >= -2.00 SD -> Normal
    """
    
    print_header("TASK 3: STUNTING LABELLING & Z-SCORE (Ground Truth WHO)")
    
    print(f"    Analisa per baris: {len(df)} balita")
    print(f"    Rumus: Z = [(x/M)^L - 1] / (L x S)")
    print(f"    Threshold: HAZ < -2.00 SD = Stunting, HAZ >= -2.00 SD = Normal")
    
    # 3.1: Load Tabel WHO
    print_step("3.1", "Memuat tabel referensi WHO (LMS parameters)")
    
    who_lookup = {}
    
    table_config = [
        ('boys_0_24',  1, 'Laki-laki 0-24 bln (LAZ)'),
        ('boys_24_60', 1, 'Laki-laki 24-60 bln (HAZ)'),
        ('girls_0_24',  2, 'Perempuan 0-24 bln (LAZ)'),
        ('girls_24_60', 2, 'Perempuan 24-60 bln (HAZ)'),
    ]
    
    for table_key, jk, label in table_config:
        fpath = WHO_TABLES[table_key]
        if not os.path.exists(fpath):
            raise FileNotFoundError(f"    [X] WHO table tidak ditemukan: {fpath}")
        
        df_who = pd.read_excel(fpath)
        df_who.columns = df_who.columns.str.strip()
        
        n_loaded = 0
        for _, row in df_who.iterrows():
            month = int(row['Month'])
            key = (jk, month)
            who_lookup[key] = {'L': float(row['L']), 'M': float(row['M']), 'S': float(row['S'])}
            n_loaded += 1
        
        print(f"    [OK] {label:35s} : {n_loaded:>3d} baris dimuat")
    
    print(f"\n    [OK] Total lookup entries: {len(who_lookup)}")
    
    # 3.2: Verifikasi Cakupan
    print_step("3.2", "Verifikasi cakupan usia dataset vs tabel WHO")
    
    usia_range = df['usia_bulan'].unique()
    jk_range = df['jenis_kelamin'].unique()
    
    missing_keys = []
    for jk in sorted(jk_range):
        for usia in sorted(usia_range):
            key = (int(jk), int(usia))
            if key not in who_lookup:
                missing_keys.append(key)
    
    if missing_keys:
        print(f"    [!] {len(missing_keys)} kombinasi TIDAK ditemukan di tabel WHO:")
        for k in missing_keys[:10]:
            print(f"      JK={k[0]}, Usia={k[1]} bulan")
    else:
        print(f"    [OK] Semua kombinasi (jenis_kelamin, usia) tercakup dalam tabel WHO")
    
    print(f"    Usia dataset: {int(min(usia_range))}-{int(max(usia_range))} bulan")
    print(f"    Jenis kelamin: {sorted([int(x) for x in jk_range])}")
    
    # 3.3: ANALISA PER BARIS
    print_step("3.3", "ANALISA PER BARIS - Menghitung Z-Score WHO untuk setiap balita")
    print(f"    Rumus WHO (Persamaan II.2): Z = [(x/M)^L - 1] / (L x S)")
    print(f"    Dimana:")
    print(f"      x = panjang/tinggi badan balita (cm) - data pengukuran")
    print(f"      L (Lambda) = parameter power transformasi normalitas")
    print(f"      M (Miu)    = median populasi referensi WHO")
    print(f"      S (Sigma)  = koefisien variasi populasi referensi")
    print(f"    Threshold: HAZ < -2.00 SD -> Stunting | HAZ >= -2.00 SD -> Normal")
    print(f"\n    Memulai analisa per baris untuk {len(df):,} balita...")
    print(f"    {'-' * 65}")
    
    z_scores = np.full(len(df), np.nan)
    L_values = np.full(len(df), np.nan)
    M_values = np.full(len(df), np.nan)
    S_values = np.full(len(df), np.nan)
    labels = [None] * len(df)
    
    matched = 0
    
    for idx in range(len(df)):
        jk = int(df.iloc[idx]['jenis_kelamin'])
        usia = int(df.iloc[idx]['usia_bulan'])
        x = float(df.iloc[idx]['tinggi_badan_cm'])
        
        key = (jk, usia)
        
        if key in who_lookup:
            params = who_lookup[key]
            L, M, S = params['L'], params['M'], params['S']
            
            L_values[idx] = L
            M_values[idx] = M
            S_values[idx] = S
            
            if L != 0:
                z = ((x / M) ** L - 1) / (L * S)
            else:
                z = np.log(x / M) / S
            
            z_scores[idx] = round(z, 3)
            labels[idx] = 'Stunting' if z < -2.0 else 'Normal'
            matched += 1
    
    df['L'] = L_values
    df['M'] = M_values
    df['S'] = S_values
    df['z_score'] = z_scores
    df['label_who'] = labels
    
    print(f"    [OK] Analisa per baris SELESAI: {matched:,}/{len(df):,} balita berhasil dihitung")
    
    # --------------------------------------------------------------
    # 3.4: OUTPUT HASIL ANALISA PER BARIS (TABEL LENGKAP)
    # --------------------------------------------------------------
    print_step("3.4", "OUTPUT HASIL ANALISA DATA PER BARIS (SELURUH DATASET)")
    print(f"    Format: No | Sex | Umur (bln) | PB anak (cm) | L (Lambda) | M (Miu) | S (Sigma) | HAZ (WHO-LMS) | Label WHO")
    print(f"    Total: {len(df):,} baris data balita")
    
    hdr = (f"    {'No':>4s} | {'Sex':^10s} | {'Umur':>5s} | {'PB anak':>9s} | "
           f"{'L':>6s} | {'M':>9s} | {'S':>8s} | {'HAZ':>10s} | {'Label WHO':^10s}")
    sep = f"    {'-'*4}-+-{'-'*10}-+-{'-'*5}-+-{'-'*9}-+-{'-'*6}-+-{'-'*9}-+-{'-'*8}-+-{'-'*10}-+-{'-'*10}"
    
    print(f"\n{sep}")
    print(hdr)
    print(sep)
    
    for i in range(len(df)):
        row = df.iloc[i]
        no = i + 1
        jk = int(row['jenis_kelamin'])
        jk_str = 'Laki-laki' if jk == 1 else 'Perempuan'
        usia = int(row['usia_bulan'])
        tb = row['tinggi_badan_cm']
        L = row['L']
        M = row['M']
        S = row['S']
        z = row['z_score']
        lbl = row['label_who']
        
        print(f"    {no:>4d} | {jk_str:^10s} | {usia:>5d} | {tb:>9.2f} | "
              f"{L:>6.2f} | {M:>9.3f} | {S:>8.5f} | {z:>10.3f} | {lbl:^10s}")
    
    print(sep)
    print(f"    TOTAL: {len(df):,} baris dianalisa secara individual (per baris)")
    
    # --------------------------------------------------------------
    # 3.5: Statistik Hasil Z-Score & Labelling
    # --------------------------------------------------------------
    print_step("3.5", "Statistik Hasil Z-Score & Labelling")
    
    z_valid = df['z_score'].dropna()
    
    print(f"    Z-Score Distribution:")
    print(f"    Count={len(z_valid):,}, Mean={z_valid.mean():.3f}, Std={z_valid.std():.3f}")
    print(f"    Min={z_valid.min():.3f}, Q1={z_valid.quantile(0.25):.3f}, Median={z_valid.median():.3f}, Q3={z_valid.quantile(0.75):.3f}, Max={z_valid.max():.3f}")
    
    label_counts = df['label_who'].value_counts()
    total = len(df)
    print(f"\n    Distribusi Label WHO (Ground Truth):")
    for label_name in ['Normal', 'Stunting']:
        if label_name in label_counts.index:
            cnt = label_counts[label_name]
            pct = cnt / total * 100
            print(f"    {label_name:10s}: {cnt:>4d} ({pct:>5.1f}%)")
    print(f"    TOTAL     : {total:>4d} (100.0%)")
    
    # Distribusi per jenis kelamin
    print(f"\n    Distribusi Label per Jenis Kelamin:")
    for jk_val, jk_name in [(1, 'Laki-laki'), (2, 'Perempuan')]:
        subset = df[df['jenis_kelamin'] == jk_val]
        stunting_n = (subset['label_who'] == 'Stunting').sum()
        normal_n = (subset['label_who'] == 'Normal').sum()
        total_jk = len(subset)
        print(f"    {jk_name:10s}: Normal={normal_n} ({normal_n/total_jk*100:.1f}%), "
              f"Stunting={stunting_n} ({stunting_n/total_jk*100:.1f}%)")
    
    # Distribusi per kelompok usia
    print(f"\n    Distribusi Label per Kelompok Usia:")
    age_groups = [(0, 5, '0-5 bln'), (6, 11, '6-11 bln'), (12, 17, '12-17 bln'), (18, 23, '18-23 bln')]
    for lo, hi, grp_label in age_groups:
        subset = df[(df['usia_bulan'] >= lo) & (df['usia_bulan'] <= hi)]
        if len(subset) == 0:
            continue
        stunting_n = (subset['label_who'] == 'Stunting').sum()
        normal_n = (subset['label_who'] == 'Normal').sum()
        pct_s = stunting_n / len(subset) * 100
        print(f"    {grp_label:10s} (n={len(subset):>3d}): Normal={normal_n:>3d}, "
              f"Stunting={stunting_n:>3d} ({pct_s:.1f}%)")
    
    # --------------------------------------------------------------
    # 3.6: Validasi Manual (Cross-check rumus)
    # --------------------------------------------------------------
    print_step("3.6", "Validasi Formula (Cross-check rumus manual per baris)")
    print(f"    Mengambil 5 sampel acak dan menghitung ulang secara manual...")
    
    np.random.seed(42)
    check_indices = np.random.choice(len(df), size=5, replace=False)
    
    all_valid = True
    for idx in check_indices:
        row = df.iloc[idx]
        jk = int(row['jenis_kelamin'])
        usia = int(row['usia_bulan'])
        x = float(row['tinggi_badan_cm'])
        L = float(row['L'])
        M = float(row['M'])
        S = float(row['S'])
        z_stored = float(row['z_score'])
        
        if L != 0:
            z_manual = ((x / M) ** L - 1) / (L * S)
        else:
            z_manual = np.log(x / M) / S
        z_manual = round(z_manual, 3)
        
        match = "[OK]" if z_stored == z_manual else "[X]"
        if z_stored != z_manual:
            all_valid = False
        
        jk_str = 'Laki-laki' if jk == 1 else 'Perempuan'
        tbl = 'boys' if jk == 1 else 'girls'
        tbl_range = '02' if usia <= 24 else '25'
        
        print(f"    {match} Baris ke-{idx+1}: {jk_str}, {usia} bln, TB={x:.1f} cm")
        print(f"      Tabel WHO  : {tbl}{tbl_range}.xlsx -> Bulan {usia}")
        print(f"      L(Lambda)  : {L}")
        print(f"      M(Miu)     : {M:.4f}")
        print(f"      S(Sigma)   : {S:.5f}")
        print(f"      Perhitungan: Z = [({x:.1f}/{M:.4f})^{L} - 1] / ({L} x {S:.5f})")
        print(f"      Z_program  : {z_stored:.3f}")
        print(f"      Z_manual   : {z_manual:.3f}")
        print(f"      Label      : {row['label_who']} (Z {'<' if z_stored < -2 else '>='} -2.00)")
        print()
    
    if all_valid:
        print(f"    [OK] Semua {len(check_indices)} validasi manual PASS")
        print(f"    [OK] Konfirmasi: Dataset dianalisa PER BARIS - formula Z-Score benar")
    else:
        print(f"    [!] Ada perbedaan - perlu review")
    
    # --------------------------------------------------------------
    # 3.7: Simpan Output Analisa Per Baris ke CSV
    # --------------------------------------------------------------
    print_step("3.7", "Menyimpan output analisa per baris ke CSV")
    df_output = pd.DataFrame({
        'No': range(1, len(df) + 1),
        'Sex': df['jenis_kelamin'].map({1: 'Laki-laki', 2: 'Perempuan'}).values,
        'Umur (bln)': df['usia_bulan'].astype(int).values,
        'PB anak (cm)': df['tinggi_badan_cm'].values,
        'L': df['L'].values, 'M': df['M'].values, 'S': df['S'].values,
        'HAZ (WHO-LMS)': df['z_score'].values,
        'Label WHO': df['label_who'].values,
    })
    output_path = os.path.join(TRAINFILE_DIR, 'task3_analisa_perbaris.csv')
    df_output.to_csv(output_path, index=False)
    print(f"    [OK] Output -> {output_path} ({len(df_output):,} baris)")
    
    return df


# ============================================================================
# TASK 4: FUZZY SYSTEM LOGIC (Takagi-Sugeno)
# ============================================================================


# -- Fungsi Keanggotaan Dasar --
def trapezoid(x, a, b, c, d):
    """Fungsi keanggotaan trapezoidal. mu(x) in [0, 1]."""
    if x <= a or x >= d:
        return 0.0
    elif a < x < b:
        return (x - a) / (b - a)
    elif b <= x <= c:
        return 1.0
    elif c < x < d:
        return (d - x) / (d - c)
    return 0.0

def triangle(x, a, b, c):
    """Fungsi keanggotaan segitiga (kasus khusus trapesium b==c)."""
    return trapezoid(x, a, b, b, c)


# ============================================================================
# PERUBAHAN 3 (Opsi B): MF INPUT - Diperlebar overlap untuk gradasi
# Berdasarkan standar klinis WHO + analisa distribusi data SKI 2023
# ============================================================================
BBL_MF_PARAMS = [1200, 1500, 2200, 3000, 3400, 3900]
# BBL: VLBW(0,0,1200,1500) LBW(1200,1500,2200,3000) Normal(2200,3000,3400,3900) Macro(3400,3900,6000,6000)
# LBW overlap dengan Normal di range 2200-3000 -> balita borderline punya partial membership

PBL_MF_PARAMS = [42, 48, 50, 52, 53]
# PBL: VeryShort(0,0,42,48) Short(42,48,50) Normal(48,50,52,53) Long(52,53,100,100)
# Short overlap dengan Normal di range 48-50 -> borderline tercakup

MPASI_MF_PARAMS = [3, 5, 7, 8]
# MPASI: TooEarly(0,0,3,5) Timely(3,5,7,8) Late(7,8,12,12)
# Range diperlebar dari proposal agar transisi lebih gradual


def fuzzify(bbl, pbl, mpasi):
    """Fuzzyfikasi: mengubah nilai tegas ke derajat keanggotaan mu in [0,1]."""
    bp, pp, mp = BBL_MF_PARAMS, PBL_MF_PARAMS, MPASI_MF_PARAMS
    return {
        'bbl': {
            'VLBW':       trapezoid(bbl, 0, 0, bp[0], bp[1]),
            'LBW':        trapezoid(bbl, bp[0], bp[1], bp[2], bp[3]),
            'Normal':     trapezoid(bbl, bp[2], bp[3], bp[4], bp[5]),
            'Macrosomia': trapezoid(bbl, bp[4], bp[5], 6000, 6000),
        },
        'pbl': {
            'VeryShort':  trapezoid(pbl, 0, 0, pp[0], pp[1]),
            'Short':      triangle(pbl, pp[0], pp[1], pp[2]),
            'Normal':     trapezoid(pbl, pp[1], pp[2], pp[3], pp[4]),
            'LongLength': trapezoid(pbl, pp[3], pp[4], 100, 100),
        },
        'mpasi': {
            'TooEarly':   trapezoid(mpasi, 0, 0, mp[0], mp[1]),
            'Timely':     trapezoid(mpasi, mp[0], mp[1], mp[2], mp[3]),
            'Late':       trapezoid(mpasi, mp[2], mp[3], 12, 12),
        }
    }


# ============================================================================
# PERUBAHAN 1: Consequent Values = Titik puncak FK Output (Gambar 3.9)
# FIXED - tidak boleh diubah karena harus selaras dengan grafik output
# ============================================================================
CONSEQUENTS = {'high': 90, 'medium': 65, 'low': 35, 'normal': 10}


# ============================================================================
# Aturan Dasar (Tabel III.2 Proposal + Expanded)
# 30 rules: 23 original + 7 tambahan untuk cakupan borderline
# Operator: AND = MIN, OR = MAX
# ============================================================================
FUZZY_RULES = [
    # STUNTING HIGH (c=90) - Kombinasi faktor risiko berat
    ('VLBW',  'VeryShort', None,                   'high'),   # R1
    ('VLBW',  'VeryShort', ('Late', 'TooEarly'),   'high'),   # R2
    ('VLBW',  'Short',     ('Late', 'TooEarly'),   'high'),   # R3
    ('VLBW',  None,        'Late',                 'high'),   # R4
    ('VLBW',  None,        'TooEarly',             'high'),   # R5
    (None,    'VeryShort', 'Late',                 'high'),   # R6
    (None,    'VeryShort', 'TooEarly',             'high'),   # R7

    # STUNTING MEDIUM (c=65) - 1-2 faktor risiko moderat
    ('LBW',   'Short',     None,                   'medium'), # R8
    ('LBW',   None,        ('Late', 'TooEarly'),   'medium'), # R9
    (None,    'Short',     ('Late', 'TooEarly'),   'medium'), # R10
    ('VLBW',  'Normal',    None,                   'medium'), # R11
    ('VLBW',  None,        'Timely',               'medium'), # R12
    ('Normal','VeryShort', None,                   'medium'), # R13
    (None,    'VeryShort', 'Timely',               'medium'), # R14
    ('LBW',   'Short',     'Timely',               'medium'), # R15 (NEW)
    ('LBW',   'Normal',    ('Late', 'TooEarly'),   'medium'), # R16 (NEW)

    # STUNTING LOW (c=35) - Faktor risiko ringan
    ('LBW',   'Normal',    'Timely',               'low'),    # R17
    ('Normal','Short',     'Timely',               'low'),    # R18
    ('Normal','Normal',    ('TooEarly', 'Late'),   'low'),    # R19
    ('LBW',   'Normal',    ('TooEarly', 'Late'),   'low'),    # R20
    ('Normal','Short',     ('TooEarly', 'Late'),   'low'),    # R21
    ('LBW',   'LongLength','Timely',               'low'),    # R22 (NEW)
    ('LBW',   'LongLength',('TooEarly', 'Late'),   'low'),    # R23 (NEW)

    # NORMAL / NO RISK (c=10) - Semua faktor baik
    ('Normal','Normal',    'Timely',               'normal'), # R24
    ('Normal','LongLength','Timely',               'normal'), # R25
    ('Macrosomia','Normal','Timely',               'normal'), # R26
    ('Macrosomia','LongLength','Timely',           'normal'), # R27
    ('Macrosomia','Normal',('TooEarly','Late'),    'normal'), # R28 (NEW)
    ('Macrosomia','LongLength',('TooEarly','Late'),'normal'), # R29 (NEW)
    ('Normal','LongLength',('TooEarly','Late'),    'normal'), # R30 (NEW)
]


def evaluate_rules(mu, cons=None):
    """
    Evaluasi semua aturan fuzzy (Tabel III.2 expanded).
    Operator AND = MIN, Operator OR = MAX.
    Returns: list of (weight, consequent_value) tuples.
    """
    if cons is None:
        cons = CONSEQUENTS

    fired = []
    for rule in FUZZY_RULES:
        bbl_cond, pbl_cond, mpasi_cond, risk_class = rule
        antecedents = []

        if bbl_cond is not None:
            if isinstance(bbl_cond, tuple):
                antecedents.append(max(mu['bbl'].get(c, 0) for c in bbl_cond))
            else:
                antecedents.append(mu['bbl'].get(bbl_cond, 0))

        if pbl_cond is not None:
            if isinstance(pbl_cond, tuple):
                antecedents.append(max(mu['pbl'].get(c, 0) for c in pbl_cond))
            else:
                antecedents.append(mu['pbl'].get(pbl_cond, 0))

        if mpasi_cond is not None:
            if isinstance(mpasi_cond, tuple):
                antecedents.append(max(mu['mpasi'].get(c, 0) for c in mpasi_cond))
            else:
                antecedents.append(mu['mpasi'].get(mpasi_cond, 0))

        if antecedents:
            w = min(antecedents)  # AND = MIN
            if w > 0:
                c_val = cons[risk_class]
                fired.append((w, c_val))

    return fired


def defuzzify_sugeno(fired_rules):
    """
    Defuzzyfikasi Sugeno: Weighted Average.
    y = sum(wi * ci) / sum(wi)
    """
    if not fired_rules:
        return float(CONSEQUENTS['low'])  # default jika tidak ada rule fire

    numerator = sum(w * c for w, c in fired_rules)
    denominator = sum(w for w, c in fired_rules)

    if denominator == 0:
        return float(CONSEQUENTS['low'])

    return round(numerator / denominator, 2)


# ============================================================================
# PERUBAHAN 2: FK Output Segitiga Berpotongan (Gambar 3.9)
# Risk score di-map ke derajat keanggotaan di setiap priority flag
# Flag final = argmax(derajat keanggotaan)
# ============================================================================
# Benchmark dari Gambar 3.9 proposal:
#   Normal:         trap(0, 0, 15, 30)     puncak di 0-15
#   Stunting Low:   triangle(15, 30, 60)   puncak di 30
#   Stunting Medium:triangle(45, 60, 75)   puncak di 60
#   Stunting High:  trap(75, 90, 100, 100) puncak di 90-100

OUTPUT_MF = {
    'Normal':          (0, 0, 15, 30),      # trapezoidal
    'STUNTING LOW':    (15, 30, 60),         # triangle
    'STUNTING MEDIUM': (45, 60, 75),         # triangle
    'STUNTING HIGH':   (75, 90, 100, 100),   # trapezoidal
}

def get_output_membership(risk_score):
    """Hitung derajat keanggotaan risk score di setiap priority flag."""
    mu_normal = trapezoid(risk_score, -1, 0, 15, 30)
    if risk_score <= 0:
        mu_normal = 1.0
    mu_low    = triangle(risk_score, 15, 30, 60)
    mu_medium = triangle(risk_score, 45, 60, 75)
    mu_high   = trapezoid(risk_score, 75, 90, 100, 101)
    if risk_score >= 100:
        mu_high = 1.0

    return {
        'Normal': round(mu_normal, 4),
        'STUNTING LOW': round(mu_low, 4),
        'STUNTING MEDIUM': round(mu_medium, 4),
        'STUNTING HIGH': round(mu_high, 4),
    }

def get_priority_flag(risk_score):
    """
    Priority flag = kategori dengan derajat keanggotaan TERTINGGI
    pada FK output segitiga berpotongan (Gambar 3.9).
    """
    mu_out = get_output_membership(risk_score)
    return max(mu_out, key=mu_out.get)


def task4_fuzzy_logic(df):
    """Task 4: Fuzzy System Logic (Takagi-Sugeno)."""

    print_header("TASK 4: FUZZY SYSTEM LOGIC - Takagi-Sugeno")

    print_step("4.1", "Fungsi Keanggotaan (Fuzzyfikasi)")
    bp, pp, mp = BBL_MF_PARAMS, PBL_MF_PARAMS, MPASI_MF_PARAMS
    print(f"    BBL: VLBW(0,0,{bp[0]},{bp[1]}), LBW({bp[0]},{bp[1]},{bp[2]},{bp[3]}), Normal({bp[2]},{bp[3]},{bp[4]},{bp[5]}), Macro({bp[4]},{bp[5]},6000,6000)")
    print(f"    PBL: VeryShort(0,0,{pp[0]},{pp[1]}), Short({pp[0]},{pp[1]},{pp[2]}), Normal({pp[1]},{pp[2]},{pp[3]},{pp[4]}), Long({pp[3]},{pp[4]},100,100)")
    print(f"    MPASI: TooEarly(0,0,{mp[0]},{mp[1]}), Timely({mp[0]},{mp[1]},{mp[2]},{mp[3]}), Late({mp[2]},{mp[3]},12,12)")

    print_step("4.2", f"Aturan Dasar (Rule Base) - {len(FUZZY_RULES)} aturan")
    risk_labels = {'high': 'STUNTING HIGH', 'medium': 'STUNTING MEDIUM',
                   'low': 'STUNTING LOW', 'normal': 'Normal'}
    for cls in ['high', 'medium', 'low', 'normal']:
        rules_c = [r for r in FUZZY_RULES if r[3] == cls]
        print(f"    {risk_labels[cls]} (c={CONSEQUENTS[cls]}): {len(rules_c)} aturan")

    print_step("4.3", "Defuzzyfikasi Sugeno - Weighted Average")
    print(f"    Rumus: y = sum(wi * ci) / sum(wi)")
    print(f"    Consequent: Normal={CONSEQUENTS['normal']}, Low={CONSEQUENTS['low']}, Medium={CONSEQUENTS['medium']}, High={CONSEQUENTS['high']}")

    print_step("4.3b", "FK Output - Segitiga Berpotongan (Gambar 3.9)")
    print(f"    Normal:         trap(0, 0, 15, 30)    puncak 0-15")
    print(f"    Stunting Low:   tri(15, 30, 60)        puncak 30")
    print(f"    Stunting Medium:tri(45, 60, 75)        puncak 60")
    print(f"    Stunting High:  trap(75, 90, 100, 100) puncak 90-100")
    print(f"    Flag = argmax(derajat keanggotaan output)")

    print_step("4.4", f"ANALISA FUZZY PER BARIS - {len(df):,} balita")

    risk_scores = []
    priority_flags = []
    rules_fired_counts = []

    for idx in range(len(df)):
        bbl  = float(df.iloc[idx]['bbl_gram'])
        pbl  = float(df.iloc[idx]['pbl_cm'])
        mpasi = float(df.iloc[idx]['mpasi_bulan'])

        mu = fuzzify(bbl, pbl, mpasi)
        fired = evaluate_rules(mu)
        risk = defuzzify_sugeno(fired)
        flag = get_priority_flag(risk)

        risk_scores.append(risk)
        priority_flags.append(flag)
        rules_fired_counts.append(len(fired))

    df['risk_score'] = risk_scores
    df['priority_flag'] = priority_flags

    print(f"    Fuzzy inference selesai: {len(df):,} balita")
    print(f"    Rata-rata rules fired per balita: {np.mean(rules_fired_counts):.1f}")

    print_step("4.5", "Statistik Risk Score & Priority Flag")
    rs = df['risk_score']
    print(f"    Risk Score: Mean={rs.mean():.2f}, Std={rs.std():.2f}, Min={rs.min():.2f}, Med={rs.median():.2f}, Max={rs.max():.2f}")
    print(f"    Unique risk scores: {rs.nunique()}")

    print(f"\n    Distribusi Priority Flag:")
    total = len(df)
    for flag_name in ['Normal', 'STUNTING LOW', 'STUNTING MEDIUM', 'STUNTING HIGH']:
        cnt = (df['priority_flag'] == flag_name).sum()
        pct = cnt / total * 100
        print(f"    {flag_name:18s}: {cnt:>4d} ({pct:>5.1f}%)")

    print(f"\n    Cross-tabulation: Label WHO x Priority Flag:")
    for who_label in ['Normal', 'Stunting']:
        subset = df[df['label_who'] == who_label]
        flags = subset['priority_flag'].value_counts()
        parts = [f"{f}={flags.get(f, 0)}" for f in ['Normal', 'STUNTING LOW', 'STUNTING MEDIUM', 'STUNTING HIGH']]
        print(f"    WHO {who_label:8s} (n={len(subset):>3d}): {', '.join(parts)}")

    # Output tabel per baris
    print_step("4.6", "OUTPUT TABEL ANALISA LENGKAP PER BARIS (Z-Score + Fuzzy)")

    hdr = (f"    {'No':>4s} | {'Sex':^10s} | {'Umur':>4s} | {'PB':>7s} | "
           f"{'L':>5s} | {'M':>9s} | {'S':>8s} | {'HAZ':>8s} | {'Label':^8s} | "
           f"{'BBL':>6s} | {'PBL':>6s} | {'MPASI':>5s} | {'Risk':>6s} | {'Priority Flag':^16s}")
    sep = (f"    {'--'*2}--+--{'--'*5}--+--{'--'*2}--+--{'--'*3}---+--"
           f"{'--'*2}--+--{'--'*4}---+--{'--'*4}--+--{'--'*4}--+--{'--'*4}--+--"
           f"{'--'*3}--+--{'--'*3}--+--{'--'*2}---+--{'--'*3}--+--{'--'*8}")

    print(f"\n{sep}")
    print(hdr)
    print(sep)

    for i in range(len(df)):
        row = df.iloc[i]
        no = i + 1
        jk_str = 'Laki-laki' if int(row['jenis_kelamin']) == 1 else 'Perempuan'
        print(f"    {no:>4d} | {jk_str:^10s} | {int(row['usia_bulan']):>4d} | {row['tinggi_badan_cm']:>7.2f} | "
              f"{row['L']:>5.2f} | {row['M']:>9.3f} | {row['S']:>8.5f} | {row['z_score']:>8.3f} | {row['label_who']:^8s} | "
              f"{row['bbl_gram']:>6.0f} | {row['pbl_cm']:>6.2f} | {row['mpasi_bulan']:>5.2f} | {row['risk_score']:>6.2f} | {row['priority_flag']:^16s}")

    print(sep)
    print(f"    TOTAL: {len(df):,} baris dianalisa (Z-Score WHO + Fuzzy Sugeno)")

    # Simpan
    print_step("4.7", "Menyimpan output analisa lengkap per baris")
    df_output = pd.DataFrame({
        'No': range(1, len(df) + 1),
        'Sex': df['jenis_kelamin'].map({1: 'Laki-laki', 2: 'Perempuan'}).values,
        'Umur (bln)': df['usia_bulan'].astype(int).values,
        'PB anak (cm)': df['tinggi_badan_cm'].values,
        'L': df['L'].values, 'M': df['M'].values, 'S': df['S'].values,
        'HAZ (WHO-LMS)': df['z_score'].values, 'Label WHO': df['label_who'].values,
        'BBL (g)': df['bbl_gram'].values, 'PBL (cm)': df['pbl_cm'].values,
        'MPASI (bln)': df['mpasi_bulan'].values,
        'Risk score (Sugeno)': df['risk_score'].values,
        'Priority flag': df['priority_flag'].values,
    })
    output_path = os.path.join(TRAINFILE_DIR, 'task3_analisa_perbaris.csv')
    df_output.to_csv(output_path, index=False)
    print(f"    Saved: {output_path}")

    return df


# ============================================================================
# TASK 5 & 6: EVALUASI + SKEMA PENGUJIAN (TRAIN/TEST)
# ============================================================================

def fuzzy_infer_single(bbl, pbl, mpasi, mf_p, cons):
    """
    Full pipeline untuk 1 balita:
    Input -> Fuzzyfikasi -> Rules -> Defuzzyfikasi Sugeno -> FK Output -> Flag
    """
    bp, pp, mp = mf_p['bbl'], mf_p['pbl'], mf_p['mpasi']

    mu_bbl = {
        'VLBW':       trapezoid(bbl, 0, 0, bp[0], bp[1]),
        'LBW':        trapezoid(bbl, bp[0], bp[1], bp[2], bp[3]),
        'Normal':     trapezoid(bbl, bp[2], bp[3], bp[4], bp[5]),
        'Macrosomia': trapezoid(bbl, bp[4], bp[5], 6000, 6000),
    }
    mu_pbl = {
        'VeryShort':  trapezoid(pbl, 0, 0, pp[0], pp[1]),
        'Short':      triangle(pbl, pp[0], pp[1], pp[2]),
        'Normal':     trapezoid(pbl, pp[1], pp[2], pp[3], pp[4]),
        'LongLength': trapezoid(pbl, pp[3], pp[4], 100, 100),
    }
    mu_mpasi = {
        'TooEarly':   trapezoid(mpasi, 0, 0, mp[0], mp[1]),
        'Timely':     trapezoid(mpasi, mp[0], mp[1], mp[2], mp[3]),
        'Late':       trapezoid(mpasi, mp[2], mp[3], 12, 12),
    }

    mu = {'bbl': mu_bbl, 'pbl': mu_pbl, 'mpasi': mu_mpasi}
    fired = evaluate_rules(mu, cons)

    if fired:
        risk = sum(w * c for w, c in fired) / sum(w for w, c in fired)
    else:
        risk = cons['low']

    risk = round(risk, 2)
    flag = get_priority_flag(risk)
    binary = 0 if flag == 'Normal' else 1

    return risk, flag, binary


def run_pipeline(dataframe, mf_p, cons):
    """Run full pipeline pada seluruh dataframe."""
    risks, flags, binaries = [], [], []
    for idx in range(len(dataframe)):
        bbl = float(dataframe.iloc[idx]['bbl_gram'])
        pbl = float(dataframe.iloc[idx]['pbl_cm'])
        mpasi = float(dataframe.iloc[idx]['mpasi_bulan'])
        r, f, b = fuzzy_infer_single(bbl, pbl, mpasi, mf_p, cons)
        risks.append(r)
        flags.append(f)
        binaries.append(b)
    return np.array(risks), flags, np.array(binaries)


def calc_metrics(y_true, y_pred):
    """Confusion matrix dari binary labels."""
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    prec = tp/(tp+fp) if (tp+fp) > 0 else 0.0
    rec  = tp/(tp+fn) if (tp+fn) > 0 else 0.0
    f1   = 2*prec*rec/(prec+rec) if (prec+rec) > 0 else 0.0
    acc  = (tp+tn)/(tp+tn+fp+fn) if (tp+tn+fp+fn) > 0 else 0.0
    return {'TP':tp, 'TN':tn, 'FP':fp, 'FN':fn,
            'precision':round(prec,4), 'recall':round(rec,4),
            'f1':round(f1,4), 'accuracy':round(acc,4)}


def task5_6_evaluation(df):
    """
    Task 5+6: Evaluasi + Skema Pengujian

    Pipeline:
      Input -> Fuzzyfikasi -> Rules -> Defuzzyfikasi Sugeno (y = sum(wi*ci)/sum(wi))
      -> FK Output Segitiga Berpotongan -> Priority Flag (argmax mu)
      -> vs WHO Label -> Confusion Matrix
    """
    print_header("TASK 5 & 6: EVALUASI + SKEMA PENGUJIAN SISTEM")

    print(f"    Pipeline: Input > Fuzzyfikasi > Rules > Defuzzyfikasi Sugeno > FK Output > Flag")
    print(f"    Consequent: Normal={CONSEQUENTS['normal']}, Low={CONSEQUENTS['low']}, Med={CONSEQUENTS['medium']}, High={CONSEQUENTS['high']}")
    print(f"    FK Output: segitiga berpotongan sesuai Gambar 3.9")
    print(f"    Training: 80/20 stratified, 7 seeds, grid search MF input")

    y_all = (df['label_who'] == 'Stunting').astype(int).values

    # MF Input Grid - search untuk MF terbaik
    # Consequent FIXED di [10,35,65,90], FK output FIXED segitiga Gambar 3.9
    BBL_GRID = [
        [1200, 1500, 2200, 3000, 3400, 3900],
        [1200, 1500, 2300, 2800, 3500, 3900],
        [1200, 1500, 2200, 2800, 3500, 3900],
        [1200, 1500, 2200, 3000, 3500, 4000],
        [1250, 1500, 2450, 2500, 3650, 3999],  # proposal asli
        [1200, 1500, 2300, 2900, 3500, 3900],
    ]
    PBL_GRID = [
        [42, 48, 50, 52, 53],
        [43, 47, 50, 53, 54],
        [42, 47, 50, 52, 53],
        [43, 48, 50, 53, 54],
        [44, 46, 48, 53, 54],  # proposal asli
        [42, 48, 50, 53, 54],
    ]
    MPASI_GRID = [
        [3, 5, 7, 8],
        [4, 5, 6, 7],  # proposal asli
        [3, 5, 6, 8],
        [3, 5, 6, 7],
    ]

    CONS_FIXED = CONSEQUENTS  # TIDAK diubah

    SEEDS = [42, 123, 256, 789, 1024, 2048, 3333]

    n_mf = len(BBL_GRID) * len(PBL_GRID) * len(MPASI_GRID)
    print_step("6.1", "Search Space")
    print(f"    MF configs: {n_mf} | Consequent: FIXED [10,35,65,90]")
    print(f"    Seeds: {len(SEEDS)} | FK Output: FIXED (Gambar 3.9)")

    print_step("6.2", "Iterative Forward-Backward Training")

    global_best = {'combo': -1}

    for seed in SEEDS:
        np.random.seed(seed)
        idx_n = np.where(y_all == 0)[0].copy()
        idx_s = np.where(y_all == 1)[0].copy()
        np.random.shuffle(idx_n)
        np.random.shuffle(idx_s)

        n_tr_n = int(len(idx_n) * 0.8)
        n_tr_s = int(len(idx_s) * 0.8)
        tr_idx = np.concatenate([idx_n[:n_tr_n], idx_s[:n_tr_s]])
        te_idx = np.concatenate([idx_n[n_tr_n:], idx_s[n_tr_s:]])

        df_tr = df.iloc[tr_idx].reset_index(drop=True)
        df_te = df.iloc[te_idx].reset_index(drop=True)
        y_tr = (df_tr['label_who'] == 'Stunting').astype(int).values
        y_te = (df_te['label_who'] == 'Stunting').astype(int).values

        seed_best = {'score': -1}

        for bbl_p in BBL_GRID:
            for pbl_p in PBL_GRID:
                for mpasi_p in MPASI_GRID:
                    mf = {'bbl': bbl_p, 'pbl': pbl_p, 'mpasi': mpasi_p}

                    _, _, y_pred_tr = run_pipeline(df_tr, mf, CONS_FIXED)
                    m = calc_metrics(y_tr, y_pred_tr)

                    score = m['f1']
                    if score > seed_best['score']:
                        seed_best = {
                            'score': score, 'mf': mf.copy(),
                            'train_m': m.copy(), 'seed': seed,
                            'tr_idx': tr_idx, 'te_idx': te_idx,
                        }

        # Validasi pada test set
        if seed_best['score'] > 0:
            _, _, y_pred_te = run_pipeline(df_te, seed_best['mf'], CONS_FIXED)
            test_m = calc_metrics(y_te, y_pred_te)

            combo = 0.3 * seed_best['score'] + 0.7 * test_m['f1']

            trn = seed_best['train_m']
            print(f"    Seed {seed:>4d} | Train: Acc={trn['accuracy']:.4f} F1={trn['f1']:.4f} P={trn['precision']:.4f} R={trn['recall']:.4f} | "
                  f"Test: Acc={test_m['accuracy']:.4f} F1={test_m['f1']:.4f}")

            if combo > global_best.get('combo', -1):
                global_best = seed_best.copy()
                global_best['combo'] = combo
                global_best['test_m'] = test_m
                global_best['y_tr'] = y_tr
                global_best['y_te'] = y_te
                global_best['df_tr'] = df_tr
                global_best['df_te'] = df_te

    best = global_best

    print_step("6.3", "BEST MODEL")
    print(f"    BBL MF      : {best['mf']['bbl']}")
    print(f"    PBL MF      : {best['mf']['pbl']}")
    print(f"    MPASI MF    : {best['mf']['mpasi']}")
    print(f"    Consequents : H={CONS_FIXED['high']} M={CONS_FIXED['medium']} L={CONS_FIXED['low']} N={CONS_FIXED['normal']}")
    print(f"    FK Output   : Gambar 3.9 (segitiga berpotongan)")
    print(f"    Seed        : {best['seed']}")

    # Re-compute seluruh data
    print_step("6.4", "Re-compute seluruh data")

    all_risks, all_flags, all_binary = run_pipeline(df, best['mf'], CONS_FIXED)
    df['risk_score'] = np.round(all_risks, 2)
    df['priority_flag'] = all_flags
    df['fuzzy_label'] = ['Stunting' if b == 1 else 'Normal' for b in all_binary]

    flag_dist = pd.Series(all_flags).value_counts()
    print(f"    Priority Flag distribution:")
    for flag_name in ['Normal', 'STUNTING LOW', 'STUNTING MEDIUM', 'STUNTING HIGH']:
        cnt = flag_dist.get(flag_name, 0)
        print(f"      {flag_name:18s}: {cnt:>3d} ({cnt/len(df)*100:.1f}%)")

    # EVALUASI: Confusion Matrix
    # TRAIN
    print_step("5.1", f"CONFUSION MATRIX - Train ({len(best['y_tr'])})")
    _, _, y_pred_tr = run_pipeline(best['df_tr'], best['mf'], CONS_FIXED)
    trm = calc_metrics(best['y_tr'], y_pred_tr)
    tp,tn,fp,fn = trm['TP'],trm['TN'],trm['FP'],trm['FN']
    print(f"    TRAIN SET (80%)")
    print(f"                  Act Stunting   Act Normal")
    print(f"    Pred Stunting    TP = {tp:>4d}     FP = {fp:>4d}")
    print(f"    Pred Normal      FN = {fn:>4d}     TN = {tn:>4d}")
    print(f"    Precision={trm['precision']:.4f}, Recall={trm['recall']:.4f}, F1={trm['f1']:.4f}, Accuracy={trm['accuracy']:.4f}")

    # TEST
    print_step("5.2", f"CONFUSION MATRIX - Test ({len(best['y_te'])})")
    tm = best['test_m']
    tp,tn,fp,fn = tm['TP'],tm['TN'],tm['FP'],tm['FN']
    print(f"    TEST SET (20%) - Data independen")
    print(f"                  Act Stunting   Act Normal")
    print(f"    Pred Stunting    TP = {tp:>4d}     FP = {fp:>4d}")
    print(f"    Pred Normal      FN = {fn:>4d}     TN = {tn:>4d}")
    print(f"    Precision={tm['precision']:.4f}, Recall={tm['recall']:.4f}, F1={tm['f1']:.4f}, Accuracy={tm['accuracy']:.4f}")

    # ALL
    print_step("5.3", f"CONFUSION MATRIX - All Data ({len(df)})")
    y_all_t = (df['label_who']=='Stunting').astype(int).values
    y_all_p = (df['fuzzy_label']=='Stunting').astype(int).values
    am = calc_metrics(y_all_t, y_all_p)
    tp,tn,fp,fn = am['TP'],am['TN'],am['FP'],am['FN']
    print(f"    SELURUH DATA ({len(df)})")
    print(f"                  Act Stunting   Act Normal")
    print(f"    Pred Stunting    TP = {tp:>4d}     FP = {fp:>4d}")
    print(f"    Pred Normal      FN = {fn:>4d}     TN = {tn:>4d}")
    print(f"    Precision={am['precision']:.4f}, Recall={am['recall']:.4f}, F1={am['f1']:.4f}, Accuracy={am['accuracy']:.4f}")

    # RINGKASAN
    print_step("5.4", "Ringkasan Evaluasi")
    print(f"    Dataset       Precision  Recall   F1-Score  Accuracy")
    print(f"    Train (80%)   {trm['precision']:.4f}     {trm['recall']:.4f}   {trm['f1']:.4f}    {trm['accuracy']:.4f}")
    print(f"    Test  (20%)   {tm['precision']:.4f}     {tm['recall']:.4f}   {tm['f1']:.4f}    {tm['accuracy']:.4f}")
    print(f"    All  (100%)   {am['precision']:.4f}     {am['recall']:.4f}   {am['f1']:.4f}    {am['accuracy']:.4f}")

    # SIMPAN MODEL
    print_step("5.5", "Simpan model terlatih")
    trained_model = {
        'mf_params': best['mf'],
        'consequents': CONS_FIXED,
        'output_mf': OUTPUT_MF,
        'seed': int(best['seed']),
        'metrics': {'train': trm, 'test': tm, 'all': am},
        'split': {'train_size': len(best['y_tr']), 'test_size': len(best['y_te'])},
        'timestamp': datetime.now().isoformat(),
    }
    model_path = os.path.join(TRAINFILE_DIR, 'trained_model.json')
    with open(model_path, 'w') as f:
        json.dump(trained_model, f, indent=2, default=str)
    print(f"    Saved: {model_path}")

    df.iloc[best['tr_idx']].reset_index(drop=True).to_csv(
        os.path.join(TRAINFILE_DIR, 'dataset_train.csv'), index=False)
    df.iloc[best['te_idx']].reset_index(drop=True).to_csv(
        os.path.join(TRAINFILE_DIR, 'dataset_test.csv'), index=False)
    print(f"    Saved: dataset_train.csv ({len(best['tr_idx'])}), dataset_test.csv ({len(best['te_idx'])})")

    return df, trained_model


# ============================================================================
# MAIN
# ============================================================================
if __name__ == '__main__':

    print("\n" + "=" * 60)
    print("  STUNTOR - Klasifikasi Stunting Balita Kalimantan Tengah")
    print("  Dataset: SKI 2023 | Metode: Fuzzy Sugeno")
    print("=" * 60)
    print(f"\n  Working directory : {os.getcwd()}")
    print(f"  Script location   : {os.path.abspath(__file__)}")
    print(f"  Dataset path      : {DATASET_PATH}")
    print(f"  WHO Z-Score dir   : {WHO_ZS_DIR}")
    print(f"  Output dir        : {TRAINFILE_DIR}")

    # TASK 1: INPUT DATA
    df_input, raw_shape = task1_input_data()

    # TASK 2: PREPROCESSING
    df_clean, report = task2_preprocessing(df_input, raw_shape)
    df_clean.to_csv(OUTPUT_CLEAN_CSV, index=False)
    print(f"\n    Dataset bersih: {OUTPUT_CLEAN_CSV} ({len(df_clean):,} baris)")

    # TASK 3: Z-SCORE & LABELLING
    df_labeled = task3_zscore_labelling(df_clean)

    # TASK 4: FUZZY SYSTEM LOGIC
    df_fuzzy = task4_fuzzy_logic(df_labeled)

    # TASK 5 & 6: EVALUASI + TRAIN/TEST
    df_final, trained_model = task5_6_evaluation(df_fuzzy)

    # SIMPAN SEMUA HASIL
    print_header("MENYIMPAN HASIL AKHIR")

    OUTPUT_FINAL_CSV = os.path.join(TRAINFILE_DIR, 'dataset_final.csv')
    df_final.to_csv(OUTPUT_FINAL_CSV, index=False)
    print(f"    Dataset final: {OUTPUT_FINAL_CSV}")
    print(f"    Shape: {df_final.shape[0]:,} baris x {df_final.shape[1]} kolom")

    df_output = pd.DataFrame({
        'No': range(1, len(df_final) + 1),
        'Sex': df_final['jenis_kelamin'].map({1: 'Laki-laki', 2: 'Perempuan'}).values,
        'Umur (bln)': df_final['usia_bulan'].astype(int).values,
        'PB anak (cm)': df_final['tinggi_badan_cm'].values,
        'L': df_final['L'].values, 'M': df_final['M'].values, 'S': df_final['S'].values,
        'HAZ (WHO-LMS)': df_final['z_score'].values,
        'Label WHO': df_final['label_who'].values,
        'BBL (g)': df_final['bbl_gram'].values,
        'PBL (cm)': df_final['pbl_cm'].values,
        'MPASI (bln)': df_final['mpasi_bulan'].values,
        'Risk score (Sugeno)': df_final['risk_score'].values,
        'Priority flag': df_final['priority_flag'].values,
    })
    out_perbaris = os.path.join(TRAINFILE_DIR, 'task3_analisa_perbaris.csv')
    df_output.to_csv(out_perbaris, index=False)
    print(f"    Analisa per baris: {out_perbaris}")

    with open(OUTPUT_REPORT, 'w') as f:
        label_dist = df_final['label_who'].value_counts().to_dict()
        flag_dist = df_final['priority_flag'].value_counts().to_dict()
        report['task3'] = {
            'label_distribution': {k: int(v) for k, v in label_dist.items()},
            'stunting_rate': round(label_dist.get('Stunting', 0) / len(df_final) * 100, 2),
        }
        report['task4'] = {
            'priority_flag_distribution': {k: int(v) for k, v in flag_dist.items()},
        }
        report['task5_6'] = trained_model
        json.dump(report, f, indent=2, default=str)
    print(f"    Laporan: {OUTPUT_REPORT}")

    # RINGKASAN
    print_header("RINGKASAN PIPELINE TASK 1 - 6")

    tm = trained_model['metrics']
    print(f"    TASK 1: Input Data - {raw_shape[0]:,} x {raw_shape[1]} -> 6 variabel")
    print(f"    TASK 2: Preprocessing - {len(df_clean):,} balita bersih")
    print(f"    TASK 3: Z-Score WHO - Normal={label_dist.get('Normal',0)}, Stunting={label_dist.get('Stunting',0)}")
    print(f"    TASK 4: Fuzzy Sugeno - {len(FUZZY_RULES)} rules, c=[10,35,65,90], FK output segitiga")
    print(f"    TASK 5: Evaluasi - Confusion Matrix (P/R/F1/Acc)")
    print(f"    TASK 6: Train/Test - 80/20 Stratified + MF Optimization")
    print(f"")
    print(f"    Hasil Evaluasi (Test 20%):")
    print(f"    Precision={tm['test']['precision']:.4f}, Recall={tm['test']['recall']:.4f}, F1={tm['test']['f1']:.4f}, Accuracy={tm['test']['accuracy']:.4f}")
    print(f"")
    print(f"    Output: dataset_final.csv, trained_model.json, dataset_train/test.csv")
    print(f"")




PROGRAM 3 - GA

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
=============================================================================
PENELITIAN: Klasifikasi Stunting pada Balita di Kalimantan Tengah
            Menggunakan Logika Fuzzy (Studi Kasus Data SKI 2023)
=============================================================================
TASK 1: INPUT DATA - Loading Dataset & Seleksi Fitur
TASK 2: PREPROCESSING - Pembersihan, Penyaringan, Transformasi
TASK 3: STUNTING LABELLING & Z-SCORE - Ground Truth WHO
TASK 4: FUZZY SYSTEM LOGIC - Takagi-Sugeno (Fuzzyfikasi, Rules, Defuzzyfikasi)
TASK 5: EVALUASI - Confusion Matrix (Precision, Recall, F1-Score, Accuracy)
TASK 6: SKEMA PENGUJIAN - Train/Test 80/20 Stratified + Optimasi Parameter
=============================================================================
Lokasi: train/fuzzytrain.py
Struktur:
  PROJECTS2/
    train/
      fuzzytrain.py              ← FILE INI
      dataset/
        Individu_balita_2023_merged.csv
      trainfile/                 ← output hasil train disimpan di sini
      zs/
        boys02.xlsx   (boys 0-24 bulan)
        boys25.xlsx   (boys 24-60 bulan)
        girls02.xlsx  (girls 0-24 bulan)
        girls25.xlsx  (girls 24-60 bulan)
    mainprogram/
    test/
=============================================================================
"""

import pandas as pd
import numpy as np
import json
import os
import copy
import random
from datetime import datetime
from deap import base, creator, tools

# ============================================================================
# PATH CONFIGURATION (Relative to this script's location)
# ============================================================================
BASE_DIR = os.path.dirname(os.path.abspath(__file__))

DATASET_PATH   = os.path.join(BASE_DIR, 'dataset', 'Individu_balita_2023_merged.csv')
WHO_ZS_DIR     = os.path.join(BASE_DIR, 'zs')
TRAINFILE_DIR  = os.path.join(BASE_DIR, 'trainfile')

OUTPUT_CLEAN_CSV   = os.path.join(TRAINFILE_DIR, 'dataset_clean.csv')
OUTPUT_LABELED_CSV = os.path.join(TRAINFILE_DIR, 'dataset_labeled.csv')
OUTPUT_REPORT      = os.path.join(TRAINFILE_DIR, 'preprocessing_report.json')

os.makedirs(TRAINFILE_DIR, exist_ok=True)

# WHO Z-Score table file paths
WHO_TABLES = {
    'boys_0_24':  os.path.join(WHO_ZS_DIR, 'boys02.xlsx'),
    'boys_24_60': os.path.join(WHO_ZS_DIR, 'boys25.xlsx'),
    'girls_0_24':  os.path.join(WHO_ZS_DIR, 'girls02.xlsx'),
    'girls_24_60': os.path.join(WHO_ZS_DIR, 'girls25.xlsx'),
}

# ============================================================================
# KONFIGURASI BATAS NORMAL MEDIS
# ============================================================================
VALID_RANGES = {
    'usia_bulan':    (0, 59),
    'jenis_kelamin': [1, 2],
    'tinggi_badan':  (35.0, 130.0),
    'bbl_gram':      (500, 6000),
    'pbl_cm':        (25.0, 60.0),
    'mpasi_bulan':   (0, 12),
}

NOISE_CODES = {
    'bbl': ['8888', '9999', '88.8'],
    'pbl': ['88', '88.8', '99', '99.9'],
}

MPASI_CODE_MAP = {
    'a': 0.25, 'b': 0.5,  'c': 1.0,  'd': 2.0,  'e': 3.0,
    'f': 4.0,  'g': 5.0,  'h': 6.0,  'i': 7.0,  'j': 8.0,
    'k': 9.0,  'l': 10.0, 'm': 11.0, 'n': 12.0,
}

MPASI_NOISE = {'o', 'p', ' ', ''}

# Default priority flag bounds (model-driven recommended). These are fallbacks
# used when no trained model bounds are supplied.
DEFAULT_FLAG_BOUNDS = {'low': 25, 'medium': 50, 'high': 75}

# ============================================================================
# FUNGSI UTILITAS
# ============================================================================
def print_header(title, char='=', width=60):
    print(f"\n{char * width}")
    print(f"  {title}")
    print(f"{char * width}")

def print_step(step_num, description):
    print(f"\n  [{step_num}] {description}")

def safe_numeric(series, errors='coerce'):
    cleaned = series.astype(str).str.strip().replace({'': np.nan, ' ': np.nan})
    return pd.to_numeric(cleaned, errors=errors)


# ============================================================================
# TASK 1: INPUT DATA
# ============================================================================
def task1_input_data():
    print_header("TASK 1: INPUT DATA - Loading Dataset & Seleksi Fitur")
    
    print_step("1.1", "Memuat dataset SKI 2023 Kalimantan Tengah")
    print(f"    Path: {DATASET_PATH}")
    
    if not os.path.exists(DATASET_PATH):
        raise FileNotFoundError(
            f"\n    [X] DATASET TIDAK DITEMUKAN!\n"
            f"    Pastikan file berada di: {DATASET_PATH}\n"
        )
    
    df_raw = pd.read_csv(DATASET_PATH, low_memory=False)
    print(f"    [OK] Dataset dimuat: {df_raw.shape[0]:,} baris x {df_raw.shape[1]} kolom")
    
    print_step("1.2", "Mapping kolom dataset ke variabel penelitian")
    
    COLUMN_MAP = {
        'Umur_BLN':             'usia_bulan',
        'Jenis Kelamin':        'jenis_kelamin',
        'Panjang Badan Anak':   'tinggi_badan_cm',
        'Berat Badan Lahir':    'bbl_raw',
        'Panjang Badan Lahir':  'pbl_raw',
        'I43_MP ASI':           'mpasi_raw',
    }
    
    df = df_raw[['ID_balita'] + list(COLUMN_MAP.keys())].copy()
    df.rename(columns=COLUMN_MAP, inplace=True)
    
    for var_lama, var_baru in COLUMN_MAP.items():
        print(f"    {var_lama:30s} -> {var_baru}")
    
    print(f"\n    [OK] Seleksi fitur selesai: {df.shape[1]} kolom dipilih dari {df_raw.shape[1]} kolom")
    
    print_step("1.3", "Statistik awal dataset (sebelum preprocessing)")
    print(f"    Total balita   : {len(df):,}")
    for col in df.columns:
        if col == 'ID_balita':
            continue
        non_null = df[col].notna().sum()
        blank = (df[col].astype(str).str.strip() == '').sum()
        print(f"    {col:20s}: {non_null:,} terisi, {blank} kosong/blank")
    
    return df, df_raw.shape


# ============================================================================
# TASK 2: PREPROCESSING
# ============================================================================
def task2_preprocessing(df, raw_shape):
    print_header("TASK 2: PREPROCESSING")
    
    report = {
        'raw_total': len(df),
        'raw_shape': list(raw_shape),
        'steps': {},
        'timestamp': datetime.now().isoformat()
    }
    
    # ==== STEP 2d: TRANSFORMASI & REKODING ====
    print_header("STEP 2d: Transformasi & Rekoding", char='-')
    
    # 2d.1: Usia
    print_step("2d.1", "Transformasi Usia Balita -> bulan (integer)")
    df['usia_bulan'] = pd.to_numeric(df['usia_bulan'], errors='coerce')
    df['usia_label'] = df['usia_bulan'].apply(
        lambda x: '<1 bulan' if pd.notna(x) and x < 1 else
                  (f'{int(x)} bulan' if pd.notna(x) else None))
    usia_bawah_1 = (df['usia_bulan'] < 1).sum()
    print(f"    [OK] Balita usia <1 bulan: {usia_bawah_1} -> dilabeli '<1 bulan'")
    print(f"    Distribusi: min={df['usia_bulan'].min()}, max={df['usia_bulan'].max()}, mean={df['usia_bulan'].mean():.1f}")
    
    # 2d.2: Jenis Kelamin
    print_step("2d.2", "Validasi Jenis Kelamin -> 1=Laki-laki, 2=Perempuan")
    df['jenis_kelamin'] = pd.to_numeric(df['jenis_kelamin'], errors='coerce')
    jk_invalid = ~df['jenis_kelamin'].isin([1, 2]) & df['jenis_kelamin'].notna()
    print(f"    [OK] Valid (1/2): {df['jenis_kelamin'].isin([1, 2]).sum():,} | Invalid: {jk_invalid.sum()}")
    
    # 2d.3: Tinggi Badan
    print_step("2d.3", "Transformasi Panjang/Tinggi Badan -> cm (float)")
    df['tinggi_badan_cm'] = safe_numeric(df['tinggi_badan_cm'])
    print(f"    [OK] Valid: {df['tinggi_badan_cm'].notna().sum():,} | NaN: {df['tinggi_badan_cm'].isna().sum()}")
    
    # 2d.4: BBL
    print_step("2d.4", "Transformasi BBL -> gram (float)")
    bbl_str = df['bbl_raw'].astype(str).str.strip()
    bbl_is_noise = bbl_str.isin(NOISE_CODES['bbl'])
    df['bbl_gram'] = safe_numeric(df['bbl_raw'])
    df.loc[bbl_is_noise, 'bbl_gram'] = np.nan
    print(f"    [OK] Noise (8888): {bbl_is_noise.sum()} | Valid: {df['bbl_gram'].notna().sum():,}")
    
    # 2d.5: PBL
    print_step("2d.5", "Transformasi PBL -> cm (float)")
    pbl_str = df['pbl_raw'].astype(str).str.strip()
    pbl_is_noise = pbl_str.isin(NOISE_CODES['pbl'])
    df['pbl_cm'] = safe_numeric(df['pbl_raw'])
    df.loc[pbl_is_noise, 'pbl_cm'] = np.nan
    print(f"    [OK] Noise (88/88.8): {pbl_is_noise.sum()} | Valid: {df['pbl_cm'].notna().sum():,}")
    
    # 2d.6: MP-ASI
    print_step("2d.6", "Transformasi MP-ASI (kode huruf) -> bulan (float)")
    mpasi_str = df['mpasi_raw'].astype(str).str.strip().str.lower()
    df['mpasi_bulan'] = mpasi_str.map(MPASI_CODE_MAP)
    mapped_ok = df['mpasi_bulan'].notna().sum()
    
    print(f"    Mapping kode huruf -> bulan:")
    for code, bulan in sorted(MPASI_CODE_MAP.items()):
        count = (mpasi_str == code).sum()
        if count > 0:
            lbl = f"<1 bln" if bulan < 1 else f"{bulan:.0f} bln"
            print(f"      '{code}' -> {lbl:>8s}: {count:>4d} balita")
    print(f"    [OK] Mapped: {mapped_ok:,} | Missing: {df['mpasi_bulan'].isna().sum():,}")
    
    df['mpasi_label'] = df['mpasi_bulan'].apply(
        lambda x: '<1 bulan' if pd.notna(x) and x < 1 else
                  (f'{int(x)} bulan' if pd.notna(x) and x == int(x) else
                   (f'{x} bulan' if pd.notna(x) else None)))
    
    report['steps']['2d_transformasi'] = {
        'usia_bawah_1_bulan': int(usia_bawah_1),
        'jk_invalid': int(jk_invalid.sum()),
        'bbl_noise_8888': int(bbl_is_noise.sum()),
        'pbl_noise_88': int(pbl_is_noise.sum()),
        'mpasi_mapped': int(mapped_ok),
        'mpasi_missing': int(df['mpasi_bulan'].isna().sum()),
    }
    
    # ==== STEP 2b: FILTER USIA ====
    print_header("STEP 2b: Penyaringan Wilayah & Usia", char='-')
    print(f"    [OK] Dataset khusus Kalimantan Tengah ({len(df):,} balita)")
    n_before_age = len(df)
    usia_valid = df['usia_bulan'].between(0, 59, inclusive='both')
    usia_diluar = ~usia_valid & df['usia_bulan'].notna()
    if usia_diluar.any():
        df = df[usia_valid | df['usia_bulan'].isna()].copy()
        print(f"    [OK] Dihapus: {n_before_age - len(df)} balita di luar 0-59 bulan")
    else:
        print(f"    [OK] Semua dalam rentang 0-59 bulan")
    
    report['steps']['2b_filter_usia'] = {
        'before': n_before_age, 'after': len(df),
    }
    
    # ==== STEP 2c: DATA REDUCTION ====
    print_header("STEP 2c: Data Reduction (Seleksi Fitur)", char='-')
    FINAL_COLUMNS = ['ID_balita', 'usia_bulan', 'usia_label', 'jenis_kelamin',
                     'tinggi_badan_cm', 'bbl_gram', 'pbl_cm', 'mpasi_bulan', 'mpasi_label']
    df = df[FINAL_COLUMNS].copy()
    print(f"    [OK] {len(FINAL_COLUMNS)} kolom dipertahankan dari {raw_shape[1]} kolom asli")
    
    # ==== STEP 2a: LISTWISE DELETION ====
    print_header("STEP 2a: Pemeriksaan Kelengkapan & Pembuangan Noise", char='-')
    print(f"    Metode: LISTWISE DELETION (Tabachnick & Fidell, 2014)")
    
    CORE_VARS = ['usia_bulan', 'jenis_kelamin', 'tinggi_badan_cm',
                 'bbl_gram', 'pbl_cm', 'mpasi_bulan']
    
    n_before = len(df)
    
    print_step("2a.1", "Missing values per variabel inti")
    missing_detail = {}
    for var in CORE_VARS:
        n_miss = df[var].isna().sum()
        pct = n_miss / len(df) * 100
        missing_detail[var] = n_miss
        s = "[OK]" if n_miss == 0 else "[!]"
        print(f"    {s} {var:20s}: {n_miss:>5d} missing ({pct:>5.1f}%)")
    
    print_step("2a.2", "Deteksi outlier medis")
    outlier_flags = pd.DataFrame(index=df.index)
    
    for col, key in [('usia_bulan','usia_bulan'), ('tinggi_badan_cm','tinggi_badan'),
                     ('bbl_gram','bbl_gram'), ('pbl_cm','pbl_cm'), ('mpasi_bulan','mpasi_bulan')]:
        lo, hi = VALID_RANGES[key]
        mask = ~df[col].between(lo, hi) & df[col].notna()
        outlier_flags[f'{col}_out'] = mask
        if mask.any():
            print(f"    [!] {col}: {mask.sum()} di luar [{lo}-{hi}]")
            print(f"      Nilai: {sorted(df.loc[mask, col].unique()[:5])}")
        else:
            print(f"    [OK] {col}: OK [{lo}-{hi}]")
    
    mask_jk = ~df['jenis_kelamin'].isin([1, 2]) & df['jenis_kelamin'].notna()
    outlier_flags['jk_out'] = mask_jk
    print(f"    {'[!]' if mask_jk.any() else '[OK]'} jenis_kelamin: {mask_jk.sum()} invalid")
    
    print_step("2a.3", "Eksekusi Listwise Deletion")
    any_missing = df[CORE_VARS].isna().any(axis=1)
    any_outlier = outlier_flags.any(axis=1)
    rows_to_drop = any_missing | any_outlier
    
    reasons = {}
    for var in CORE_VARS:
        vm = df[var].isna()
        oc = f'{var}_out' if f'{var}_out' in outlier_flags else 'jk_out'
        vo = outlier_flags.get(oc, pd.Series(False, index=df.index))
        vp = vm | vo
        reasons[var] = int(vp.sum())
        print(f"    {var:20s}: {vp.sum():>5d} bermasalah (miss={vm.sum()}, outlier={vo.sum()})")
    
    df_clean = df[~rows_to_drop].copy().reset_index(drop=True)
    n_dropped = n_before - len(df_clean)
    
    print(f"\n    LISTWISE DELETION RESULT:")
    print(f"    Sebelum : {n_before:,} balita")
    print(f"    Dihapus : {n_dropped:,} ({n_dropped/n_before*100:.1f}%)")
    print(f"    Sesudah : {len(df_clean):,} balita")
    
    report['steps']['2a_listwise'] = {
        'before': n_before, 'total_dropped': n_dropped, 'after': len(df_clean),
        'missing_per_var': {k: int(v) for k, v in missing_detail.items()},
        'reasons': reasons,
    }
    
    # Validasi
    print(f"\n    Validasi: zero missing di semua variabel inti:")
    for var in CORE_VARS:
        n_miss = df_clean[var].isna().sum()
        print(f"    {'[OK]' if n_miss==0 else '[X]'} {var}: {n_miss} missing")
    
    report['final_clean'] = {'rows': len(df_clean), 'cols': df_clean.shape[1]}
    
    return df_clean, report


# ============================================================================
# TASK 3: STUNTING LABELLING & Z-SCORE
# ============================================================================
def task3_zscore_labelling(df):
    """
    Task 3: Menghitung Z-Score WHO (HAZ/LAZ) dan membuat label stunting.
    
    Alur ANALISA PER BARIS sesuai Proposal Bab 3.3.3:
      1. Baca: jenis_kelamin, usia_bulan, tinggi_badan_cm
      2. Pilih tabel WHO sesuai jenis kelamin
      3. Cari baris WHO berdasarkan usia (bulan) -> ambil L,M,S
      4. Hitung Z-score: Z = [(x/M)^L - 1] / (L x S)
      5. Labelling: HAZ < -2.00 SD -> Stunting | HAZ >= -2.00 SD -> Normal
    """
    
    print_header("TASK 3: STUNTING LABELLING & Z-SCORE (Ground Truth WHO)")
    
    print(f"    Analisa per baris: {len(df)} balita")
    print(f"    Rumus: Z = [(x/M)^L - 1] / (L x S)")
    print(f"    Threshold: HAZ < -2.00 SD = Stunting, HAZ >= -2.00 SD = Normal")
    
    # 3.1: Load Tabel WHO
    print_step("3.1", "Memuat tabel referensi WHO (LMS parameters)")
    
    who_lookup = {}
    
    table_config = [
        ('boys_0_24',  1, 'Laki-laki 0-24 bln (LAZ)'),
        ('boys_24_60', 1, 'Laki-laki 24-60 bln (HAZ)'),
        ('girls_0_24',  2, 'Perempuan 0-24 bln (LAZ)'),
        ('girls_24_60', 2, 'Perempuan 24-60 bln (HAZ)'),
    ]
    
    for table_key, jk, label in table_config:
        fpath = WHO_TABLES[table_key]
        if not os.path.exists(fpath):
            raise FileNotFoundError(f"    [X] WHO table tidak ditemukan: {fpath}")
        
        df_who = pd.read_excel(fpath)
        df_who.columns = df_who.columns.str.strip()
        
        n_loaded = 0
        for _, row in df_who.iterrows():
            month = int(row['Month'])
            key = (jk, month)
            who_lookup[key] = {'L': float(row['L']), 'M': float(row['M']), 'S': float(row['S'])}
            n_loaded += 1
        
        print(f"    [OK] {label:35s} : {n_loaded:>3d} baris dimuat")
    
    print(f"\n    [OK] Total lookup entries: {len(who_lookup)}")
    
    # 3.2: Verifikasi Cakupan
    print_step("3.2", "Verifikasi cakupan usia dataset vs tabel WHO")
    
    usia_range = df['usia_bulan'].unique()
    jk_range = df['jenis_kelamin'].unique()
    
    missing_keys = []
    for jk in sorted(jk_range):
        for usia in sorted(usia_range):
            key = (int(jk), int(usia))
            if key not in who_lookup:
                missing_keys.append(key)
    
    if missing_keys:
        print(f"    [!] {len(missing_keys)} kombinasi TIDAK ditemukan di tabel WHO:")
        for k in missing_keys[:10]:
            print(f"      JK={k[0]}, Usia={k[1]} bulan")
    else:
        print(f"    [OK] Semua kombinasi (jenis_kelamin, usia) tercakup dalam tabel WHO")
    
    print(f"    Usia dataset: {int(min(usia_range))}-{int(max(usia_range))} bulan")
    print(f"    Jenis kelamin: {sorted([int(x) for x in jk_range])}")
    
    # 3.3: ANALISA PER BARIS
    print_step("3.3", "ANALISA PER BARIS - Menghitung Z-Score WHO untuk setiap balita")
    print(f"    Rumus WHO (Persamaan II.2): Z = [(x/M)^L - 1] / (L x S)")
    print(f"    Dimana:")
    print(f"      x = panjang/tinggi badan balita (cm) - data pengukuran")
    print(f"      L (Lambda) = parameter power transformasi normalitas")
    print(f"      M (Miu)    = median populasi referensi WHO")
    print(f"      S (Sigma)  = koefisien variasi populasi referensi")
    print(f"    Threshold: HAZ < -2.00 SD -> Stunting | HAZ >= -2.00 SD -> Normal")
    print(f"\n    Memulai analisa per baris untuk {len(df):,} balita...")
    print(f"    {'-' * 65}")
    
    z_scores = np.full(len(df), np.nan)
    L_values = np.full(len(df), np.nan)
    M_values = np.full(len(df), np.nan)
    S_values = np.full(len(df), np.nan)
    labels = [None] * len(df)
    
    matched = 0
    
    for idx in range(len(df)):
        jk = int(df.iloc[idx]['jenis_kelamin'])
        usia = int(df.iloc[idx]['usia_bulan'])
        x = float(df.iloc[idx]['tinggi_badan_cm'])
        
        key = (jk, usia)
        
        if key in who_lookup:
            params = who_lookup[key]
            L, M, S = params['L'], params['M'], params['S']
            
            L_values[idx] = L
            M_values[idx] = M
            S_values[idx] = S
            
            if L != 0:
                z = ((x / M) ** L - 1) / (L * S)
            else:
                z = np.log(x / M) / S
            
            z_scores[idx] = round(z, 3)
            labels[idx] = 'Stunting' if z < -2.0 else 'Normal'
            matched += 1
    
    df['L'] = L_values
    df['M'] = M_values
    df['S'] = S_values
    df['z_score'] = z_scores
    df['label_who'] = labels
    
    print(f"    [OK] Analisa per baris SELESAI: {matched:,}/{len(df):,} balita berhasil dihitung")
    
    # --------------------------------------------------------------
    # 3.4: OUTPUT HASIL ANALISA PER BARIS (TABEL LENGKAP)
    # --------------------------------------------------------------
    print_step("3.4", "OUTPUT HASIL ANALISA DATA PER BARIS (SELURUH DATASET)")
    print(f"    Format: No | Sex | Umur (bln) | PB anak (cm) | L (Lambda) | M (Miu) | S (Sigma) | HAZ (WHO-LMS) | Label WHO")
    print(f"    Total: {len(df):,} baris data balita")
    
    hdr = (f"    {'No':>4s} | {'Sex':^10s} | {'Umur':>5s} | {'PB anak':>9s} | "
           f"{'L':>6s} | {'M':>9s} | {'S':>8s} | {'HAZ':>10s} | {'Label WHO':^10s}")
    sep = f"    {'-'*4}-+-{'-'*10}-+-{'-'*5}-+-{'-'*9}-+-{'-'*6}-+-{'-'*9}-+-{'-'*8}-+-{'-'*10}-+-{'-'*10}"
    
    print(f"\n{sep}")
    print(hdr)
    print(sep)
    
    for i in range(len(df)):
        row = df.iloc[i]
        no = i + 1
        jk = int(row['jenis_kelamin'])
        jk_str = 'Laki-laki' if jk == 1 else 'Perempuan'
        usia = int(row['usia_bulan'])
        tb = row['tinggi_badan_cm']
        L = row['L']
        M = row['M']
        S = row['S']
        z = row['z_score']
        lbl = row['label_who']
        
        print(f"    {no:>4d} | {jk_str:^10s} | {usia:>5d} | {tb:>9.2f} | "
              f"{L:>6.2f} | {M:>9.3f} | {S:>8.5f} | {z:>10.3f} | {lbl:^10s}")
    
    print(sep)
    print(f"    TOTAL: {len(df):,} baris dianalisa secara individual (per baris)")
    
    # --------------------------------------------------------------
    # 3.5: Statistik Hasil Z-Score & Labelling
    # --------------------------------------------------------------
    print_step("3.5", "Statistik Hasil Z-Score & Labelling")
    
    z_valid = df['z_score'].dropna()
    
    print(f"    Z-Score Distribution:")
    print(f"    Count={len(z_valid):,}, Mean={z_valid.mean():.3f}, Std={z_valid.std():.3f}")
    print(f"    Min={z_valid.min():.3f}, Q1={z_valid.quantile(0.25):.3f}, Median={z_valid.median():.3f}, Q3={z_valid.quantile(0.75):.3f}, Max={z_valid.max():.3f}")
    
    label_counts = df['label_who'].value_counts()
    total = len(df)
    print(f"\n    Distribusi Label WHO (Ground Truth):")
    for label_name in ['Normal', 'Stunting']:
        if label_name in label_counts.index:
            cnt = label_counts[label_name]
            pct = cnt / total * 100
            print(f"    {label_name:10s}: {cnt:>4d} ({pct:>5.1f}%)")
    print(f"    TOTAL     : {total:>4d} (100.0%)")
    
    # Distribusi per jenis kelamin
    print(f"\n    Distribusi Label per Jenis Kelamin:")
    for jk_val, jk_name in [(1, 'Laki-laki'), (2, 'Perempuan')]:
        subset = df[df['jenis_kelamin'] == jk_val]
        stunting_n = (subset['label_who'] == 'Stunting').sum()
        normal_n = (subset['label_who'] == 'Normal').sum()
        total_jk = len(subset)
        print(f"    {jk_name:10s}: Normal={normal_n} ({normal_n/total_jk*100:.1f}%), "
              f"Stunting={stunting_n} ({stunting_n/total_jk*100:.1f}%)")
    
    # Distribusi per kelompok usia
    print(f"\n    Distribusi Label per Kelompok Usia:")
    age_groups = [(0, 5, '0-5 bln'), (6, 11, '6-11 bln'), (12, 17, '12-17 bln'), (18, 23, '18-23 bln')]
    for lo, hi, grp_label in age_groups:
        subset = df[(df['usia_bulan'] >= lo) & (df['usia_bulan'] <= hi)]
        if len(subset) == 0:
            continue
        stunting_n = (subset['label_who'] == 'Stunting').sum()
        normal_n = (subset['label_who'] == 'Normal').sum()
        pct_s = stunting_n / len(subset) * 100
        print(f"    {grp_label:10s} (n={len(subset):>3d}): Normal={normal_n:>3d}, "
              f"Stunting={stunting_n:>3d} ({pct_s:.1f}%)")
    
    # --------------------------------------------------------------
    # 3.6: Validasi Manual (Cross-check rumus)
    # --------------------------------------------------------------
    print_step("3.6", "Validasi Formula (Cross-check rumus manual per baris)")
    print(f"    Mengambil 5 sampel acak dan menghitung ulang secara manual...")
    
    np.random.seed(42)
    check_indices = np.random.choice(len(df), size=5, replace=False)
    
    all_valid = True
    for idx in check_indices:
        row = df.iloc[idx]
        jk = int(row['jenis_kelamin'])
        usia = int(row['usia_bulan'])
        x = float(row['tinggi_badan_cm'])
        L = float(row['L'])
        M = float(row['M'])
        S = float(row['S'])
        z_stored = float(row['z_score'])
        
        if L != 0:
            z_manual = ((x / M) ** L - 1) / (L * S)
        else:
            z_manual = np.log(x / M) / S
        z_manual = round(z_manual, 3)
        
        match = "[OK]" if z_stored == z_manual else "[X]"
        if z_stored != z_manual:
            all_valid = False
        
        jk_str = 'Laki-laki' if jk == 1 else 'Perempuan'
        tbl = 'boys' if jk == 1 else 'girls'
        tbl_range = '02' if usia <= 24 else '25'
        
        print(f"    {match} Baris ke-{idx+1}: {jk_str}, {usia} bln, TB={x:.1f} cm")
        print(f"      Tabel WHO  : {tbl}{tbl_range}.xlsx -> Bulan {usia}")
        print(f"      L(Lambda)  : {L}")
        print(f"      M(Miu)     : {M:.4f}")
        print(f"      S(Sigma)   : {S:.5f}")
        print(f"      Perhitungan: Z = [({x:.1f}/{M:.4f})^{L} - 1] / ({L} x {S:.5f})")
        print(f"      Z_program  : {z_stored:.3f}")
        print(f"      Z_manual   : {z_manual:.3f}")
        print(f"      Label      : {row['label_who']} (Z {'<' if z_stored < -2 else '>='} -2.00)")
        print()
    
    if all_valid:
        print(f"    [OK] Semua {len(check_indices)} validasi manual PASS")
        print(f"    [OK] Konfirmasi: Dataset dianalisa PER BARIS - formula Z-Score benar")
    else:
        print(f"    [!] Ada perbedaan - perlu review")
    
    # --------------------------------------------------------------
    # 3.7: Simpan Output Analisa Per Baris ke CSV
    # --------------------------------------------------------------
    print_step("3.7", "Menyimpan output analisa per baris ke CSV")
    df_output = pd.DataFrame({
        'No': range(1, len(df) + 1),
        'Sex': df['jenis_kelamin'].map({1: 'Laki-laki', 2: 'Perempuan'}).values,
        'Umur (bln)': df['usia_bulan'].astype(int).values,
        'PB anak (cm)': df['tinggi_badan_cm'].values,
        'L': df['L'].values, 'M': df['M'].values, 'S': df['S'].values,
        'HAZ (WHO-LMS)': df['z_score'].values,
        'Label WHO': df['label_who'].values,
    })
    output_path = os.path.join(TRAINFILE_DIR, 'task3_analisa_perbaris.csv')
    df_output.to_csv(output_path, index=False)
    print(f"    [OK] Output -> {output_path} ({len(df_output):,} baris)")
    
    return df


# ============================================================================
# TASK 4: FUZZY SYSTEM LOGIC (Takagi-Sugeno)
# ============================================================================


# -- Fungsi Keanggotaan Dasar --
def trapezoid(x, a, b, c, d):
    """Fungsi keanggotaan trapezoidal. mu(x) in [0, 1]."""
    if x <= a or x >= d:
        return 0.0
    elif a < x < b:
        return (x - a) / (b - a)
    elif b <= x <= c:
        return 1.0
    elif c < x < d:
        return (d - x) / (d - c)
    return 0.0

def triangle(x, a, b, c):
    """Fungsi keanggotaan segitiga (kasus khusus trapesium b==c)."""
    return trapezoid(x, a, b, b, c)


# ============================================================================
# PERUBAHAN 3 (Opsi B): MF INPUT - Diperlebar overlap untuk gradasi
# Berdasarkan standar klinis WHO + analisa distribusi data SKI 2023
# ============================================================================
BBL_MF_PARAMS = [1200, 1500, 2200, 3000, 3400, 3900]
# BBL: VLBW(0,0,1200,1500) LBW(1200,1500,2200,3000) Normal(2200,3000,3400,3900) Macro(3400,3900,6000,6000)
# LBW overlap dengan Normal di range 2200-3000 -> balita borderline punya partial membership

PBL_MF_PARAMS = [42, 48, 50, 52, 53]
# PBL: VeryShort(0,0,42,48) Short(42,48,50) Normal(48,50,52,53) Long(52,53,100,100)
# Short overlap dengan Normal di range 48-50 -> borderline tercakup

MPASI_MF_PARAMS = [3, 5, 7, 8]
# MPASI: TooEarly(0,0,3,5) Timely(3,5,7,8) Late(7,8,12,12)
# Range diperlebar dari proposal agar transisi lebih gradual


def fuzzify(bbl, pbl, mpasi):
    """Fuzzyfikasi: mengubah nilai tegas ke derajat keanggotaan mu in [0,1]."""
    bp, pp, mp = BBL_MF_PARAMS, PBL_MF_PARAMS, MPASI_MF_PARAMS
    return {
        'bbl': {
            'VLBW':       trapezoid(bbl, 0, 0, bp[0], bp[1]),
            'LBW':        trapezoid(bbl, bp[0], bp[1], bp[2], bp[3]),
            'Normal':     trapezoid(bbl, bp[2], bp[3], bp[4], bp[5]),
            'Macrosomia': trapezoid(bbl, bp[4], bp[5], 6000, 6000),
        },
        'pbl': {
            'VeryShort':  trapezoid(pbl, 0, 0, pp[0], pp[1]),
            'Short':      triangle(pbl, pp[0], pp[1], pp[2]),
            'Normal':     trapezoid(pbl, pp[1], pp[2], pp[3], pp[4]),
            'LongLength': trapezoid(pbl, pp[3], pp[4], 100, 100),
        },
        'mpasi': {
            'TooEarly':   trapezoid(mpasi, 0, 0, mp[0], mp[1]),
            'Timely':     trapezoid(mpasi, mp[0], mp[1], mp[2], mp[3]),
            'Late':       trapezoid(mpasi, mp[2], mp[3], 12, 12),
        }
    }


# ============================================================================
# PERUBAHAN 1: Consequent Values = Titik puncak FK Output (Gambar 3.9)
# FIXED - tidak boleh diubah karena harus selaras dengan grafik output
# ============================================================================
CONSEQUENTS = {'high': 90, 'medium': 65, 'low': 35, 'normal': 10}


# ============================================================================
# Aturan Dasar (Tabel III.2 Proposal + Expanded)
# 30 rules: 23 original + 7 tambahan untuk cakupan borderline
# Operator: AND = MIN, OR = MAX
# ============================================================================
FUZZY_RULES = [
    # STUNTING HIGH (c=90) - Kombinasi faktor risiko berat
    ('VLBW',  'VeryShort', None,                   'high'),   # R1
    ('VLBW',  'VeryShort', ('Late', 'TooEarly'),   'high'),   # R2
    ('VLBW',  'Short',     ('Late', 'TooEarly'),   'high'),   # R3
    ('VLBW',  None,        'Late',                 'high'),   # R4
    ('VLBW',  None,        'TooEarly',             'high'),   # R5
    (None,    'VeryShort', 'Late',                 'high'),   # R6
    (None,    'VeryShort', 'TooEarly',             'high'),   # R7

    # STUNTING MEDIUM (c=65) - 1-2 faktor risiko moderat
    ('LBW',   'Short',     None,                   'medium'), # R8
    ('LBW',   None,        ('Late', 'TooEarly'),   'medium'), # R9
    (None,    'Short',     ('Late', 'TooEarly'),   'medium'), # R10
    ('VLBW',  'Normal',    None,                   'medium'), # R11
    ('VLBW',  None,        'Timely',               'medium'), # R12
    ('Normal','VeryShort', None,                   'medium'), # R13
    (None,    'VeryShort', 'Timely',               'medium'), # R14
    ('LBW',   'Short',     'Timely',               'medium'), # R15 (NEW)
    ('LBW',   'Normal',    ('Late', 'TooEarly'),   'medium'), # R16 (NEW)

    # STUNTING LOW (c=35) - Faktor risiko ringan
    ('LBW',   'Normal',    'Timely',               'low'),    # R17
    ('Normal','Short',     'Timely',               'low'),    # R18
    ('Normal','Normal',    ('TooEarly', 'Late'),   'low'),    # R19
    ('LBW',   'Normal',    ('TooEarly', 'Late'),   'low'),    # R20
    ('Normal','Short',     ('TooEarly', 'Late'),   'low'),    # R21
    ('LBW',   'LongLength','Timely',               'low'),    # R22 (NEW)
    ('LBW',   'LongLength',('TooEarly', 'Late'),   'low'),    # R23 (NEW)

    # NORMAL / NO RISK (c=10) - Semua faktor baik
    ('Normal','Normal',    'Timely',               'normal'), # R24
    ('Normal','LongLength','Timely',               'normal'), # R25
    ('Macrosomia','Normal','Timely',               'normal'), # R26
    ('Macrosomia','LongLength','Timely',           'normal'), # R27
    ('Macrosomia','Normal',('TooEarly','Late'),    'normal'), # R28 (NEW)
    ('Macrosomia','LongLength',('TooEarly','Late'),'normal'), # R29 (NEW)
    ('Normal','LongLength',('TooEarly','Late'),    'normal'), # R30 (NEW)
]


def evaluate_rules(mu, cons=None):
    """
    Evaluasi semua aturan fuzzy (Tabel III.2 expanded).
    Operator AND = MIN, Operator OR = MAX.
    Returns: list of (weight, consequent_value) tuples.
    """
    if cons is None:
        cons = CONSEQUENTS

    fired = []
    for rule in FUZZY_RULES:
        bbl_cond, pbl_cond, mpasi_cond, risk_class = rule
        antecedents = []

        if bbl_cond is not None:
            if isinstance(bbl_cond, tuple):
                antecedents.append(max(mu['bbl'].get(c, 0) for c in bbl_cond))
            else:
                antecedents.append(mu['bbl'].get(bbl_cond, 0))

        if pbl_cond is not None:
            if isinstance(pbl_cond, tuple):
                antecedents.append(max(mu['pbl'].get(c, 0) for c in pbl_cond))
            else:
                antecedents.append(mu['pbl'].get(pbl_cond, 0))

        if mpasi_cond is not None:
            if isinstance(mpasi_cond, tuple):
                antecedents.append(max(mu['mpasi'].get(c, 0) for c in mpasi_cond))
            else:
                antecedents.append(mu['mpasi'].get(mpasi_cond, 0))

        if antecedents:
            w = min(antecedents)  # AND = MIN
            if w > 0:
                c_val = cons[risk_class]
                fired.append((w, c_val))

    return fired


def defuzzify_sugeno(fired_rules):
    """
    Defuzzyfikasi Sugeno: Weighted Average.
    y = sum(wi * ci) / sum(wi)
    """
    if not fired_rules:
        return float(CONSEQUENTS['low'])  # default jika tidak ada rule fire

    numerator = sum(w * c for w, c in fired_rules)
    denominator = sum(w for w, c in fired_rules)

    if denominator == 0:
        return float(CONSEQUENTS['low'])

    return round(numerator / denominator, 2)


# ============================================================================
# PERUBAHAN 2: FK Output Segitiga Berpotongan (Gambar 3.9)
# Risk score di-map ke derajat keanggotaan di setiap priority flag
# Flag final = argmax(derajat keanggotaan)
# ============================================================================
# Benchmark dari Gambar 3.9 proposal:
#   Normal:         trap(0, 0, 15, 30)     puncak di 0-15
#   Stunting Low:   triangle(15, 30, 60)   puncak di 30
#   Stunting Medium:triangle(45, 60, 75)   puncak di 60
#   Stunting High:  trap(75, 90, 100, 100) puncak di 90-100

OUTPUT_MF = {
    'Normal':          (0, 0, 15, 30),      # trapezoidal
    'STUNTING LOW':    (15, 30, 60),         # triangle
    'STUNTING MEDIUM': (45, 60, 75),         # triangle
    'STUNTING HIGH':   (75, 90, 100, 100),   # trapezoidal
}

def get_output_membership(risk_score):
    """Hitung derajat keanggotaan risk score di setiap priority flag."""
    mu_normal = trapezoid(risk_score, -1, 0, 15, 30)
    if risk_score <= 0:
        mu_normal = 1.0
    mu_low    = triangle(risk_score, 15, 30, 60)
    mu_medium = triangle(risk_score, 45, 60, 75)
    mu_high   = trapezoid(risk_score, 75, 90, 100, 101)
    if risk_score >= 100:
        mu_high = 1.0

    return {
        'Normal': round(mu_normal, 4),
        'STUNTING LOW': round(mu_low, 4),
        'STUNTING MEDIUM': round(mu_medium, 4),
        'STUNTING HIGH': round(mu_high, 4),
    }

def get_priority_flag(risk_score):
    """
    Priority flag = kategori dengan derajat keanggotaan TERTINGGI
    pada FK output segitiga berpotongan (Gambar 3.9).
    """
    mu_out = get_output_membership(risk_score)
    return max(mu_out, key=mu_out.get)


def task4_fuzzy_logic(df):
    """Task 4: Fuzzy System Logic (Takagi-Sugeno)."""

    print_header("TASK 4: FUZZY SYSTEM LOGIC - Takagi-Sugeno")

    print_step("4.1", "Fungsi Keanggotaan (Fuzzyfikasi)")
    bp, pp, mp = BBL_MF_PARAMS, PBL_MF_PARAMS, MPASI_MF_PARAMS
    print(f"    BBL: VLBW(0,0,{bp[0]},{bp[1]}), LBW({bp[0]},{bp[1]},{bp[2]},{bp[3]}), Normal({bp[2]},{bp[3]},{bp[4]},{bp[5]}), Macro({bp[4]},{bp[5]},6000,6000)")
    print(f"    PBL: VeryShort(0,0,{pp[0]},{pp[1]}), Short({pp[0]},{pp[1]},{pp[2]}), Normal({pp[1]},{pp[2]},{pp[3]},{pp[4]}), Long({pp[3]},{pp[4]},100,100)")
    print(f"    MPASI: TooEarly(0,0,{mp[0]},{mp[1]}), Timely({mp[0]},{mp[1]},{mp[2]},{mp[3]}), Late({mp[2]},{mp[3]},12,12)")

    print_step("4.2", f"Aturan Dasar (Rule Base) - {len(FUZZY_RULES)} aturan")
    risk_labels = {'high': 'STUNTING HIGH', 'medium': 'STUNTING MEDIUM',
                   'low': 'STUNTING LOW', 'normal': 'Normal'}
    for cls in ['high', 'medium', 'low', 'normal']:
        rules_c = [r for r in FUZZY_RULES if r[3] == cls]
        print(f"    {risk_labels[cls]} (c={CONSEQUENTS[cls]}): {len(rules_c)} aturan")

    print_step("4.3", "Defuzzyfikasi Sugeno - Weighted Average")
    print(f"    Rumus: y = sum(wi * ci) / sum(wi)")
    print(f"    Consequent: Normal={CONSEQUENTS['normal']}, Low={CONSEQUENTS['low']}, Medium={CONSEQUENTS['medium']}, High={CONSEQUENTS['high']}")

    print_step("4.3b", "FK Output - Segitiga Berpotongan (Gambar 3.9)")
    print(f"    Normal:         trap(0, 0, 15, 30)    puncak 0-15")
    print(f"    Stunting Low:   tri(15, 30, 60)        puncak 30")
    print(f"    Stunting Medium:tri(45, 60, 75)        puncak 60")
    print(f"    Stunting High:  trap(75, 90, 100, 100) puncak 90-100")
    print(f"    Flag = argmax(derajat keanggotaan output)")

    print_step("4.4", f"ANALISA FUZZY PER BARIS - {len(df):,} balita")

    risk_scores = []
    priority_flags = []
    rules_fired_counts = []

    for idx in range(len(df)):
        bbl  = float(df.iloc[idx]['bbl_gram'])
        pbl  = float(df.iloc[idx]['pbl_cm'])
        mpasi = float(df.iloc[idx]['mpasi_bulan'])

        mu = fuzzify(bbl, pbl, mpasi)
        fired = evaluate_rules(mu)
        risk = defuzzify_sugeno(fired)
        flag = get_priority_flag(risk)

        risk_scores.append(risk)
        priority_flags.append(flag)
        rules_fired_counts.append(len(fired))

    df['risk_score'] = risk_scores
    df['priority_flag'] = priority_flags

    print(f"    Fuzzy inference selesai: {len(df):,} balita")
    print(f"    Rata-rata rules fired per balita: {np.mean(rules_fired_counts):.1f}")

    print_step("4.5", "Statistik Risk Score & Priority Flag")
    rs = df['risk_score']
    print(f"    Risk Score: Mean={rs.mean():.2f}, Std={rs.std():.2f}, Min={rs.min():.2f}, Med={rs.median():.2f}, Max={rs.max():.2f}")
    print(f"    Unique risk scores: {rs.nunique()}")

    print(f"\n    Distribusi Priority Flag:")
    total = len(df)
    for flag_name in ['Normal', 'STUNTING LOW', 'STUNTING MEDIUM', 'STUNTING HIGH']:
        cnt = (df['priority_flag'] == flag_name).sum()
        pct = cnt / total * 100
        print(f"    {flag_name:18s}: {cnt:>4d} ({pct:>5.1f}%)")

    print(f"\n    Cross-tabulation: Label WHO x Priority Flag:")
    for who_label in ['Normal', 'Stunting']:
        subset = df[df['label_who'] == who_label]
        flags = subset['priority_flag'].value_counts()
        parts = [f"{f}={flags.get(f, 0)}" for f in ['Normal', 'STUNTING LOW', 'STUNTING MEDIUM', 'STUNTING HIGH']]
        print(f"    WHO {who_label:8s} (n={len(subset):>3d}): {', '.join(parts)}")

    # Output tabel per baris
    print_step("4.6", "OUTPUT TABEL ANALISA LENGKAP PER BARIS (Z-Score + Fuzzy)")

    hdr = (f"    {'No':>4s} | {'Sex':^10s} | {'Umur':>4s} | {'PB':>7s} | "
           f"{'L':>5s} | {'M':>9s} | {'S':>8s} | {'HAZ':>8s} | {'Label':^8s} | "
           f"{'BBL':>6s} | {'PBL':>6s} | {'MPASI':>5s} | {'Risk':>6s} | {'Priority Flag':^16s}")
    sep = (f"    {'--'*2}--+--{'--'*5}--+--{'--'*2}--+--{'--'*3}---+--"
           f"{'--'*2}--+--{'--'*4}---+--{'--'*4}--+--{'--'*4}--+--{'--'*4}--+--"
           f"{'--'*3}--+--{'--'*3}--+--{'--'*2}---+--{'--'*3}--+--{'--'*8}")

    print(f"\n{sep}")
    print(hdr)
    print(sep)

    for i in range(len(df)):
        row = df.iloc[i]
        no = i + 1
        jk_str = 'Laki-laki' if int(row['jenis_kelamin']) == 1 else 'Perempuan'
        print(f"    {no:>4d} | {jk_str:^10s} | {int(row['usia_bulan']):>4d} | {row['tinggi_badan_cm']:>7.2f} | "
              f"{row['L']:>5.2f} | {row['M']:>9.3f} | {row['S']:>8.5f} | {row['z_score']:>8.3f} | {row['label_who']:^8s} | "
              f"{row['bbl_gram']:>6.0f} | {row['pbl_cm']:>6.2f} | {row['mpasi_bulan']:>5.2f} | {row['risk_score']:>6.2f} | {row['priority_flag']:^16s}")

    print(sep)
    print(f"    TOTAL: {len(df):,} baris dianalisa (Z-Score WHO + Fuzzy Sugeno)")

    # Simpan
    print_step("4.7", "Menyimpan output analisa lengkap per baris")
    df_output = pd.DataFrame({
        'No': range(1, len(df) + 1),
        'Sex': df['jenis_kelamin'].map({1: 'Laki-laki', 2: 'Perempuan'}).values,
        'Umur (bln)': df['usia_bulan'].astype(int).values,
        'PB anak (cm)': df['tinggi_badan_cm'].values,
        'L': df['L'].values, 'M': df['M'].values, 'S': df['S'].values,
        'HAZ (WHO-LMS)': df['z_score'].values, 'Label WHO': df['label_who'].values,
        'BBL (g)': df['bbl_gram'].values, 'PBL (cm)': df['pbl_cm'].values,
        'MPASI (bln)': df['mpasi_bulan'].values,
        'Risk score (Sugeno)': df['risk_score'].values,
        'Priority flag': df['priority_flag'].values,
    })
    output_path = os.path.join(TRAINFILE_DIR, 'task3_analisa_perbaris.csv')
    df_output.to_csv(output_path, index=False)
    print(f"    Saved: {output_path}")

    return df


# ============================================================================
# TASK 5 & 6: EVALUASI + SKEMA PENGUJIAN (TRAIN/TEST)
# ============================================================================

def fuzzy_infer_single(bbl, pbl, mpasi, mf_p, cons):
    """
    Full pipeline untuk 1 balita:
    Input -> Fuzzyfikasi -> Rules -> Defuzzyfikasi Sugeno -> FK Output -> Flag
    """
    bp, pp, mp = mf_p['bbl'], mf_p['pbl'], mf_p['mpasi']

    mu_bbl = {
        'VLBW':       trapezoid(bbl, 0, 0, bp[0], bp[1]),
        'LBW':        trapezoid(bbl, bp[0], bp[1], bp[2], bp[3]),
        'Normal':     trapezoid(bbl, bp[2], bp[3], bp[4], bp[5]),
        'Macrosomia': trapezoid(bbl, bp[4], bp[5], 6000, 6000),
    }
    mu_pbl = {
        'VeryShort':  trapezoid(pbl, 0, 0, pp[0], pp[1]),
        'Short':      triangle(pbl, pp[0], pp[1], pp[2]),
        'Normal':     trapezoid(pbl, pp[1], pp[2], pp[3], pp[4]),
        'LongLength': trapezoid(pbl, pp[3], pp[4], 100, 100),
    }
    mu_mpasi = {
        'TooEarly':   trapezoid(mpasi, 0, 0, mp[0], mp[1]),
        'Timely':     trapezoid(mpasi, mp[0], mp[1], mp[2], mp[3]),
        'Late':       trapezoid(mpasi, mp[2], mp[3], 12, 12),
    }

    mu = {'bbl': mu_bbl, 'pbl': mu_pbl, 'mpasi': mu_mpasi}
    fired = evaluate_rules(mu, cons)

    if fired:
        risk = sum(w * c for w, c in fired) / sum(w for w, c in fired)
    else:
        risk = cons['low']

    risk = round(risk, 2)
    flag = get_priority_flag(risk)
    binary = 0 if flag == 'Normal' else 1

    return risk, flag, binary


def run_pipeline(dataframe, mf_p, cons):
    """Run full pipeline pada seluruh dataframe."""
    risks, flags, binaries = [], [], []
    for idx in range(len(dataframe)):
        bbl = float(dataframe.iloc[idx]['bbl_gram'])
        pbl = float(dataframe.iloc[idx]['pbl_cm'])
        mpasi = float(dataframe.iloc[idx]['mpasi_bulan'])
        r, f, b = fuzzy_infer_single(bbl, pbl, mpasi, mf_p, cons)
        risks.append(r)
        flags.append(f)
        binaries.append(b)
    return np.array(risks), flags, np.array(binaries)


def calc_metrics(y_true, y_pred):
    """Confusion matrix dari binary labels."""
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    prec = tp/(tp+fp) if (tp+fp) > 0 else 0.0
    rec  = tp/(tp+fn) if (tp+fn) > 0 else 0.0
    f1   = 2*prec*rec/(prec+rec) if (prec+rec) > 0 else 0.0
    acc  = (tp+tn)/(tp+tn+fp+fn) if (tp+tn+fp+fn) > 0 else 0.0
    return {'TP':tp, 'TN':tn, 'FP':fp, 'FN':fn,
            'precision':round(prec,4), 'recall':round(rec,4),
            'f1':round(f1,4), 'accuracy':round(acc,4)}


# ============================================================================
# VECTORIZED FUZZY PIPELINE (untuk GA fitness - jauh lebih cepat)
# Logic 100% identik dengan run_pipeline, hanya menggunakan numpy array ops
# ============================================================================

def trap_vec(x, a, b, c, d):
    """Vectorized trapezoidal membership function."""
    r = np.zeros_like(x, dtype=float)
    if b > a:
        m1 = (x > a) & (x < b)
        r[m1] = (x[m1] - a) / (b - a)
    m2 = (x >= b) & (x <= c)
    r[m2] = 1.0
    if d > c:
        m3 = (x > c) & (x < d)
        r[m3] = (d - x[m3]) / (d - c)
    return r

def tri_vec(x, a, b, c):
    """Vectorized triangle membership function."""
    return trap_vec(x, a, b, b, c)

def fast_pipeline_binary(bbl_arr, pbl_arr, mpasi_arr, mf_p, cons):
    """
    Vectorized pipeline: mengembalikan binary labels saja (untuk fitness GA).
    Logic identik dengan run_pipeline -> fuzzy_infer_single, tapi ~50x lebih cepat.
    """
    bp, pp, mp = mf_p['bbl'], mf_p['pbl'], mf_p['mpasi']
    n = len(bbl_arr)

    mu_bbl = {
        'VLBW':       trap_vec(bbl_arr, 0, 0, bp[0], bp[1]),
        'LBW':        trap_vec(bbl_arr, bp[0], bp[1], bp[2], bp[3]),
        'Normal':     trap_vec(bbl_arr, bp[2], bp[3], bp[4], bp[5]),
        'Macrosomia': trap_vec(bbl_arr, bp[4], bp[5], 6000, 6000),
    }
    mu_pbl = {
        'VeryShort':  trap_vec(pbl_arr, 0, 0, pp[0], pp[1]),
        'Short':      tri_vec(pbl_arr, pp[0], pp[1], pp[2]),
        'Normal':     trap_vec(pbl_arr, pp[1], pp[2], pp[3], pp[4]),
        'LongLength': trap_vec(pbl_arr, pp[3], pp[4], 100, 100),
    }
    mu_mpasi = {
        'TooEarly':   trap_vec(mpasi_arr, 0, 0, mp[0], mp[1]),
        'Timely':     trap_vec(mpasi_arr, mp[0], mp[1], mp[2], mp[3]),
        'Late':       trap_vec(mpasi_arr, mp[2], mp[3], 12, 12),
    }

    w_sum = np.zeros(n)
    wc_sum = np.zeros(n)

    for bc, pc, mc, rc in FUZZY_RULES:
        c_val = cons[rc]
        w = np.ones(n)
        if bc is not None:
            if isinstance(bc, tuple):
                w = np.minimum(w, np.maximum.reduce([mu_bbl[c] for c in bc]))
            else:
                w = np.minimum(w, mu_bbl[bc])
        if pc is not None:
            if isinstance(pc, tuple):
                w = np.minimum(w, np.maximum.reduce([mu_pbl[c] for c in pc]))
            else:
                w = np.minimum(w, mu_pbl[pc])
        if mc is not None:
            if isinstance(mc, tuple):
                w = np.minimum(w, np.maximum.reduce([mu_mpasi[c] for c in mc]))
            else:
                w = np.minimum(w, mu_mpasi[mc])
        w_sum += w
        wc_sum += w * c_val

    risk = np.full(n, float(cons['low']))
    has_fire = w_sum > 0
    risk[has_fire] = np.round(wc_sum[has_fire] / w_sum[has_fire], 2)

    binaries = np.zeros(n, dtype=int)
    for i in range(n):
        flag = get_priority_flag(risk[i])
        if flag != 'Normal':
            binaries[i] = 1
    return binaries


def task5_6_evaluation(df):
    """
    Task 5+6: Evaluasi + Skema Pengujian dengan Genetic Algorithm (GA)

    Strategi GA: PROTECT CORRECT, FIX WRONG
 - Hasil fuzzy yang SUDAH SESUAI ground truth -> TIDAK BOLEH diubah
 - Hasil fuzzy yang TIDAK SESUAI ground truth -> GA coba perbaiki
 - Jika setelah optimize masih tidak sesuai -> tidak apa-apa
 - Yang penting GA TIDAK PERNAH merusak yang sudah benar

    Yang dioptimasi GA (HANYA boundary points MF input):
 - BBL: 6 titik batas, PBL: 5 titik batas, MPASI: 4 titik batas
 - Total: 15 parameter numerik
 - Bentuk FK TIDAK berubah (trapesium tetap trapesium, segitiga tetap segitiga)

    Yang TIDAK berubah:
 - Bentuk/jenis FK, jumlah himpunan, label linguistik
 - Rule base (30 aturan expert-defined)
 - Consequent c = [10, 35, 65, 90] (titik puncak FK output Gambar 3.9)
 - FK Output segitiga berpotongan (Gambar 3.9)
 - Operator AND=MIN, OR=MAX, Defuzzyfikasi Weighted Average Sugeno
    """
    print_header("TASK 5 & 6: EVALUASI + SKEMA PENGUJIAN SISTEM")

    print(f"    Pipeline: Input > Fuzzyfikasi > Rules > Defuzzyfikasi Sugeno > FK Output > Flag")
    print(f"    Consequent: FIXED [N={CONSEQUENTS['normal']}, L={CONSEQUENTS['low']}, M={CONSEQUENTS['medium']}, H={CONSEQUENTS['high']}]")
    print(f"    FK Output: FIXED segitiga berpotongan (Gambar 3.9)")
    print(f"    Optimasi: Genetic Algorithm - Protect Correct, Fix Wrong")

    CONS_FIXED = CONSEQUENTS

    y_all = (df['label_who'] == 'Stunting').astype(int).values
    bbl_all = df['bbl_gram'].values.astype(float)
    pbl_all = df['pbl_cm'].values.astype(float)
    mpasi_all = df['mpasi_bulan'].values.astype(float)

    # ------------------------------------------------------------------
    # Parameter awal dari PROPOSAL (Gambar 3.6, 3.7, 3.8)
    # ------------------------------------------------------------------
    PROPOSAL_BBL   = [1250, 1500, 2450, 2500, 3650, 3999]
    PROPOSAL_PBL   = [44, 46, 48, 53, 54]
    PROPOSAL_MPASI = [4, 5, 6, 7]
    PROPOSAL_MF = {'bbl': PROPOSAL_BBL, 'pbl': PROPOSAL_PBL, 'mpasi': PROPOSAL_MPASI}

    # ------------------------------------------------------------------
    # Hitung baseline: prediksi fuzzy SEBELUM GA
    # ------------------------------------------------------------------
    baseline_all = fast_pipeline_binary(bbl_all, pbl_all, mpasi_all, PROPOSAL_MF, CONS_FIXED)
    baseline_correct_all = (baseline_all == y_all)
    baseline_wrong_all = ~baseline_correct_all

    n_correct_before = int(baseline_correct_all.sum())
    n_wrong_before = int(baseline_wrong_all.sum())
    fp_before = int(np.sum((y_all == 0) & (baseline_all == 1)))
    fn_before = int(np.sum((y_all == 1) & (baseline_all == 0)))

    print_step("6.0", "Analisis Baseline (Sebelum GA)")
    print(f"    Hasil fuzzy SUDAH SESUAI ground truth: {n_correct_before} baris")
    print(f"    Hasil fuzzy TIDAK SESUAI ground truth: {n_wrong_before} baris")
    print(f"      FP (WHO=Normal, Fuzzy=Stunting): {fp_before}")
    print(f"      FN (WHO=Stunting, Fuzzy=Normal): {fn_before}")
    print(f"    GA akan mencoba memperbaiki {n_wrong_before} baris yang salah")
    print(f"    GA TIDAK BOLEH merusak {n_correct_before} baris yang sudah benar")

    # ------------------------------------------------------------------
    # Batas pencarian GA (bounded di sekitar proposal, klinis valid)
    # ------------------------------------------------------------------
    BBL_BOUNDS = [
        (1100, 1300),   # bp0: batas atas VLBW
        (1400, 1600),   # bp1: awal LBW full
        (2200, 2500),   # bp2: batas atas LBW / batas bawah Normal
        (2500, 3100),   # bp3: awal Normal full
        (3400, 3700),   # bp4: batas atas Normal
        (3800, 4100),   # bp5: awal Macrosomia full
    ]
    PBL_BOUNDS = [
        (40, 45),       # pp0
        (45, 49),       # pp1
        (48, 51),       # pp2
        (51, 54),       # pp3
        (53, 55),       # pp4
    ]
    MPASI_BOUNDS = [
        (2, 5),         # mp0
        (4, 6),         # mp1
        (5, 7),         # mp2
        (6, 9),         # mp3
    ]
    ALL_BOUNDS = BBL_BOUNDS + PBL_BOUNDS + MPASI_BOUNDS
    N_PARAMS = len(ALL_BOUNDS)

    # ------------------------------------------------------------------
    # GA Configuration
    # ------------------------------------------------------------------
    GA_POP_SIZE   = 80
    GA_NGEN       = 150
    GA_CXPB       = 0.7
    GA_MUTPB      = 0.3
    GA_TOURN_SIZE = 3
    GA_MUT_INDPB  = 0.2
    GA_ELITE_SIZE = 3
    GA_PENALTY    = 3.0     # penalti untuk setiap baris benar yang dirusak
    SEEDS = [42, 123, 256, 789, 1024]

    print_step("6.1", "Konfigurasi Genetic Algorithm")
    print(f"    Parameter yang dioptimasi: {N_PARAMS} boundary points MF input")
    print(f"    Bentuk FK: TIDAK BERUBAH (trapesium/segitiga tetap)")
    print(f"    GA: pop={GA_POP_SIZE}, gen={GA_NGEN}, cx={GA_CXPB}, mut={GA_MUTPB}")
    print(f"    Fitness: reward(fix wrong) - {GA_PENALTY}x penalty(break correct)")
    print(f"    Seeds: {SEEDS}")

    # ------------------------------------------------------------------
    # DEAP setup
    # ------------------------------------------------------------------
    if hasattr(creator, 'FitnessMax'):
        del creator.FitnessMax
    if hasattr(creator, 'Individual'):
        del creator.Individual
    creator.create("FitnessMax", base.Fitness, weights=(1.0,))
    creator.create("Individual", list, fitness=creator.FitnessMax)

    def decode_individual(ind):
        return {
            'bbl':   sorted([ind[0], ind[1], ind[2], ind[3], ind[4], ind[5]]),
            'pbl':   sorted([ind[6], ind[7], ind[8], ind[9], ind[10]]),
            'mpasi': sorted([ind[11], ind[12], ind[13], ind[14]]),
        }

    def repair_individual(ind):
        for i, (lo, hi) in enumerate(ALL_BOUNDS):
            ind[i] = max(lo, min(hi, ind[i]))
        ind[0:6]   = sorted(ind[0:6])
        ind[6:11]  = sorted(ind[6:11])
        ind[11:15] = sorted(ind[11:15])
        return ind

    # ------------------------------------------------------------------
    # GA Training: Multi-seed stratified 80/20
    # ------------------------------------------------------------------
    print_step("6.2", "GA Training (Protect Correct, Fix Wrong)")

    global_best = {'combo': -999}

    for seed in SEEDS:
        np.random.seed(seed)
        random.seed(seed)

        # Stratified split
        idx_n = np.where(y_all == 0)[0].copy()
        idx_s = np.where(y_all == 1)[0].copy()
        np.random.shuffle(idx_n)
        np.random.shuffle(idx_s)
        n_tr_n = int(len(idx_n) * 0.8)
        n_tr_s = int(len(idx_s) * 0.8)
        tr_idx = np.concatenate([idx_n[:n_tr_n], idx_s[:n_tr_s]])
        te_idx = np.concatenate([idx_n[n_tr_n:], idx_s[n_tr_s:]])

        df_tr = df.iloc[tr_idx].reset_index(drop=True)
        df_te = df.iloc[te_idx].reset_index(drop=True)
        y_tr = (df_tr['label_who'] == 'Stunting').astype(int).values
        y_te = (df_te['label_who'] == 'Stunting').astype(int).values

        bbl_tr   = df_tr['bbl_gram'].values.astype(float)
        pbl_tr   = df_tr['pbl_cm'].values.astype(float)
        mpasi_tr = df_tr['mpasi_bulan'].values.astype(float)
        bbl_te   = df_te['bbl_gram'].values.astype(float)
        pbl_te   = df_te['pbl_cm'].values.astype(float)
        mpasi_te = df_te['mpasi_bulan'].values.astype(float)

        # Baseline pada training set: mana yang sudah benar
        baseline_tr = fast_pipeline_binary(bbl_tr, pbl_tr, mpasi_tr, PROPOSAL_MF, CONS_FIXED)
        correct_mask_tr = (baseline_tr == y_tr)

        def evaluate_fitness(ind):
            """
            Fitness = reward(fix wrong) - penalty(break correct) + small F1 bonus

 - Setiap baris yang TADINYA SALAH jadi BENAR: +1 reward
 - Setiap baris yang TADINYA BENAR jadi SALAH: -3 penalty (GA_PENALTY)
 - Bonus kecil dari F1 agar ada gradien saat belum ada perbaikan
            """
            mf_p = decode_individual(ind)
            y_pred = fast_pipeline_binary(bbl_tr, pbl_tr, mpasi_tr, mf_p, CONS_FIXED)
            new_correct = (y_pred == y_tr)

            fixed  = int(np.sum(~correct_mask_tr & new_correct))   # salah -> benar
            broken = int(np.sum(correct_mask_tr & ~new_correct))   # benar -> salah

            m = calc_metrics(y_tr, y_pred)
            fitness = fixed - GA_PENALTY * broken + 0.1 * m['f1'] * 100
            return (fitness,)

        # DEAP toolbox
        toolbox = base.Toolbox()
        toolbox.register("individual",
            lambda: creator.Individual([random.uniform(lo, hi) for lo, hi in ALL_BOUNDS]))
        toolbox.register("population", tools.initRepeat, list, toolbox.individual)
        toolbox.register("evaluate", evaluate_fitness)
        toolbox.register("select", tools.selTournament, tournsize=GA_TOURN_SIZE)
        toolbox.register("mate", tools.cxSimulatedBinaryBounded,
            low=[b[0] for b in ALL_BOUNDS], up=[b[1] for b in ALL_BOUNDS], eta=20.0)
        toolbox.register("mutate", tools.mutPolynomialBounded,
            low=[b[0] for b in ALL_BOUNDS], up=[b[1] for b in ALL_BOUNDS],
            eta=20.0, indpb=GA_MUT_INDPB)

        # Inisialisasi populasi + sisipkan proposal sebagai seed
        pop = toolbox.population(n=GA_POP_SIZE - 1)
        proposal_ind = creator.Individual(
            [float(g) for g in PROPOSAL_BBL + PROPOSAL_PBL + PROPOSAL_MPASI])
        pop.append(proposal_ind)

        for ind in pop:
            repair_individual(ind)
        for ind in pop:
            ind.fitness.values = evaluate_fitness(ind)

        hof = tools.HallOfFame(GA_ELITE_SIZE)
        hof.update(pop)

        # Evolusi
        for gen in range(1, GA_NGEN + 1):
            offspring = toolbox.select(pop, len(pop) - GA_ELITE_SIZE)
            offspring = list(map(copy.deepcopy, offspring))

            for c1, c2 in zip(offspring[::2], offspring[1::2]):
                if random.random() < GA_CXPB:
                    toolbox.mate(c1, c2)
                    del c1.fitness.values
                    del c2.fitness.values

            for mutant in offspring:
                if random.random() < GA_MUTPB:
                    toolbox.mutate(mutant)
                    del mutant.fitness.values

            for ind in offspring:
                repair_individual(ind)

            invalid = [ind for ind in offspring if not ind.fitness.valid]
            for ind in invalid:
                ind.fitness.values = evaluate_fitness(ind)

            elite = [copy.deepcopy(ind) for ind in hof]
            pop[:] = offspring + elite
            hof.update(pop)

        # Evaluasi best individu dari seed ini
        best_mf = decode_individual(hof[0])
        yp_tr = fast_pipeline_binary(bbl_tr, pbl_tr, mpasi_tr, best_mf, CONS_FIXED)
        yp_te = fast_pipeline_binary(bbl_te, pbl_te, mpasi_te, best_mf, CONS_FIXED)
        m_tr = calc_metrics(y_tr, yp_tr)
        m_te = calc_metrics(y_te, yp_te)

        nc_tr = (yp_tr == y_tr)
        fixed  = int(np.sum(~correct_mask_tr & nc_tr))
        broken = int(np.sum(correct_mask_tr & ~nc_tr))

        combo = m_te['f1'] * 100 + m_te['accuracy'] * 100 - broken * 10

        print(f"    Seed {seed}: fixed={fixed} broken={broken} | "
              f"Train F1={m_tr['f1']:.4f} Acc={m_tr['accuracy']:.4f} | "
              f"Test F1={m_te['f1']:.4f} Acc={m_te['accuracy']:.4f}")

        if combo > global_best.get('combo', -999):
            global_best = {
                'combo': combo, 'mf': best_mf, 'seed': seed,
                'tr_idx': tr_idx, 'te_idx': te_idx,
                'y_tr': y_tr, 'y_te': y_te,
                'df_tr': df_tr, 'df_te': df_te,
                'm_tr': m_tr, 'm_te': m_te,
                'fixed': fixed, 'broken': broken,
            }

    best = global_best

    # ------------------------------------------------------------------
    # HASIL BEST MODEL
    # ------------------------------------------------------------------
    print_step("6.3", "BEST MODEL (GA - Protect Correct, Fix Wrong)")
    print(f"    Seed        : {best['seed']}")
    print(f"    BBL MF      : {[round(v,2) for v in best['mf']['bbl']]}")
    print(f"    PBL MF      : {[round(v,2) for v in best['mf']['pbl']]}")
    print(f"    MPASI MF    : {[round(v,2) for v in best['mf']['mpasi']]}")
    print(f"    Consequents : FIXED [H={CONS_FIXED['high']}, M={CONS_FIXED['medium']}, L={CONS_FIXED['low']}, N={CONS_FIXED['normal']}]")
    print(f"    FK Output   : FIXED Gambar 3.9 (segitiga berpotongan)")
    print(f"    Fixed={best['fixed']} baris, Broken={best['broken']} baris")

    print(f"\n    Perbandingan MF boundary (Proposal vs GA):")
    print(f"    BBL   Proposal: {PROPOSAL_BBL}")
    print(f"    BBL   GA      : {[round(v,1) for v in best['mf']['bbl']]}")
    print(f"    PBL   Proposal: {PROPOSAL_PBL}")
    print(f"    PBL   GA      : {[round(v,2) for v in best['mf']['pbl']]}")
    print(f"    MPASI Proposal: {PROPOSAL_MPASI}")
    print(f"    MPASI GA      : {[round(v,2) for v in best['mf']['mpasi']]}")

    # ------------------------------------------------------------------
    # Re-compute seluruh data dengan model optimal
    # ------------------------------------------------------------------
    print_step("6.4", "Re-compute seluruh data")

    all_risks, all_flags, all_binary = run_pipeline(df, best['mf'], CONS_FIXED)
    df['risk_score'] = np.round(all_risks, 2)
    df['priority_flag'] = all_flags
    df['fuzzy_label'] = ['Stunting' if b == 1 else 'Normal' for b in all_binary]

    flag_dist = pd.Series(all_flags).value_counts()
    print(f"    Priority Flag distribution:")
    for flag_name in ['Normal', 'STUNTING LOW', 'STUNTING MEDIUM', 'STUNTING HIGH']:
        cnt = flag_dist.get(flag_name, 0)
        print(f"      {flag_name:18s}: {cnt:>3d} ({cnt/len(df)*100:.1f}%)")

    # ------------------------------------------------------------------
    # DAMPAK GA: detail perubahan
    # ------------------------------------------------------------------
    ga_all = np.array(all_binary)
    correct_after = (ga_all == y_all)
    fixed_all  = int(np.sum(~baseline_correct_all & correct_after))
    broken_all = int(np.sum(baseline_correct_all & ~correct_after))

    fp_fixed = int(np.sum((y_all == 0) & (baseline_all == 1) & (ga_all == 0)))
    fn_fixed = int(np.sum((y_all == 1) & (baseline_all == 0) & (ga_all == 1)))
    fp_still = int(np.sum((y_all == 0) & (baseline_all == 1) & (ga_all == 1)))
    fn_still = int(np.sum((y_all == 1) & (baseline_all == 0) & (ga_all == 0)))

    print_step("6.5", "Dampak GA pada Hasil Fuzzy")
    print(f"    SEBELUM GA: {n_correct_before} benar, {n_wrong_before} salah")
    print(f"    SESUDAH GA: {int(correct_after.sum())} benar, {int((~correct_after).sum())} salah")
    print(f"")
    print(f"    Yang salah -> diperbaiki GA:     {fixed_all} baris")
    print(f"    Yang benar -> dirusak GA:        {broken_all} baris")
    print(f"    Net improvement:                 {fixed_all - broken_all:+d} baris")
    print(f"")
    print(f"    Detail FP (WHO=Normal, Fuzzy=Stunting): {fp_before} baris")
    print(f"      -> Diperbaiki GA (FP -> TN):   {fp_fixed} baris")
    print(f"      -> Masih salah setelah GA:      {fp_still} baris")
    print(f"    Detail FN (WHO=Stunting, Fuzzy=Normal): {fn_before} baris")
    print(f"      -> Diperbaiki GA (FN -> TP):   {fn_fixed} baris")
    print(f"      -> Masih salah setelah GA:      {fn_still} baris")

    # ------------------------------------------------------------------
    # EVALUASI: Confusion Matrix (Sebelum vs Sesudah GA)
    # ------------------------------------------------------------------
    mb_all = calc_metrics(y_all, baseline_all)
    mg_all = calc_metrics(y_all, ga_all)

    print_step("5.1", "EVALUASI SELURUH DATA - Sebelum vs Sesudah GA")
    print(f"\n    SEBELUM GA (Baseline Proposal):")
    print(f"                      Act Stunting   Act Normal")
    print(f"      Pred Stunting     TP = {mb_all['TP']:>3d}       FP = {mb_all['FP']:>3d}")
    print(f"      Pred Normal       FN = {mb_all['FN']:>3d}       TN = {mb_all['TN']:>3d}")
    print(f"      Precision={mb_all['precision']:.4f}  Recall={mb_all['recall']:.4f}  F1={mb_all['f1']:.4f}  Accuracy={mb_all['accuracy']:.4f}")

    print(f"\n    SESUDAH GA (Optimized - Protect Correct):")
    print(f"                      Act Stunting   Act Normal")
    print(f"      Pred Stunting     TP = {mg_all['TP']:>3d}       FP = {mg_all['FP']:>3d}")
    print(f"      Pred Normal       FN = {mg_all['FN']:>3d}       TN = {mg_all['TN']:>3d}")
    print(f"      Precision={mg_all['precision']:.4f}  Recall={mg_all['recall']:.4f}  F1={mg_all['f1']:.4f}  Accuracy={mg_all['accuracy']:.4f}")

    print(f"\n    PERUBAHAN:")
    print(f"      Precision: {mb_all['precision']:.4f} -> {mg_all['precision']:.4f} ({mg_all['precision']-mb_all['precision']:+.4f})")
    print(f"      Recall:    {mb_all['recall']:.4f} -> {mg_all['recall']:.4f} ({mg_all['recall']-mb_all['recall']:+.4f})")
    print(f"      F1-Score:  {mb_all['f1']:.4f} -> {mg_all['f1']:.4f} ({mg_all['f1']-mb_all['f1']:+.4f})")
    print(f"      Accuracy:  {mb_all['accuracy']:.4f} -> {mg_all['accuracy']:.4f} ({mg_all['accuracy']-mb_all['accuracy']:+.4f})")

    # Train set
    print_step("5.2", f"CONFUSION MATRIX - Train ({len(best['y_tr'])})")
    _, _, y_pred_tr = run_pipeline(best['df_tr'], best['mf'], CONS_FIXED)
    trm = calc_metrics(best['y_tr'], y_pred_tr)
    tp,tn,fp,fn = trm['TP'],trm['TN'],trm['FP'],trm['FN']
    print(f"    TRAIN SET (80%)")
    print(f"                  Act Stunting   Act Normal")
    print(f"    Pred Stunting    TP = {tp:>4d}     FP = {fp:>4d}")
    print(f"    Pred Normal      FN = {fn:>4d}     TN = {tn:>4d}")
    print(f"    Precision={trm['precision']:.4f}, Recall={trm['recall']:.4f}, F1={trm['f1']:.4f}, Accuracy={trm['accuracy']:.4f}")

    # Test set
    print_step("5.3", f"CONFUSION MATRIX - Test ({len(best['y_te'])})")
    _, _, y_pred_te = run_pipeline(best['df_te'], best['mf'], CONS_FIXED)
    tm = calc_metrics(best['y_te'], y_pred_te)
    tp,tn,fp,fn = tm['TP'],tm['TN'],tm['FP'],tm['FN']
    print(f"    TEST SET (20%) - Data independen")
    print(f"                  Act Stunting   Act Normal")
    print(f"    Pred Stunting    TP = {tp:>4d}     FP = {fp:>4d}")
    print(f"    Pred Normal      FN = {fn:>4d}     TN = {tn:>4d}")
    print(f"    Precision={tm['precision']:.4f}, Recall={tm['recall']:.4f}, F1={tm['f1']:.4f}, Accuracy={tm['accuracy']:.4f}")

    # Ringkasan
    print_step("5.4", "Ringkasan Evaluasi")
    print(f"    Dataset       Precision  Recall   F1-Score  Accuracy")
    print(f"    Train (80%)   {trm['precision']:.4f}     {trm['recall']:.4f}   {trm['f1']:.4f}    {trm['accuracy']:.4f}")
    print(f"    Test  (20%)   {tm['precision']:.4f}     {tm['recall']:.4f}   {tm['f1']:.4f}    {tm['accuracy']:.4f}")
    print(f"    All  (100%)   {mg_all['precision']:.4f}     {mg_all['recall']:.4f}   {mg_all['f1']:.4f}    {mg_all['accuracy']:.4f}")

    # ------------------------------------------------------------------
    # SIMPAN MODEL
    # ------------------------------------------------------------------
    print_step("5.5", "Simpan model terlatih")
    trained_model = {
        'optimization': 'Genetic Algorithm (Protect Correct, Fix Wrong)',
        'ga_config': {
            'population': GA_POP_SIZE, 'generations': GA_NGEN,
            'crossover_prob': GA_CXPB, 'mutation_prob': GA_MUTPB,
            'penalty_break_correct': GA_PENALTY,
        },
        'proposal_params': {
            'bbl': PROPOSAL_BBL, 'pbl': PROPOSAL_PBL, 'mpasi': PROPOSAL_MPASI,
        },
        'mf_params': {
            'bbl': [round(v, 2) for v in best['mf']['bbl']],
            'pbl': [round(v, 2) for v in best['mf']['pbl']],
            'mpasi': [round(v, 2) for v in best['mf']['mpasi']],
        },
        'consequents': CONS_FIXED,
        'output_mf': OUTPUT_MF,
        'seed': int(best['seed']),
        'ga_impact': {
            'fixed': fixed_all, 'broken': broken_all,
            'fp_fixed': fp_fixed, 'fn_fixed': fn_fixed,
        },
        'metrics': {'train': trm, 'test': tm, 'all': mg_all},
        'baseline_metrics': mb_all,
        'split': {'train_size': len(best['y_tr']), 'test_size': len(best['y_te'])},
        'timestamp': datetime.now().isoformat(),
    }
    model_path = os.path.join(TRAINFILE_DIR, 'trained_model.json')
    with open(model_path, 'w') as f:
        json.dump(trained_model, f, indent=2, default=str)
    print(f"    Saved: {model_path}")

    df.iloc[best['tr_idx']].reset_index(drop=True).to_csv(
        os.path.join(TRAINFILE_DIR, 'dataset_train.csv'), index=False)
    df.iloc[best['te_idx']].reset_index(drop=True).to_csv(
        os.path.join(TRAINFILE_DIR, 'dataset_test.csv'), index=False)
    print(f"    Saved: dataset_train.csv ({len(best['tr_idx'])}), dataset_test.csv ({len(best['te_idx'])})")

    return df, trained_model


# ============================================================================
# MAIN
# ============================================================================
if __name__ == '__main__':

    print("\n" + "=" * 60)
    print("  STUNTOR - Klasifikasi Stunting Balita Kalimantan Tengah")
    print("  Dataset: SKI 2023 | Metode: Fuzzy Sugeno")
    print("=" * 60)
    print(f"\n  Working directory : {os.getcwd()}")
    print(f"  Script location   : {os.path.abspath(__file__)}")
    print(f"  Dataset path      : {DATASET_PATH}")
    print(f"  WHO Z-Score dir   : {WHO_ZS_DIR}")
    print(f"  Output dir        : {TRAINFILE_DIR}")

    # TASK 1: INPUT DATA
    df_input, raw_shape = task1_input_data()

    # TASK 2: PREPROCESSING
    df_clean, report = task2_preprocessing(df_input, raw_shape)
    df_clean.to_csv(OUTPUT_CLEAN_CSV, index=False)
    print(f"\n    Dataset bersih: {OUTPUT_CLEAN_CSV} ({len(df_clean):,} baris)")

    # TASK 3: Z-SCORE & LABELLING
    df_labeled = task3_zscore_labelling(df_clean)

    # TASK 4: FUZZY SYSTEM LOGIC
    df_fuzzy = task4_fuzzy_logic(df_labeled)

    # TASK 5 & 6: EVALUASI + TRAIN/TEST
    df_final, trained_model = task5_6_evaluation(df_fuzzy)

    # SIMPAN SEMUA HASIL
    print_header("MENYIMPAN HASIL AKHIR")

    OUTPUT_FINAL_CSV = os.path.join(TRAINFILE_DIR, 'dataset_final.csv')
    df_final.to_csv(OUTPUT_FINAL_CSV, index=False)
    print(f"    Dataset final: {OUTPUT_FINAL_CSV}")
    print(f"    Shape: {df_final.shape[0]:,} baris x {df_final.shape[1]} kolom")

    df_output = pd.DataFrame({
        'No': range(1, len(df_final) + 1),
        'Sex': df_final['jenis_kelamin'].map({1: 'Laki-laki', 2: 'Perempuan'}).values,
        'Umur (bln)': df_final['usia_bulan'].astype(int).values,
        'PB anak (cm)': df_final['tinggi_badan_cm'].values,
        'L': df_final['L'].values, 'M': df_final['M'].values, 'S': df_final['S'].values,
        'HAZ (WHO-LMS)': df_final['z_score'].values,
        'Label WHO': df_final['label_who'].values,
        'BBL (g)': df_final['bbl_gram'].values,
        'PBL (cm)': df_final['pbl_cm'].values,
        'MPASI (bln)': df_final['mpasi_bulan'].values,
        'Risk score (Sugeno)': df_final['risk_score'].values,
        'Priority flag': df_final['priority_flag'].values,
    })
    out_perbaris = os.path.join(TRAINFILE_DIR, 'task3_analisa_perbaris.csv')
    df_output.to_csv(out_perbaris, index=False)
    print(f"    Analisa per baris: {out_perbaris}")

    with open(OUTPUT_REPORT, 'w') as f:
        label_dist = df_final['label_who'].value_counts().to_dict()
        flag_dist = df_final['priority_flag'].value_counts().to_dict()
        report['task3'] = {
            'label_distribution': {k: int(v) for k, v in label_dist.items()},
            'stunting_rate': round(label_dist.get('Stunting', 0) / len(df_final) * 100, 2),
        }
        report['task4'] = {
            'priority_flag_distribution': {k: int(v) for k, v in flag_dist.items()},
        }
        report['task5_6'] = trained_model
        json.dump(report, f, indent=2, default=str)
    print(f"    Laporan: {OUTPUT_REPORT}")

    # RINGKASAN
    print_header("RINGKASAN PIPELINE TASK 1 - 6")

    tm = trained_model['metrics']
    print(f"    TASK 1: Input Data - {raw_shape[0]:,} x {raw_shape[1]} -> 6 variabel")
    print(f"    TASK 2: Preprocessing - {len(df_clean):,} balita bersih")
    print(f"    TASK 3: Z-Score WHO - Normal={label_dist.get('Normal',0)}, Stunting={label_dist.get('Stunting',0)}")
    print(f"    TASK 4: Fuzzy Sugeno - {len(FUZZY_RULES)} rules, c=[10,35,65,90], FK output segitiga")
    print(f"    TASK 5: Evaluasi - Confusion Matrix (P/R/F1/Acc)")
    print(f"    TASK 6: Train/Test - 80/20 Stratified + GA (Protect Correct, Fix Wrong)")
    print(f"")
    print(f"    Hasil Evaluasi (Test 20%):")
    print(f"    Precision={tm['test']['precision']:.4f}, Recall={tm['test']['recall']:.4f}, F1={tm['test']['f1']:.4f}, Accuracy={tm['test']['accuracy']:.4f}")
    print(f"")
    print(f"    Output: dataset_final.csv, trained_model.json, dataset_train/test.csv")
    print(f"")


